# MS2 neural ranking pilot — Colab T4

Select **Runtime → Change runtime type → T4 GPU**, then **Run all**. This notebook contains a snapshot of its Python sources: no GitHub push or repository upload is required.

It compares a spectrum/GNN contrastive ranker against the same ranker with auxiliary neural spectrum reconstruction, uniform ranking, and the existing Ridge/prior/mass baselines. It downloads the pinned MassSpecGym table (~249 MiB) and a 64 MiB candidate prefix. Default: 5,000 training molecules, 20 epochs per neural arm, 200 validation queries, batch size 32, FP32.

**Scope:** restricted-domain candidate retrieval. No quantum calculations. This does not train the Mamba completion decoder, use predicted substructures, learn a dedicated molecular prior, or establish physical stability. Benchmark metadata lacks energy-count information; unknown is encoded as 0. Official connectivity-group splits and duplicate-spectrum exclusions are reused; scaffold disjointness is not established. The previously inspected validation slice supports exploratory comparison, not an untouched test claim. One seed is a pilot, not a reproducibility study.

Epoch selection uses only held-out training calibration loss. The final validation fold is evaluated after checkpoint selection. Test spectra are scanned by the existing loader but never used to train, select, or evaluate models. All-query top-25 denominators retain failures and unsupported chemistry. Zero-scored queries still have seeded fallback ranks, so these are raw retrieval metrics, not accepted confident predictions.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.6.1', 'requests==2.32.5'], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before running.'
assert 'T4' in torch.cuda.get_device_name(), f'T4 required; allocated {torch.cuda.get_device_name()}'
print(torch.__version__, torch.cuda.get_device_name())
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
from pathlib import Path
import os
WORK = Path('/content/ms2_neural_pilot')
WORK.mkdir(exist_ok=True)
os.chdir(WORK)
SOURCES = {'tools/ms2_neural_ranker.py': '"""T4 neural ranking pilot; no stability, Mamba-completion, or FPNet claims.\n\nReuses the frozen Ridge experiment\'s selection and leakage exclusions. Graph\nencoding has the existing restricted neutral typed-graph domain. Missing graph\nscores stay in a seeded tail and every selected query stays in the denominator.\n"""\nimport argparse\nimport copy\nimport json\nimport math\nimport random\nimport re\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom tools.ms2_fp_predictor import collect_train_molecules, select_val\nfrom tools.ms2_msgym_corpus import standardize_smiles\nfrom tools.ms2_spectral_rank import content_fp, parse_spectrum, seeded_positions, sha256_file, short_key\n\nELEMENTS = ("C", "N", "O", "F", "Cl", "Br", "I", "S")\nN_BINS = 2000\nN_BONDS = 4  # single, double, triple, and the shared parser\'s aromatic marker\n\n\ndef spectrum_features(peaks, precursor):\n    """FPNet-style filter/cap/sqrt, then fixed 1-Da bins (not FPNet itself)."""\n    if not math.isfinite(precursor) or precursor <= 0:\n        raise ValueError("precursor_mz must be finite and positive")\n    if any(not math.isfinite(m) or not math.isfinite(i) or i < 0 for m, i in peaks):\n        raise ValueError("peaks must be finite with nonnegative intensities")\n    kept = [(m, i, j) for j, (m, i) in enumerate(peaks) if 0 < m <= precursor + 2]\n    maximum = max((i for _, i, _ in kept), default=0)\n    result = np.zeros(N_BINS, dtype=np.float32)\n    if maximum <= 0:\n        return result\n    kept = [(m, i / maximum, j) for m, i, j in kept if i / maximum >= 1e-3]\n    for m, i, j in sorted(kept, key=lambda p: (-p[1], p[2]))[:160]:\n        if m < N_BINS:\n            result[int(m)] = max(result[int(m)], math.sqrt(i))\n    norm = np.linalg.norm(result)\n    return result / norm if norm else result\n\n\ndef metadata(row, adduct_ids):\n    precursor = float(row["precursor_mz"])\n    adduct = row["adduct"]\n    polarity = 1 if adduct.endswith("+") else -1 if adduct.endswith("-") else 0\n    # Do not interpret percentages or normalized CE as eV.\n    energy_text = row.get("collision_energy", "").strip()\n    match = re.fullmatch(r"([0-9]+(?:\\.[0-9]+)?)\\s*(?:eV)?", energy_text)\n    energy = float(match.group(1)) if match else 0.0\n    count_text = row.get("energy_count", "").strip()\n    count = int(count_text) if count_text else 0\n    if not 0 <= count <= 8:\n        raise ValueError("energy_count must be in [0, 8]")\n    values = [precursor / 2000, polarity, energy / 100, float(match is not None), count / 8]\n    if not all(math.isfinite(x) for x in values):\n        raise ValueError("metadata must be finite")\n    return np.asarray(values, dtype=np.float32), adduct_ids.get(adduct, 0)\n\n\ndef graph(smiles):\n    record, reason = standardize_smiles(smiles, "neural-pilot", smiles)\n    if record is None:\n        return None\n    atoms = record["atom_types"]\n    x = np.zeros((len(atoms), len(ELEMENTS) + 2), dtype=np.float32)\n    for index, (element, hydrogens, valence) in enumerate(atoms):\n        x[index, ELEMENTS.index(element)] = 1\n        x[index, -2:] = hydrogens / 4, valence / 4\n    adjacency = np.zeros((N_BONDS, len(atoms), len(atoms)), dtype=np.float32)\n    for a, b, order in record["edges"]:\n        adjacency[order - 1, a, b] = adjacency[order - 1, b, a] = 1\n    return x, adjacency\n\n\ndef pack_graphs(graphs, device):\n    if not graphs or any(g is None for g in graphs):\n        raise ValueError("pack_graphs requires supported, nonempty graphs")\n    n = max(len(g[0]) for g in graphs)\n    x = np.zeros((len(graphs), n, len(ELEMENTS) + 2), dtype=np.float32)\n    a = np.zeros((len(graphs), N_BONDS, n, n), dtype=np.float32)\n    mask = np.zeros((len(graphs), n, 1), dtype=np.float32)\n    for j, (nodes, bonds) in enumerate(graphs):\n        k = len(nodes)\n        x[j, :k], a[j, :, :k, :k], mask[j, :k] = nodes, bonds, 1\n    return tuple(torch.as_tensor(v, device=device) for v in (x, a, mask))\n\n\nclass Ranker(nn.Module):\n    def __init__(self, n_adducts, width=128):\n        super().__init__()\n        self.adduct = nn.Embedding(n_adducts, 16)\n        self.spectrum = nn.Sequential(nn.Linear(N_BINS + 21, width), nn.GELU(), nn.Linear(width, width))\n        self.atom = nn.Linear(len(ELEMENTS) + 2, width)\n        self.self_layers = nn.ModuleList(nn.Linear(width, width) for _ in range(3))\n        self.bond_layers = nn.ModuleList(nn.ModuleList(nn.Linear(width, width, bias=False) for _ in range(N_BONDS)) for _ in range(3))\n        self.pool = nn.Linear(width * 2, width)\n        self.forward_spectrum = nn.Sequential(nn.Linear(width + 21, width), nn.GELU(), nn.Linear(width, N_BINS), nn.Softplus())\n\n    def encode_graph(self, packed):\n        nodes, bonds, mask = packed\n        h = F.gelu(self.atom(nodes)) * mask\n        for own, messages in zip(self.self_layers, self.bond_layers):\n            total = sum(bonds[:, b] @ layer(h) for b, layer in enumerate(messages))\n            h = F.gelu(own(h) + total) * mask\n        summed = h.sum(1)\n        return self.pool(torch.cat([summed / mask.sum(1).clamp_min(1), summed], dim=-1))\n\n    def context(self, meta, ids):\n        return torch.cat([meta, self.adduct(ids)], -1)\n\n    def encode_spectrum(self, spec, meta, ids):\n        return F.normalize(self.spectrum(torch.cat([spec, self.context(meta, ids)], -1)), dim=-1)\n\n    def predict_spectrum(self, graphs, meta, ids):\n        return F.normalize(self.forward_spectrum(torch.cat([graphs, self.context(meta, ids)], -1)), dim=-1)\n\n\ndef tensors(molecules, ids, device):\n    features, metas, adducts = [], [], []\n    for m in molecules:\n        meta, aid = metadata(m["row"], ids)\n        features.append(spectrum_features(m["peaks"], float(m["row"]["precursor_mz"])))\n        metas.append(meta)\n        adducts.append(aid)\n    return (torch.as_tensor(np.stack(features), device=device),\n            torch.as_tensor(np.stack(metas), device=device),\n            torch.as_tensor(adducts, device=device))\n\n\ndef loss(model, batch, ids, device, auxiliary):\n    spec, meta, aid = tensors(batch, ids, device)\n    g = model.encode_graph(pack_graphs([graph(m["smiles"]) for m in batch], device))\n    s = model.encode_spectrum(spec, meta, aid)\n    logits = s @ F.normalize(g, dim=-1).T / 0.1\n    # Same connectivity variants must not be false negatives.\n    positive = torch.tensor([[a["group"] == b["group"] for b in batch] for a in batch], device=device)\n    target = positive.float() / positive.sum(1, keepdim=True)\n    contrast = -(target * F.log_softmax(logits, dim=1)).sum(1).mean()\n    forward = (1 - (model.predict_spectrum(g, meta, aid) * spec).sum(-1)).mean()\n    return contrast + auxiliary * forward\n\n\ndef train(fit, calibration, ids, device, args, auxiliary):\n    random.seed(args.seed)\n    torch.manual_seed(args.seed)\n    model = Ranker(len(ids) + 1).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)\n    best, best_loss, history = None, math.inf, []\n    for epoch in range(args.epochs):\n        model.train()\n        order = list(fit)\n        random.shuffle(order)\n        values = []\n        for start in range(0, len(order), args.batch_size):\n            batch = order[start:start + args.batch_size]\n            if len(batch) < 2:\n                continue\n            optimizer.zero_grad(set_to_none=True)\n            value = loss(model, batch, ids, device, auxiliary)\n            if not torch.isfinite(value):\n                raise RuntimeError("nonfinite training loss")\n            value.backward()\n            nn.utils.clip_grad_norm_(model.parameters(), 1)\n            optimizer.step()\n            values.append(value.item())\n        model.eval()\n        with torch.no_grad():\n            chunks = [calibration[i:i + args.batch_size] for i in range(0, len(calibration), args.batch_size)]\n            valid = [loss(model, b, ids, device, auxiliary).item() for b in chunks if len(b) >= 2]\n        if not values or not valid or not all(math.isfinite(v) for v in valid):\n            raise ValueError("need at least two fit and calibration molecules and finite losses")\n        metric = float(np.mean(valid))\n        history.append({"epoch": epoch + 1, "train_loss": float(np.mean(values)), "calibration_loss": metric})\n        print(json.dumps(history[-1]), flush=True)\n        if metric < best_loss:\n            best, best_loss = copy.deepcopy(model.state_dict()), metric\n    model.load_state_dict(best)\n    model.eval()\n    return model, history\n\n\ndef order_scores(scores, positions):\n    return sorted(range(len(scores)), key=lambda i: (scores[i] is None, -(scores[i] or 0), positions[i]))\n\n\n@torch.no_grad()\ndef evaluate(model, pools, ids, device, batch_size):\n    rows = []\n    for qid, query in pools.items():\n        pool = query["distinct"]\n        positions = seeded_positions(qid, len(pool))\n        peaks, error = parse_spectrum(query["row"]["mzs"], query["row"]["intensities"])\n        scores = [None] * len(pool)\n        if error is None:\n            batch = [{"row": query["row"], "peaks": peaks}]\n            spec, meta, aid = tensors(batch, ids, device)\n            if spec.norm() > 0:\n                embedded = model.encode_spectrum(spec, meta, aid)\n                supported = [(i, graph(s)) for i, s in enumerate(pool)]\n                supported = [(i, g) for i, g in supported if g is not None]\n                for start in range(0, len(supported), batch_size):\n                    chunk = supported[start:start + batch_size]\n                    g = F.normalize(model.encode_graph(pack_graphs([g for _, g in chunk], device)), dim=-1)\n                    values = (embedded @ g.T).flatten().cpu().tolist()\n                    for (index, _), score in zip(chunk, values):\n                        if not math.isfinite(score):\n                            raise RuntimeError("nonfinite candidate score")\n                        scores[index] = score\n        order = order_scores(scores, positions)\n        target = pool.index(query["smiles"]) if query["smiles"] in pool else None\n        uniform = sorted(range(len(pool)), key=lambda i: positions[i])\n        rows.append({"qid": qid, "n_pool": len(pool), "n_scored": sum(s is not None for s in scores),\n                     "rank": order.index(target) + 1 if target is not None else None,\n                     "uniform_rank": uniform.index(target) + 1 if target is not None else None})\n    return rows\n\n\ndef bootstrap_difference(a, b, seed, repeats=2000):\n    if len(a) != len(b) or not a:\n        raise ValueError("paired bootstrap needs equal nonempty samples")\n    delta = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)\n    rng = np.random.default_rng(seed)\n    means = [delta[rng.integers(len(delta), size=len(delta))].mean() for _ in range(repeats)]\n    return {"difference": float(delta.mean()), "ci95": np.quantile(means, [0.025, 0.975]).tolist()}\n\n\ndef run(args):\n    device = torch.device(args.device)\n    if device.type == "cuda":\n        if not torch.cuda.is_available():\n            raise RuntimeError("CUDA is unavailable: select a Colab GPU runtime")\n        name = torch.cuda.get_device_name()\n        if args.require_t4 and "T4" not in name:\n            raise RuntimeError(f"Requested T4, allocated {name}")\n    elif args.require_t4:\n        raise ValueError("--require-t4 requires --device cuda")\n    selected = select_val(args.tsv, args.prefix, args.max_queries)\n    val_rows = selected["val_groups"]\n    val_keys = {short_key(r["inchikey"]) for r in val_rows.values() if r["inchikey"]}\n    contents = set()\n    for row in val_rows.values():\n        peaks, error = parse_spectrum(row["mzs"], row["intensities"])\n        if error is None:\n            contents.add(content_fp(peaks))\n    _, fit, calibration, _, _, audit = collect_train_molecules(args.tsv, args.max_train, set(val_rows), val_keys, contents)\n    # Missing/empty spectrum evidence cannot be a training target.\n    def usable(m):\n        return spectrum_features(m["peaks"], float(m["row"]["precursor_mz"])).any()\n    n_before = [len(fit), len(calibration)]\n    fit, calibration = [m for m in fit if usable(m)], [m for m in calibration if usable(m)]\n    ids = {adduct: i + 1 for i, adduct in enumerate(sorted({m["row"]["adduct"] for m in fit}))}\n    output = Path(args.out)\n    output.mkdir(parents=True, exist_ok=True)\n    all_rows, histories = {}, {}\n    for arm, auxiliary in (("contrastive", 0.0), ("contrastive_forward", 0.2)):\n        print(f"Training {arm} on {device}", flush=True)\n        model, histories[arm] = train(fit, calibration, ids, device, args, auxiliary)\n        torch.save({"model": model.cpu().state_dict(), "adduct_ids": ids, "args": vars(args),\n                    "scope": "restricted-domain neural retrieval pilot"}, output / f"{arm}.pt")\n        model.to(device)\n        all_rows[arm] = evaluate(model, selected["pools"], ids, device, args.batch_size)\n    metrics = {}\n    for arm, rows in all_rows.items():\n        metrics[arm] = {f"top{k}": sum(r["rank"] is not None and r["rank"] <= k for r in rows) / len(rows) for k in (1, 10, 25)}\n        metrics[arm]["unrankable"] = sum(r["n_scored"] == 0 for r in rows)\n    rows = all_rows["contrastive"]\n    metrics["uniform"] = {f"top{k}": sum(r["uniform_rank"] is not None and r["uniform_rank"] <= k for r in rows) / len(rows) for k in (1, 10, 25)}\n    hits = lambda arm: [r["rank"] is not None and r["rank"] <= 25 for r in all_rows[arm]]\n    summary = {"scope": "neural retrieval pilot; no stability validation or completion training",\n               "device": str(device), "gpu": torch.cuda.get_device_name() if device.type == "cuda" else None,\n               "torch": torch.__version__, "config": vars(args), "fit": len(fit), "calibration": len(calibration),\n               "empty_exclusions": [n_before[0] - len(fit), n_before[1] - len(calibration)],\n               "audit": dict(audit), "metrics": metrics,\n               "forward_minus_contrastive_top25": bootstrap_difference(hits("contrastive_forward"), hits("contrastive"), args.seed),\n               "contrastive_minus_uniform_top25": bootstrap_difference(hits("contrastive"),\n                   [r["uniform_rank"] is not None and r["uniform_rank"] <= 25 for r in rows], args.seed),\n               "data_sha256": {"tsv": sha256_file(args.tsv), "prefix": sha256_file(args.prefix)},\n               "script_sha256": sha256_file(__file__), "histories": histories,\n               "limitations": ["no predicted-substructure inputs or learned molecular prior",\n                               "no Mamba decoder or de novo generation",\n                               "connectivity-group disjointness; not scaffold disjointness",\n                               "restricted typed parser excludes unsupported candidates into seeded tails",\n                               "graph encoder does not represent stereochemistry",\n                               "fixed 1-Da binned FPNet-style preprocessing; not pretrained FPNet",\n                               "energy_count unavailable in benchmark, represented as unknown=0",\n                               "previously inspected validation slice; not an untouched test result",\n                               "in-batch training negatives; same-formula negatives not implemented",\n                               "no stability labels, calibration, or physical stability claims"]}\n    (output / "predictions.json").write_text(json.dumps(all_rows, indent=2, allow_nan=False))\n    (output / "summary.json").write_text(json.dumps(summary, indent=2, allow_nan=False))\n    print(json.dumps(metrics, indent=2), flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--tsv", default="data/pinned/MassSpecGym1.5.tsv")\n    parser.add_argument("--prefix", default="data/pinned/msgym_candidates_formula_prefix64.json")\n    parser.add_argument("--out", default="experiments/molecular_completion/neural_t4")\n    parser.add_argument("--device", choices=("cpu", "cuda"), default="cuda")\n    parser.add_argument("--require-t4", action="store_true")\n    parser.add_argument("--epochs", type=int, default=20)\n    parser.add_argument("--batch-size", type=int, default=32)\n    parser.add_argument("--max-train", type=int, default=5000)\n    parser.add_argument("--max-queries", type=int, default=200)\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n    if min(args.epochs, args.max_train, args.max_queries) < 1 or args.batch_size < 2:\n        parser.error("epochs/train/query caps must be positive; batch-size must be >=2")\n    run(args)\n', 'tools/test_ms2_neural_ranker.py': '"""Behavior tests for the standalone neural ranking pilot."""\nimport unittest\nimport argparse\nimport csv\nimport json\nimport tempfile\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom tools.ms2_neural_ranker import (Ranker, bootstrap_difference, evaluate,\n                                     graph, loss, metadata, order_scores,\n                                     pack_graphs, spectrum_features)\nfrom tools.ms2_neural_ranker import run\nfrom tools.ms2_spectral_rank import WANTED_COLUMNS\n\n\nclass RankerTests(unittest.TestCase):\n    def setUp(self):\n        torch.manual_seed(4)\n        torch.set_num_threads(1)\n\n    def test_peak_contract_and_errors(self):\n        vec = spectrum_features([(10, 4), (20, 1), (30, 0.001), (103, 4)], 100)\n        self.assertAlmostEqual(float(vec[10] / vec[20]), 2)\n        self.assertEqual(vec[30], 0)\n        self.assertEqual(vec[103], 0)\n        self.assertEqual(float(np.linalg.norm(spectrum_features([], 100))), 0)\n        with self.assertRaises(ValueError):\n            spectrum_features([(1, -1)], 100)\n\n    def test_energy_missing_is_distinct_from_zero(self):\n        base = {"precursor_mz": "100", "adduct": "[M+H]+"}\n        missing, _ = metadata(base, {})\n        zero, _ = metadata(dict(base, collision_energy="0"), {})\n        normalized, _ = metadata(dict(base, collision_energy="20%"), {})\n        self.assertEqual(missing[3], 0)\n        self.assertEqual(zero[3], 1)\n        self.assertEqual(normalized[3], 0)\n        self.assertEqual(missing[4], 0)\n        with self.assertRaises(ValueError):\n            metadata(dict(base, energy_count="9"), {})\n\n    def test_graph_permutation_and_padding(self):\n        model = Ranker(2).eval()\n        x, a = graph("CCO")\n        p = [2, 0, 1]\n        permuted = (x[p], a[:, p][:, :, p])\n        with torch.no_grad():\n            single = model.encode_graph(pack_graphs([(x, a)], "cpu"))[0]\n            together = model.encode_graph(pack_graphs([permuted, graph("CCCCCC")], "cpu"))[0]\n        torch.testing.assert_close(single, together, rtol=1e-5, atol=1e-5)\n\n    def test_aromatic_graphs_are_encodable(self):\n        model = Ranker(2).eval()\n        aromatic = graph("c1ccccc1")\n        self.assertIsNotNone(aromatic)\n        self.assertEqual(float(aromatic[1][3].sum()), 12)\n        with torch.no_grad():\n            encoded = model.encode_graph(pack_graphs([aromatic, graph("CCO")], "cpu"))\n        self.assertTrue(torch.isfinite(encoded).all())\n\n    def test_tail_preserves_candidates(self):\n        self.assertEqual(order_scores([None, -2, 0, None], {0: 3, 1: 2, 2: 1, 3: 0}), [2, 1, 3, 0])\n        model = Ranker(2).eval()\n        pools = {"q": {"distinct": ["CCO", "[Na+]"], "smiles": "[Na+]",\n                       "row": {"mzs": "10,20", "intensities": "0,0",\n                               "precursor_mz": "100", "adduct": "[M+H]+"}}}\n        result = evaluate(model, pools, {}, "cpu", 2)\n        self.assertEqual(len(result), 1)\n        self.assertEqual(result[0]["n_pool"], 2)\n        self.assertEqual(result[0]["n_scored"], 0)\n        self.assertIsNotNone(result[0]["rank"])\n\n    def training_check(self, device):\n        molecules = []\n        for i, smiles in enumerate(("CCO", "COC", "CCC", "CCN")):\n            molecules.append({"smiles": smiles, "group": str(i),\n                              "peaks": [(10 + 20 * i, 1), (15 + 20 * i, 0.5)],\n                              "row": {"precursor_mz": "100", "adduct": "[M+H]+"}})\n        model = Ranker(2, width=32).to(device)\n        optimizer = torch.optim.AdamW(model.parameters(), lr=0.003)\n        initial = loss(model, molecules, {"[M+H]+": 1}, device, 0.2).item()\n        for _ in range(40):\n            optimizer.zero_grad()\n            value = loss(model, molecules, {"[M+H]+": 1}, device, 0.2)\n            value.backward()\n            optimizer.step()\n        final = loss(model, molecules, {"[M+H]+": 1}, device, 0.2).item()\n        self.assertLess(final, initial * 0.5)\n\n    def test_cpu_training_learns(self):\n        self.training_check("cpu")\n\n    @unittest.skipUnless(torch.cuda.is_available(), "CUDA GPU unavailable")\n    def test_cuda_training_and_cpu_parity(self):\n        self.training_check("cuda")\n        cpu = Ranker(2).eval()\n        gpu = Ranker(2).cuda().eval()\n        gpu.load_state_dict(cpu.state_dict())\n        graphs = [graph("CCO"), graph("COC")]\n        with torch.no_grad():\n            torch.testing.assert_close(cpu.encode_graph(pack_graphs(graphs, "cpu")),\n                                       gpu.encode_graph(pack_graphs(graphs, "cuda")).cpu(), rtol=1e-4, atol=1e-4)\n\n    def test_paired_bootstrap(self):\n        result = bootstrap_difference([1, 0], [0, 0], 42)\n        self.assertEqual(result["difference"], 0.5)\n        self.assertEqual(result, bootstrap_difference([1, 0], [0, 0], 42))\n\n    def test_file_backed_training_and_reporting(self):\n        with tempfile.TemporaryDirectory() as directory:\n            root = Path(directory)\n            tsv, prefix = root / "data.tsv", root / "pools.json"\n            smiles = ("CCO", "COC", "CCC", "CCN", "CCCO", "CCOC", "CCCC", "CCCN", "CCNC", "CC(C)O", "CN")\n            with tsv.open("w") as stream:\n                writer = csv.DictWriter(stream, fieldnames=WANTED_COLUMNS, delimiter="\\t")\n                writer.writeheader()\n                for i, smi in enumerate(smiles):\n                    row = dict.fromkeys(WANTED_COLUMNS, "")\n                    row.update(identifier=str(i), mzs=f"{10 + i},{30 + i}", intensities="1,0.5",\n                               smiles=smi, inchikey=f"{i:014d}-TEST", precursor_mz="100",\n                               adduct="[M+H]+", fold="val" if i == 10 else "train")\n                    writer.writerow(row)\n            prefix.write_text(json.dumps({"CN": ["CCN", "CN", "[Na+]"]}))\n            args = argparse.Namespace(tsv=str(tsv), prefix=str(prefix), out=str(root / "output"),\n                                      device="cpu", require_t4=False, epochs=1, batch_size=4,\n                                      max_train=10, max_queries=1, seed=42)\n            run(args)\n            summary = json.loads((root / "output/summary.json").read_text())\n            predictions = json.loads((root / "output/predictions.json").read_text())\n            self.assertEqual(summary["fit"], 8)\n            self.assertEqual(summary["calibration"], 2)\n            self.assertEqual(predictions["contrastive"][0]["n_pool"], 3)\n            self.assertEqual(predictions["contrastive"][0]["n_scored"], 2)\n            self.assertTrue((root / "output/contrastive.pt").exists())\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'tools/ms2_fp_predictor.py': '"""Trained measured-spectrum to structure-fingerprint predictor + formula-pool ranking.\n\nSixth molecular-completion run: a small TRAINED spectrum->fingerprint model\nevaluated on the IDENTICAL 200 val formula pools as tools/ms2_spectral_rank.py.\nNothing is tuned on val; test-fold rows are scanned-but-unused.\n\nRepresentation (fixed a priori, never selected on val):\n  spectrum: 1.0 Da bins over [0, 2000) = 2000 dims; per-bin max intensity,\n    sqrt transform, per-spectrum L2 normalization; sparse CSR (bounded,\n    no huge dense tensors).\n  fingerprint: fixed stable-hash atom / typed-bond / length-2-path counts,\n    FP_DIM=256, hashed with zlib.crc32 (deterministic, never Python hash()),\n    permutation-invariant by construction (multiset counts only). This is a\n    local fixed hash fingerprint, NOT a standard Morgan/ECFP or official\n    benchmark fingerprint; no such claim is made. Coarse typed-edge counts\n    alone alias isomers: collisions are measured and the ceiling admitted.\n  model: sparse multi-output Ridge (alpha=1.0, solver lsqr, tol 1e-3,\n    max_iter 200, fit_intercept False), fixed a priori, fit on FIT molecules\n    only. No hyperparameter search on val; no refit on calibration.\n  scoring: cosine between L2-normalized predicted fp and L2-normalized\n    candidate fp. Missing candidate fps rank in seeded tail, retained.\n  prior: untrained TRAIN-fit-mean fingerprint (fit-only), same scoring,\n    isolates spectral signal from fingerprint popularity.\n  cheap baseline: massresid via existing standardize_smiles (measured\n    parent_mass as given query evidence), identical pools.\n\nIdentity: exact provider-canonical SMILES string for dedup/grouping/audit\nplus InChIKey-14 connectivity groups for split disjointness. No RDKit.\nQuery graph fingerprints, formula labels, IDs, target-in-pool flags and\nsupplier order are never model inputs (query spectra only).\n\nLeakage rules (asserted, unit-tested):\n  * train/val connectivity overlap (SMILES or InChIKey-14) excludes the\n    train molecule (counted); residual overlap aborts;\n  * duplicate exact canonical spectrum content (content_fp) excluded:\n    within-fit first-wins, calibration duplicating fit excluded, train\n    duplicating any val query content excluded (all counted);\n  * same-molecule spectra never straddle fit/calibration (one spectrum per\n    molecule; split by connectivity group);\n  * val/test labels never influence training, model selection or\n    calibration; test rows pass through the scanner unused.\n\nCalibration: fixed 90% precision gate on (margin, pool-size cap), thresholds\nchosen on held-out TRAIN calibration formula-pool queries only. If too few\neligible calibration queries or no threshold meets target, fail closed\n(abstain-all) with reason. Calibration uses the identical ranking/pool\ndomain as val (full pools, seeded ties, missing-fp tail).\n\nCPU stdlib + scientific baseline only. GPU and Rust parity NOT APPLICABLE.\n"""\n\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\n\n# Bound BLAS/OpenMP thread pools best-effort before heavy imports (observed\n# cpu/wall is still reported; no strict single-threaded claim).\nfor _k in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):\n    os.environ.setdefault(_k, "1")\n\nimport resource\nimport sys\nimport time\nimport unittest\nimport zlib\nfrom collections import Counter\n\nsys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))\n\nfrom tools.ms2_spectral_rank import (  # noqa: E402\n    candidate_residuals,\n    content_fp,\n    parse_spectrum,\n    seeded_positions,\n    sha256_file,\n    short_key,\n)\nfrom tools.ms2_msgym_corpus import iter_json_entries  # noqa: E402\n\n# --------------------------------------------------------------------------\n# Fixed a priori representation + hyperparameters (never tuned on val).\n# --------------------------------------------------------------------------\n\nSPEC_BIN_WIDTH = 1.0\nSPEC_MZ_MIN = 0.0\nSPEC_MZ_MAX = 2000.0\nN_SPEC = int((SPEC_MZ_MAX - SPEC_MZ_MIN) / SPEC_BIN_WIDTH)  # 2000\nFP_DIM = 256\nFP_MAX_PATH_BONDS = 2  # atoms (0), bonds (1), length-2 paths (2 bonds)\nRIDGE_ALPHA = 1.0\nRIDGE_TOL = 1e-3\nRIDGE_MAX_ITER = 200\nRIDGE_SOLVER = "lsqr"\nRIDGE_FIT_INTERCEPT = False\nMAX_TRAIN_MOLECULES = 5000\nFIT_GROUP_FRAC = 0.8\nMIN_CALIB_ELIGIBLE = 20\nCALIB_TARGET_PRECISION = 0.90\nTAU_GRID = (0.0, 0.005, 0.01, 0.02, 0.05, 0.10, 0.20)\nKMAX_GRID = (50, 100, 200, 500, 10 ** 9)\nSHUFFLE_DOMAIN = "qid-crc32"\nARMS = ("uniform", "massresid", "predictor", "prior")\n\n\ndef progress(msg):\n    sys.stderr.write(f"# progress {msg}\\n")\n    sys.stderr.flush()\n\n\n# --------------------------------------------------------------------------\n# Spectrum features: 1 Da bins + sqrt + L2 (per-sample only, no fitting).\n# --------------------------------------------------------------------------\n\ndef bin_spectrum_1da(peaks):\n    """Bin to {bin: max_intensity} with fixed 1 Da bounds. Returns (vec, dropped)."""\n    vec = {}\n    dropped = 0\n    for mz, it in peaks:\n        if not (SPEC_MZ_MIN <= mz < SPEC_MZ_MAX):\n            dropped += 1\n            continue\n        b = int((mz - SPEC_MZ_MIN) / SPEC_BIN_WIDTH)\n        if it > vec.get(b, 0.0):\n            vec[b] = it\n    return vec, dropped\n\n\ndef featurize_spectrum(peaks):\n    """Fixed per-spectrum features: max-bin, sqrt, L2-normalize.\n\n    Returns (feat_dict, norm_before). No vocabulary or statistics are fit;\n    each spectrum is processed independently (fit-only claim trivially holds).\n    """\n    vec, dropped = bin_spectrum_1da(peaks)\n    sq = {b: math.sqrt(v) for b, v in vec.items() if v > 0.0}\n    norm = math.sqrt(sum(v * v for v in sq.values()))\n    if norm > 0.0:\n        sq = {b: v / norm for b, v in sq.items()}\n    return sq, norm, dropped\n\n\n# --------------------------------------------------------------------------\n# Fingerprint: stable-hash atom/bond/path counts, permutation-invariant.\n# --------------------------------------------------------------------------\n\ndef _atom_key(el, h, v):\n    return f"A:{el}:{h}:{v}"\n\n\ndef fp_from_typed(atom_types, edges):\n    """Counted hash fingerprint (length FP_DIM), L2-normalized.\n\n    atom_types: tuple of (element, h, valence); edges: tuple of (a, b, order).\n    Multiset counts only, so any atom-index permutation yields identical\n    output. Hashing uses zlib.crc32 (stable across runs), never Python-hash.\n    """\n    counts = [0.0] * FP_DIM\n    akeys = [_atom_key(el, h, v) for el, h, v in atom_types]\n    for k in akeys:\n        counts[zlib.crc32(k.encode()) % FP_DIM] += 1.0\n    for a, b, o in edges:\n        k1, k2 = akeys[a], akeys[b]\n        if k1 > k2:\n            k1, k2 = k2, k1\n        key = f"B:{k1}|{k2}:{o}"\n        counts[zlib.crc32(key.encode()) % FP_DIM] += 1.0\n    n = len(atom_types)\n    adj = [[] for _ in range(n)]\n    for a, b, o in edges:\n        adj[a].append((b, o))\n        adj[b].append((a, o))\n    for c in range(n):\n        neigh = adj[c]\n        for i in range(len(neigh)):\n            for j in range(i + 1, len(neigh)):\n                (a, oa) = neigh[i]\n                (d, ob) = neigh[j]\n                da = f"{akeys[a]}:{oa}"\n                db = f"{akeys[d]}:{ob}"\n                if da > db:\n                    da, db = db, da\n                key = f"P:{akeys[c]}|{da}|{db}"\n                counts[zlib.crc32(key.encode()) % FP_DIM] += 1.0\n    norm = math.sqrt(sum(c * c for c in counts))\n    if norm > 0.0:\n        counts = [c / norm for c in counts]\n    return counts, norm\n\n\ndef fp_from_smiles(smi):\n    """Fingerprint from a candidate/train SMILES via the typed graph parser.\n\n    Returns (vec_or_None, norm_or_0, reason_or_None). Unparseable chemistry\n    yields (None, 0.0, reason); callers retain the candidate and count it.\n    """\n    from tools.ms2_msgym_corpus import standardize_smiles\n    try:\n        rec, reason = standardize_smiles(smi, "fp-pool", smi)\n    except Exception:\n        return None, 0.0, "exception"\n    if rec is None:\n        return None, 0.0, reason\n    vec, norm = fp_from_typed(rec["atom_types"], rec["edges"])\n    return vec, norm, None\n\n\ndef cosine_dense(a, na, b, nb):\n    if na <= 0.0 or nb <= 0.0:\n        return 0.0\n    return sum(x * y for x, y in zip(a, b))\n\n\n# --------------------------------------------------------------------------\n# Ranking (query spectra only; never the query graph/fingerprint/label).\n# --------------------------------------------------------------------------\n\ndef score_candidates_fp(pred_vec, pred_norm, pool_fps):\n    """Cosine of predicted fp vs each candidate fp. Missing -> None, kept.\n\n    Non-finite predictions or scores (NaN/inf/out-of-range) yield None\n    (unscored, ranked last) rather than propagating NaN into ranking.\n    """\n    pred_ok = (\n        pred_vec is not None\n        and isinstance(pred_norm, (int, float))\n        and not isinstance(pred_norm, bool)\n        and math.isfinite(float(pred_norm))\n        and float(pred_norm) > 0.0\n        and len(pred_vec) == FP_DIM\n        and all(isinstance(v, (int, float)) and not isinstance(v, bool)\n                and math.isfinite(float(v)) for v in pred_vec)\n        and all(abs(float(v)) <= 10.0 for v in pred_vec)\n    )\n    out = []\n    for idx, (cvec, cnorm) in enumerate(pool_fps):\n        if not pred_ok or cvec is None or not math.isfinite(cnorm) \\\n                or cnorm <= 0.0:\n            out.append({"idx": idx, "score": None,\n                        "parseable": cvec is not None})\n            continue\n        s = cosine_dense(pred_vec, pred_norm, cvec, cnorm)\n        if not isinstance(s, float) or not math.isfinite(s):\n            out.append({"idx": idx, "score": None, "parseable": True})\n        else:\n            out.append({"idx": idx, "score": s, "parseable": True})\n    return out\n\n\ndef rank_by_score(scored, pos_of):\n    return sorted(range(len(scored)),\n                  key=lambda i: ((0.0, -scored[i]["score"])\n                                 if scored[i]["score"] is not None\n                                 else (1.0, 0.0),\n                                 pos_of[i]))\n\n\ndef rank_uniform(n, pos_of):\n    return sorted(range(n), key=lambda i: pos_of[i])\n\n\ndef rank_massresid_order(residuals, pos_of):\n    return sorted(range(len(residuals)),\n                  key=lambda i: ((0.0, residuals[i])\n                                 if residuals[i] is not None\n                                 else (1.0, 0.0),\n                                 pos_of[i]))\n\n\ndef margin_of(scored_order_scores):\n    """Legacy wrapper: finite-scored gap; kept for compatibility.\n\n    Returns margin over actual finite scores only (None/NaN/inf excluded).\n    Single finite score -> 0.0 conservative; no finite scores -> None.\n    """\n    margin, _ = margin_from_finite_scores(scored_order_scores)\n    return margin\n\n\ndef finite_scored_values(scored_order_scores):\n    """Actual finite scored values only (excludes None/NaN/inf)."""\n    vals = []\n    for v in scored_order_scores or []:\n        if isinstance(v, bool):\n            continue\n        if isinstance(v, (int, float)) and math.isfinite(float(v)):\n            vals.append(float(v))\n    return vals\n\n\ndef margin_from_finite_scores(scored_order_scores):\n    """Confidence gap over the actual finite scored order.\n\n    Missing/unscored candidates (None) rank last and are EXCLUDED from the\n    gap: the margin is top1-top2 over finite scores only. Single finite\n    score -> (0.0, \'single_scored\') conservatively (no observed gap, so\n    zero confidence). No finite scores -> (None, \'no_finite_scores\').\n    Non-finite gap -> (None, \'non_finite_margin\').\n    """\n    vals = finite_scored_values(scored_order_scores)\n    if not vals:\n        return None, "no_finite_scores"\n    if len(vals) == 1:\n        return 0.0, "single_scored"\n    vals.sort(reverse=True)\n    gap = vals[0] - vals[1]\n    if not math.isfinite(gap):\n        return None, "non_finite_margin"\n    return gap, "ok"\n\n\ndef spectrum_has_usable_features(peaks):\n    """Nonempty usable spectrum features (inference input check)."""\n    if not peaks:\n        return False\n    try:\n        feat, norm, _ = featurize_spectrum(peaks)\n    except Exception:\n        return False\n    return bool(feat) and isinstance(norm, (int, float)) \\\n        and not isinstance(norm, bool) \\\n        and math.isfinite(float(norm)) and float(norm) > 0.0\n\n\ndef prediction_is_usable(pred_vec, pred_norm):\n    """Finite nonzero prediction (never accept zero/NaN/out-of-range)."""\n    if pred_vec is None or pred_norm is None:\n        return False\n    if isinstance(pred_norm, bool):\n        return False\n    if not isinstance(pred_norm, (int, float)):\n        return False\n    pred_norm = float(pred_norm)\n    if not math.isfinite(pred_norm) or pred_norm <= 0.0:\n        return False\n    if len(pred_vec) != FP_DIM:\n        return False\n    for v in pred_vec:\n        if isinstance(v, bool) or not isinstance(v, (int, float)):\n            return False\n        fv = float(v)\n        if not math.isfinite(fv) or abs(fv) > 10.0:\n            return False\n    tot = sum(float(v) * float(v) for v in pred_vec)\n    return math.isfinite(tot) and tot > 0.0\n\n\ndef capability_for_query(peaks, pred_vec, pred_norm, scored):\n    """Shared inference-capability predicate for calibration AND val.\n\n    Requires: nonempty usable spectrum features, finite nonzero\n    prediction, >=1 finite candidate score, finite margin. Deliberately\n    INDEPENDENT of true target parseability/identity/hit (those are not\n    inference inputs; target-parseable is an EVALUATION-only subgroup).\n    Returns (rankable_bool, reason_str).\n    """\n    if not spectrum_has_usable_features(peaks):\n        return False, "no_features"\n    if not prediction_is_usable(pred_vec, pred_norm):\n        return False, "predictor_unavailable"\n    scores = [s.get("score") for s in (scored or [])]\n    n_finite = len(finite_scored_values(scores))\n    if n_finite == 0:\n        return False, "predictor_unavailable"\n    margin, mstatus = margin_from_finite_scores(scores)\n    if margin is None or not math.isfinite(margin):\n        return False, "predictor_unavailable"\n    return True, ("ok" if mstatus == "ok" else f"ok:{mstatus}")\n\n\ndef target_parse_info(smiles):\n    """Evaluation-only target fingerprint parseability (never an input).\n\n    Returns (parseable_bool, status_str, reason_or_None).\n    """\n    try:\n        vec, norm, reason = fp_from_smiles(smiles)\n    except Exception:\n        return False, "error", "exception"\n    if vec is None:\n        return False, "unsupported", reason\n    return True, "ok", None\n\n\ndef evaluate_pool(pool, target_smiles, pred_scores, prior_scores, residuals,\n                  pos_of):\n    """Ranks/hits for all four arms on one identical full pool."""\n    n = len(pool)\n    tgt = pool.index(target_smiles) if target_smiles in pool else None\n    ou = rank_uniform(n, pos_of)\n    om = rank_massresid_order(residuals, pos_of)\n    op = rank_by_score(pred_scores, pos_of)\n    orr = rank_by_score(prior_scores, pos_of)\n\n    def rank_of(order):\n        return None if tgt is None else order.index(tgt) + 1\n\n    ranks = {"uniform": rank_of(ou), "massresid": rank_of(om),\n             "predictor": rank_of(op), "prior": rank_of(orr)}\n\n    def hits(rank):\n        if rank is None:\n            return {1: None, 3: None, 10: None}\n        return {1: rank <= 1, 3: rank <= min(3, n), 10: rank <= min(10, n)}\n\n    margin_pred, margin_status = margin_from_finite_scores(\n        [s["score"] for s in pred_scores])\n    n_scored = len(finite_scored_values([s["score"] for s in pred_scores]))\n    return {"in_pool": tgt is not None, "n_pool": n,\n            "ranks": ranks, "hits": {a: hits(r) for a, r in ranks.items()},\n            "margin_pred": margin_pred,\n            "margin_status": margin_status,\n            "n_scored_finite": n_scored,\n            "n_parseable": sum(1 for s in pred_scores if s["parseable"])}\n\n\n# --------------------------------------------------------------------------\n# Data selection (val identical to spectral run; train capped + grouped).\n# --------------------------------------------------------------------------\n\ndef load_wanted_rows(tsv_path):\n    from tools.ms2_spectral_rank import load_tsv_rows\n    return load_tsv_rows(tsv_path)\n\n\ndef select_val(tsv_path, prefix_path, max_queries):\n    """EXACT same selection as spectral run: file-order prefix entries\n    joining first-row-per-SMILES val groups; distinct pools deduped in\n    supplied order; qids VAL-%04d in join order."""\n    val_groups = {}\n    train_smiles_all, train_keys_all = set(), set()\n    val_bias = Counter()\n    n_rows = n_test = 0\n    for row in load_wanted_rows(tsv_path):\n        n_rows += 1\n        if row["fold"] == "test":\n            n_test += 1\n            continue\n        if row["fold"] == "train":\n            train_smiles_all.add(row["smiles"])\n            if row["inchikey"]:\n                train_keys_all.add(short_key(row["inchikey"]))\n        elif row["fold"] == "val":\n            val_bias[f"adduct:{row[\'adduct\']}"] += 1\n            val_bias[f"inst:{row[\'instrument_type\'] or \'missing\'}"] += 1\n            if row["smiles"] not in val_groups:\n                val_groups[row["smiles"]] = row\n    entries = [(q, c) for q, c in\n               iter_json_entries(prefix_path, max_queries * 40)]\n    joined = [(q, c) for q, c in entries if q in val_groups][:max_queries]\n    if not joined:\n        raise ValueError("no prefix entries join to val rows")\n    pools = {}\n    for qi, (q, cands) in enumerate(joined):\n        qid = f"VAL-{qi:04d}"\n        seen, distinct = set(), []\n        for s in cands:\n            if s not in seen:\n                seen.add(s)\n                distinct.append(s)\n        pools[qid] = {"smiles": q, "supplied": list(cands),\n                      "distinct": distinct, "row": val_groups[q]}\n    return {"pools": pools, "val_groups": val_groups, "entries": entries,\n            "joined": joined, "n_rows": n_rows, "n_test": n_test,\n            "val_bias": val_bias,\n            "train_smiles_all": train_smiles_all,\n            "train_keys_all": train_keys_all}\n\n\ndef collect_train_molecules(tsv_path, cap, val_smiles, val_keys, val_content):\n    """Capped train molecules, one spectrum per SMILES (file order).\n\n    Returns (molecules_in_order, info). molecules carry row, inchikey,\n    group, content_fp, peaks. Spectrum-unparseable rows are marked excluded\n    here (counted). Test rows are skipped (counted).\n    """\n    from tools.ms2_spectral_rank import load_tsv_rows\n    first = {}\n    order = []\n    info = Counter()\n    for row in load_tsv_rows(tsv_path):\n        if row["fold"] == "test":\n            info["n_test_skipped"] += 1\n            continue\n        if row["fold"] != "train":\n            continue\n        info["n_train_rows_scanned"] += 1\n        smi = row["smiles"]\n        if smi not in first:\n            first[smi] = row\n            order.append(smi)\n    molecules = []\n    for smi in order:\n        row = first[smi]\n        peaks, err = parse_spectrum(row["mzs"], row["intensities"])\n        if err is not None:\n            info[f"train_spec_excl:{err}"] += 1\n            molecules.append({"smiles": smi, "row": row, "status": "excluded",\n                              "reason": f"spec:{err}", "group": None,\n                              "fp_content": None, "peaks": None})\n            continue\n        fp = content_fp(peaks)\n        key = short_key(row["inchikey"]) if row["inchikey"] else ""\n        if not row["inchikey"]:\n            info["n_train_missing_key"] += 1\n            molecules.append({"smiles": smi, "row": row, "status": "excluded",\n                              "reason": "missing_connectivity_key",\n                              "group": None, "fp_content": fp, "peaks": peaks})\n            continue\n        molecules.append({"smiles": smi, "row": row, "status": "candidate",\n                          "reason": None, "group": key, "fp_content": fp,\n                          "peaks": peaks})\n    # Group-order cap: groups in first-appearance order until cap molecules.\n    cand = [m for m in molecules if m["status"] == "candidate"]\n    gorder, gmembers = [], {}\n    for m in cand:\n        if m["group"] not in gmembers:\n            gmembers[m["group"]] = []\n            gorder.append(m["group"])\n        gmembers[m["group"]].append(m)\n    capped_groups, capped = [], []\n    for g in gorder:\n        if len(capped) + len(gmembers[g]) > cap and capped:\n            info["n_groups_truncated_by_cap"] += 1\n            break\n        capped_groups.append(g)\n        capped.extend(gmembers[g])\n        if len(capped) >= cap:\n            break\n    capped_set = {id(m) for m in capped}\n    for m in cand:\n        if id(m) not in capped_set:\n            m["status"] = "excluded"\n            m["reason"] = "over_cap"\n            info["n_over_cap"] += 1\n    info["n_capped_molecules"] = len(capped)\n    info["n_capped_groups"] = len(capped_groups)\n    # Split capped groups: first FIT_GROUP_FRAC -> fit, rest -> calibration.\n    n_fit_g = int(len(capped_groups) * FIT_GROUP_FRAC)\n    fit_groups = set(capped_groups[:n_fit_g])\n    calib_groups = set(capped_groups[n_fit_g:])\n    for m in capped:\n        m["split"] = "fit" if m["group"] in fit_groups else "calibration"\n    info["n_fit_groups"] = len(fit_groups)\n    info["n_calib_groups"] = len(calib_groups)\n    # Exclusions on capped set: val overlap, val content, unsupported chem,\n    # content duplicates (fit first-wins; calib vs fit; val content).\n    for m in capped:\n        if m["smiles"] in val_smiles or m["group"] in val_keys:\n            m["status"] = "excluded"\n            m["reason"] = "train_val_overlap"\n            info["n_train_val_overlap_excluded"] += 1\n        elif m["fp_content"] in val_content:\n            m["status"] = "excluded"\n            m["reason"] = "val_content_duplicate"\n            info["n_val_content_dup_excluded"] += 1\n    # Unsupported chemistry (target fp needed for Y): standardize now.\n    for m in capped:\n        if m["status"] != "candidate":\n            continue\n        vec, norm, reason = fp_from_smiles(m["smiles"])\n        if vec is None:\n            m["status"] = "excluded"\n            m["reason"] = f"chem:{reason}"\n            info[f"train_chem_excl:{reason.split(\':\')[0]}"] += 1\n        else:\n            m["fp_target"] = vec\n            m["fp_target_norm"] = norm\n    # Content dedup: fit first-wins, then calib-vs-fit.\n    seen_fit = set()\n    for m in [x for x in capped if x.get("split") == "fit"\n              and x["status"] == "candidate"]:\n        if m["fp_content"] in seen_fit:\n            m["status"] = "excluded"\n            m["reason"] = "fit_content_duplicate"\n            info["n_fit_content_dup"] += 1\n        else:\n            seen_fit.add(m["fp_content"])\n    for m in [x for x in capped if x.get("split") == "calibration"\n              and x["status"] == "candidate"]:\n        if m["fp_content"] in seen_fit:\n            m["status"] = "excluded"\n            m["reason"] = "calib_content_duplicate_fit"\n            info["n_calib_content_dup_fit"] += 1\n            continue\n        seen_fit.add(m["fp_content"])\n    fit = [m for m in capped if m.get("split") == "fit"\n           and m["status"] == "candidate"]\n    calib = [m for m in capped if m.get("split") == "calibration"\n             and m["status"] == "candidate"]\n    # Enforce: no shared groups, no shared content, no val overlap remain.\n    fit_groups_final = {m["group"] for m in fit}\n    calib_groups_final = {m["group"] for m in calib}\n    assert not (fit_groups_final & calib_groups_final), "group straddle"\n    assert not (fit_groups_final & set(val_keys)), "fit/val key overlap"\n    assert not (calib_groups_final & set(val_keys)), "calib/val key overlap"\n    assert not ({m["smiles"] for m in fit} & set(val_smiles)), "fit/val smi"\n    assert not ({m["smiles"] for m in calib} & set(val_smiles)), "cal/val smi"\n    return molecules, fit, calib, fit_groups_final, calib_groups_final, info\n\n\ndef build_matrices(fit):\n    import numpy as np\n    import scipy.sparse as sp\n    rows, cols, data = [], [], []\n    Y = np.zeros((len(fit), FP_DIM), dtype=np.float64)\n    for i, m in enumerate(fit):\n        feat, _, _ = featurize_spectrum(m["peaks"])\n        for b, v in feat.items():\n            rows.append(i)\n            cols.append(b)\n            data.append(v)\n        Y[i, :] = m["fp_target"]\n    X = sp.csr_matrix((data, (rows, cols)), shape=(len(fit), N_SPEC))\n    return X, Y\n\n\ndef train_ridge(X, Y):\n    from sklearn.linear_model import Ridge\n    model = Ridge(alpha=RIDGE_ALPHA, solver=RIDGE_SOLVER, tol=RIDGE_TOL,\n                  max_iter=RIDGE_MAX_ITER,\n                  fit_intercept=RIDGE_FIT_INTERCEPT, random_state=0)\n    model.fit(X, Y)\n    return model\n\n\ndef predict_fp(model, peaks):\n    import numpy as np\n    feat, norm, _ = featurize_spectrum(peaks)\n    if norm <= 0.0 or not feat:\n        return [0.0] * FP_DIM, 0.0\n    import scipy.sparse as sp\n    X = sp.csr_matrix(([v for v in feat.values()],\n                       ([0] * len(feat), list(feat.keys()))),\n                      shape=(1, N_SPEC))\n    pred = np.asarray(model.predict(X))[0].tolist()\n    n = math.sqrt(sum(v * v for v in pred))\n    if n > 0.0:\n        pred = [v / n for v in pred]\n    return pred, n\n\n\ndef mean_prior(fit):\n    import numpy as np\n    Y = np.array([m["fp_target"] for m in fit], dtype=np.float64)\n    mean = Y.mean(axis=0)\n    n = float(np.linalg.norm(mean))\n    if n > 0.0:\n        mean = mean / n\n    return mean.tolist(), n\n\n\n# --------------------------------------------------------------------------\n# Calibration (held-out TRAIN pools only; fail closed).\n# --------------------------------------------------------------------------\n\ndef calibrate_gate(calib_rows):\n    """Choose (tau, kmax) on calibration rankable rows only.\n\n    calib_rows: list of dicts with rankable (capability predicate),\n    hit_top1 (True/False/None), margin_pred, n_pool. Rows with hit None\n    (absent target) count as incorrect when predicted. rankable MUST be\n    the shared inference-capability predicate (usable features, finite\n    nonzero prediction, >=1 finite score, finite margin) and MUST NOT\n    depend on target parseability/identity/hit. Returns gate dict with\n    tau=None (strict-JSON null) when abstaining; fail closed (predict\n    nothing) when too few rankable or no threshold meets target.\n    """\n    elig = [r for r in calib_rows if r.get("rankable") is True\n            and isinstance(r.get("margin_pred"), (int, float))\n            and not isinstance(r.get("margin_pred"), bool)\n            and math.isfinite(float(r["margin_pred"]))]\n    if len(elig) < MIN_CALIB_ELIGIBLE:\n        return {"tau": None, "kmax": 0, "status": "abstain_all",\n                "reason": f"insufficient_eligible:{len(elig)}",\n                "n_eligible": len(elig), "table": []}\n    table = []\n    for tau in TAU_GRID:\n        for kmax in KMAX_GRID:\n            pred = [r for r in elig if r["margin_pred"] >= tau\n                    and r["n_pool"] <= kmax]\n            if not pred:\n                table.append({"tau": tau, "kmax": kmax, "predicted": 0,\n                              "precision": None, "coverage": 0.0})\n                continue\n            good = sum(1 for r in pred if r.get("hit_top1") is True)\n            prec = good / len(pred)\n            table.append({"tau": tau, "kmax": kmax, "predicted": len(pred),\n                          "precision": prec,\n                          "coverage": len(pred) / len(elig)})\n    feas = [t for t in table if t["precision"] is not None\n            and t["precision"] >= CALIB_TARGET_PRECISION]\n    if not feas:\n        return {"tau": None, "kmax": 0, "status": "abstain_all",\n                "reason": "no_threshold_meets_target", "n_eligible": len(elig),\n                "table": table}\n    feas.sort(key=lambda t: (-t["predicted"], -t["tau"], t["kmax"]))\n    best = feas[0]\n    return {"tau": best["tau"], "kmax": best["kmax"], "status": "calibrated",\n            "reason": None, "n_eligible": len(elig), "table": table,\n            "best": best}\n\n\ndef gate_accepts(gate, margin_pred, n_pool):\n    """Fail-closed acceptance: False when gate abstains or margin unusable."""\n    if gate.get("status") != "calibrated":\n        return False\n    tau = gate.get("tau")\n    kmax = gate.get("kmax")\n    if tau is None or isinstance(tau, bool):\n        return False\n    if not isinstance(margin_pred, (int, float)) \\\n            or isinstance(margin_pred, bool):\n        return False\n    if not math.isfinite(float(margin_pred)) \\\n            or not math.isfinite(float(tau)):\n        return False\n    if not isinstance(n_pool, int):\n        return False\n    return float(margin_pred) >= float(tau) and n_pool <= kmax\n\n\n# --------------------------------------------------------------------------\n# Summaries.\n# --------------------------------------------------------------------------\n\ndef summarize_val(rows, label):\n    out = {"label": label, "n_selected": len(rows),\n           "n_complete": sum(1 for r in rows if r.get("status") == "complete"),\n           "n_rankable": sum(1 for r in rows if r.get("rankable") is True),\n           "n_accepted": sum(1 for r in rows if r.get("accepted") is True),\n           "n_model_failure": sum(\n               1 for r in rows if r.get("status") == "complete"\n               and r.get("rankable") is not True),\n           "rankable_reason_counts": dict(Counter(\n               str(r.get("rankable_reason", "?")) for r in rows\n               if r.get("status") == "complete")),\n           "status_counts": dict(Counter(r.get("status", "?") for r in rows)),\n           "pool_recall_all": (sum(1 for r in rows if r.get("in_pool"))\n                               / len(rows) if rows else None)}\n    subsets = (("all", lambda r: True),\n               ("eligible_complete",\n                lambda r: r.get("status") == "complete" and r["in_pool"]),\n               ("capability_any_candidate",\n                lambda r: r.get("status") == "complete" and r["in_pool"]\n                and (r.get("n_parseable") or 0) > 0),\n               ("eligible_parseable",\n                lambda r: r.get("status") == "complete" and r["in_pool"]\n                and r.get("target_parseable") is True),\n               ("accepted",\n                lambda r: r.get("accepted") is True and r["in_pool"]))\n    for name, sel in subsets:\n        sub = [r for r in rows if sel(r)]\n        m = {"n": len(sub)}\n        for arm in ARMS:\n            for k in (1, 3, 10):\n                vals = [r["hits"][arm][k] for r in sub\n                        if r["hits"].get(arm, {}).get(k) is not None]\n                m[f"{arm}@top{k}"] = (sum(vals) / len(vals) if vals else None)\n                m[f"{arm}@top{k}_den"] = len(vals)\n        pred = [r for r in sub if r.get("accepted")]\n        good = [r for r in pred if r["hits"].get("predictor", {}).get(1)]\n        m["predicted"] = len(pred)\n        m["coverage"] = len(pred) / len(rows) if rows else 0.0\n        m["abstention_precision"] = (len(good) / len(pred) if pred else None)\n        out[name] = m\n    return out\n\n\ndef empty_hits():\n    return {arm: {1: None, 3: None, 10: None} for arm in ARMS}\n\n\n# --------------------------------------------------------------------------\n# Experiment driver.\n# --------------------------------------------------------------------------\n\ndef run_experiment(tsv_path, prefix_path, max_queries, max_train, out_dir):\n    import numpy as np\n    t0 = time.perf_counter()\n    c0 = time.process_time()\n    progress("select-val")\n    sel = select_val(tsv_path, prefix_path, max_queries)\n    pools = sel["pools"]\n    # Val query spectra content (for train content-leak exclusion).\n    val_content, val_smiles, val_keys = set(), [], set()\n    val_qinfo = {}\n    for qid, v in pools.items():\n        peaks, err = parse_spectrum(v["row"]["mzs"], v["row"]["intensities"])\n        if err is None:\n            val_content.add(content_fp(peaks))\n            val_qinfo[qid] = peaks\n        else:\n            val_qinfo[qid] = None\n        val_smiles.append(v["smiles"])\n        if v["row"]["inchikey"]:\n            val_keys.add(short_key(v["row"]["inchikey"]))\n    progress(f"collect-train cap={max_train}")\n    molecules, fit, calib, fit_g, calib_g, tinfo = collect_train_molecules(\n        tsv_path, max_train, set(val_smiles), val_keys, val_content)\n    progress(f"train fit={len(fit)} calib={len(calib)}")\n    t_train0 = time.perf_counter()\n    X_fit, Y_fit = build_matrices(fit)\n    model = train_ridge(X_fit, Y_fit)\n    t_train1 = time.perf_counter()\n    prior_vec, prior_norm = mean_prior(fit)\n    # Candidate fingerprint cache (shared across calib + val).\n    fp_cache = {}\n\n    def pool_fps(pool):\n        out = []\n        for smi in pool:\n            if smi not in fp_cache:\n                fp_cache[smi] = fp_from_smiles(smi)\n            vec, norm, _ = fp_cache[smi]\n            out.append((vec, norm))\n        return out\n\n    # Calibration: held-out TRAIN molecules with prefix pools.\n    progress("calibration-pools")\n    prefix_index = {}\n    for q, c in iter_json_entries(prefix_path, 1 << 30):\n        if q not in prefix_index:\n            prefix_index[q] = c\n    calib_rows = []\n    for i, m in enumerate(calib):\n        qid = f"CAL-{i:04d}"\n        cands = prefix_index.get(m["smiles"])\n        if cands is None:\n            calib_rows.append({"qid": qid, "smiles": m["smiles"],\n                               "status": "no_pool_in_prefix",\n                               "rankable": False,\n                               "rankable_reason": "no_pool",\n                               "has_features": True,\n                               "pred_norm": None,\n                               "n_scored_finite": 0,\n                               "target_parseable": True,\n                               "target_fp_status": "ok",\n                               "in_pool": None,\n                               "n_pool": 0, "n_parseable": 0,\n                               "margin_pred": None,\n                               "margin_status": "no_finite_scores",\n                               "hit_top1": None,\n                               "hits": empty_hits()})\n            continue\n        seen, distinct = set(), []\n        for s in cands:\n            if s not in seen:\n                seen.add(s)\n                distinct.append(s)\n        pfps = pool_fps(distinct)\n        pred, pn = predict_fp(model, m["peaks"])\n        pscores = score_candidates_fp(pred, pn, pfps)\n        prscores = score_candidates_fp(prior_vec, prior_norm, pfps)\n        try:\n            observed = m["row"]["parent_mass"]\n        except KeyError:\n            observed = ""\n        residuals, _ = candidate_residuals(distinct, observed)\n        pos = seeded_positions(qid, len(distinct))\n        ev = evaluate_pool(distinct, m["smiles"], pscores, prscores,\n                           residuals, pos)\n        # Capability (inference-only): never uses target parseability.\n        rankable, rreason = capability_for_query(\n            m["peaks"], pred, pn, pscores)\n        tparse, tstatus, _ = target_parse_info(m["smiles"])\n        calib_rows.append({"qid": qid, "smiles": m["smiles"],\n                           "status": "complete", "rankable": rankable,\n                           "rankable_reason": rreason,\n                           "has_features": spectrum_has_usable_features(\n                               m["peaks"]),\n                           "pred_norm": pn,\n                           "n_scored_finite": ev["n_scored_finite"],\n                           "target_parseable": tparse,\n                           "target_fp_status": tstatus,\n                           "in_pool": ev["in_pool"], "n_pool": ev["n_pool"],\n                           "n_parseable": ev["n_parseable"],\n                           "margin_pred": ev["margin_pred"],\n                           "margin_status": ev["margin_status"],\n                           "hit_top1": ev["hits"]["predictor"][1],\n                           "hits": ev["hits"], "ranks": ev["ranks"]})\n    gate = calibrate_gate(calib_rows)\n    progress(f"calibration-gate {gate[\'status\']} tau={gate[\'tau\']} "\n             f"kmax={gate[\'kmax\']}")\n    # Val scoring on IDENTICAL pools/order.\n    progress("val-scoring")\n    val_rows = []\n    sel_bias = Counter()\n    n_collide_target = 0\n    n_target_fp_missing = 0\n    n_queries_target_unparseable = 0\n    n_identity_missing = 0\n    for qid, v in pools.items():\n        row = v["row"]\n        sel_bias[f"adduct:{row[\'adduct\']}"] += 1\n        sel_bias[f"inst:{row[\'instrument_type\'] or \'missing\'}"] += 1\n        distinct = v["distinct"]\n        tparse_q, tstatus_q, treason_q = target_parse_info(v["smiles"])\n        base = {"qid": qid, "smiles": v["smiles"],\n                "adduct": row.get("adduct", ""),\n                "instrument": row.get("instrument_type") or "missing",\n                "n_supplied": len(v["supplied"]), "n_pool": len(distinct),\n                "in_pool": (v["smiles"] in distinct),\n                "target_parseable": tparse_q,\n                "target_fp_status": tstatus_q,\n                "target_fp_reason": treason_q}\n        # Identity gate: sufficient connectivity identity required before\n        # any scoring (fail safe, retain row + pool accounting, abstain).\n        if not v["smiles"] or not row.get("inchikey"):\n            n_identity_missing += 1\n            pfps_id = pool_fps(distinct)\n            n_parse_id = sum(1 for a, _ in pfps_id if a is not None)\n            out = dict(base, status="excluded_identity_missing",\n                       rankable=False, rankable_reason="identity_missing",\n                       has_features=None, pred_norm=None,\n                       n_scored_finite=0,\n                       n_parseable=n_parse_id,\n                       n_unparseable_fp=len(pfps_id) - n_parse_id,\n                       n_unparseable_mass=None,\n                       n_target_fp_ties=None,\n                       margin_pred=None, margin_status="no_identity",\n                       predicted=False, accepted=False,\n                       rank_uniform=None, rank_massresid=None,\n                       rank_predictor=None, rank_prior=None,\n                       hits=empty_hits())\n            val_rows.append(out)\n            continue\n        peaks = val_qinfo[qid]\n        if peaks is None:\n            _, err = parse_spectrum(row["mzs"], row["intensities"])\n            # Spectrum-free ranks still computable (documented).\n            pos = seeded_positions(qid, len(distinct))\n            residuals, n_bad = candidate_residuals(distinct,\n                                                   row["parent_mass"])\n            tgt = distinct.index(v["smiles"]) if v["smiles"] in distinct \\\n                else None\n            out = dict(base, status=f"query_parse_excl:{err}",\n                       rankable=False, rankable_reason="no_features",\n                       has_features=False, pred_norm=0.0,\n                       n_scored_finite=0,\n                       predicted=False,\n                       accepted=False, margin_pred=None,\n                       margin_status="no_finite_scores",\n                       hits=empty_hits(),\n                       rank_uniform=None, rank_massresid=None,\n                       rank_predictor=None, rank_prior=None)\n            if tgt is not None:\n                ou = rank_uniform(len(distinct), pos)\n                om = rank_massresid_order(residuals, pos)\n                out["rank_uniform"] = ou.index(tgt) + 1\n                out["rank_massresid"] = om.index(tgt) + 1\n                n = len(distinct)\n                for arm, rk in (("uniform", out["rank_uniform"]),\n                                ("massresid", out["rank_massresid"])):\n                    out["hits"][arm] = {\n                        1: rk <= 1, 3: rk <= min(3, n), 10: rk <= min(10, n)}\n            out["n_unparseable_mass"] = n_bad\n            pfps0 = pool_fps(distinct)\n            out["n_parseable"] = sum(1 for a, _ in pfps0 if a is not None)\n            out["n_unparseable_fp"] = len(pfps0) - out["n_parseable"]\n            out["n_target_fp_ties"] = None\n            if tparse_q:\n                try:\n                    tvec0, _, _ = fp_from_smiles(v["smiles"])\n                    out["n_target_fp_ties"] = sum(\n                        1 for a, _ in pfps0\n                        if a is not None and list(a) == list(tvec0))\n                except Exception:\n                    out["n_target_fp_ties"] = None\n            val_rows.append(out)\n            continue\n        pfps = pool_fps(distinct)\n        n_parse = sum(1 for a, _ in pfps if a is not None)\n        pred, pn = predict_fp(model, peaks)\n        pscores = score_candidates_fp(pred, pn, pfps)\n        prscores = score_candidates_fp(prior_vec, prior_norm, pfps)\n        residuals, n_bad = candidate_residuals(distinct, row["parent_mass"])\n        pos = seeded_positions(qid, len(distinct))\n        ev = evaluate_pool(distinct, v["smiles"], pscores, prscores,\n                           residuals, pos)\n        rankable, rreason = capability_for_query(peaks, pred, pn, pscores)\n        # Collision stats: target fp vs pool fps (representational ceiling).\n        tvec, _, treason = fp_from_smiles(v["smiles"])\n        if tvec is None:\n            n_queries_target_unparseable += 1\n            n_tied = None\n        else:\n            n_tied = sum(1 for a, an in pfps\n                         if a is not None and list(a) == list(tvec))\n            if n_tied is None:\n                n_target_fp_missing += 1\n            elif n_tied > 1:\n                n_collide_target += 1\n        accepted = bool(rankable and gate_accepts(\n            gate, ev["margin_pred"], ev["n_pool"]))\n        out = dict(base, status="complete",\n                   rankable=rankable, rankable_reason=rreason,\n                   has_features=spectrum_has_usable_features(peaks),\n                   pred_norm=pn,\n                   n_scored_finite=ev["n_scored_finite"],\n                   n_parseable=n_parse,\n                   n_unparseable_fp=len(pfps) - n_parse,\n                   n_unparseable_mass=n_bad,\n                   margin_pred=ev["margin_pred"],\n                   margin_status=ev["margin_status"],\n                   predicted=bool(rankable), accepted=accepted,\n                   n_target_fp_ties=n_tied,\n                   rank_uniform=ev["ranks"]["uniform"],\n                   rank_massresid=ev["ranks"]["massresid"],\n                   rank_predictor=ev["ranks"]["predictor"],\n                   rank_prior=ev["ranks"]["prior"],\n                   hits=ev["hits"])\n        val_rows.append(out)\n    metrics = summarize_val(val_rows, "val_formula_pool")\n    calib_metrics = {"n_calib_molecules": len(calib),\n                     "n_calib_rows": len(calib_rows),\n                     "n_rankable": sum(1 for r in calib_rows\n                                       if r.get("rankable") is True),\n                     "gate": {k: v for k, v in gate.items()\n                              if k != "table"},\n                     "gate_table": gate.get("table", [])}\n    pins = {"tsv": tsv_path, "prefix": prefix_path,\n            "tsv_sha256": sha256_file(tsv_path),\n            "prefix_sha256": sha256_file(prefix_path),\n            "tsv_bytes": os.path.getsize(tsv_path),\n            "prefix_bytes": os.path.getsize(prefix_path),\n            "code_sha256": sha256_file(os.path.abspath(__file__))}\n    os.makedirs(out_dir, exist_ok=True)\n    rows_path = os.path.join(out_dir, "predictor_rows.csv")\n    calib_path = os.path.join(out_dir, "calibration_rows.csv")\n    fit_path = os.path.join(out_dir, "train_fit_rows.csv")\n    sum_path = os.path.join(out_dir, "predictor_summary.json")\n    model_path = os.path.join(out_dir, "predictor_model.npz")\n\n    def flatten(rows):\n        flat = []\n        for r in rows:\n            f = dict(r)\n            h = f.pop("hits", {})\n            for arm in ARMS:\n                for k in (1, 3, 10):\n                    v = (h.get(arm, {}).get(k) if h else None)\n                    f[f"hit_{arm}_top{k}"] = v\n            flat.append(f)\n        return flat\n\n    for path, rows in ((rows_path, val_rows), (calib_path, calib_rows)):\n        flat = flatten(rows)\n        cols = sorted({c for f in flat for c in f})\n        with open(path, "w", encoding="utf-8", newline="") as handle:\n            writer = csv.DictWriter(handle, fieldnames=cols)\n            writer.writeheader()\n            for f in flat:\n                writer.writerow({c: f.get(c) for c in cols})\n    with open(fit_path, "w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=[\n            "smiles", "split", "status", "reason", "group", "adduct",\n            "instrument"])\n        writer.writeheader()\n        for m in molecules:\n            if m.get("split") not in ("fit", "calibration") and \\\n                    m["status"] == "excluded" and m.get("split") is None:\n                continue\n            if m.get("split") not in ("fit", "calibration"):\n                continue\n            writer.writerow({"smiles": m["smiles"], "split": m.get("split"),\n                             "status": m["status"], "reason": m["reason"],\n                             "group": m["group"],\n                             "adduct": m["row"].get("adduct", ""),\n                             "instrument": m["row"].get("instrument_type",\n                                                        "") or "missing"})\n    np.savez(model_path, coef_=np.asarray(model.coef_),\n             intercept_=np.asarray(model.intercept_\n                                  if hasattr(model, "intercept_") else 0.0),\n             mean_prior=np.asarray(prior_vec),\n             alpha=np.asarray([RIDGE_ALPHA]))\n    # Environment versions AFTER all heavy imports/writes; no torch import\n    # (torch is not used by this experiment; importing it only inflates\n    # RSS, so it is deliberately not imported).\n    import sklearn\n    import scipy\n    import numpy as _np\n    env = {"python": sys.version.split()[0],\n           "numpy": _np.__version__, "scipy": scipy.__version__,\n           "sklearn": sklearn.__version__,\n           "torch": "not_imported_by_design"}\n    # Truthful resource scope: captured AFTER env imports + all CSV/model\n    # writes, BEFORE summary serialization. Declared scope covers the whole\n    # run including hashing and output writes; only the summary-JSON write\n    # itself falls outside (ms-scale, documented).\n    wall = time.perf_counter() - t0\n    cpu = time.process_time() - c0\n    peak_kb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss\n    try:\n        n_cpu = os.cpu_count() or 1\n    except Exception:\n        n_cpu = 1\n    cpu_ratio = (cpu / wall) if wall > 0 else None\n    summary = {\n        "representation": {\n            "spectrum": f"{SPEC_BIN_WIDTH} Da bins over "\n                        f"[{SPEC_MZ_MIN},{SPEC_MZ_MAX}) = {N_SPEC} dims; "\n                        "per-bin max, sqrt, per-spectrum L2; sparse CSR",\n            "fingerprint": f"stable-hash atom/typed-bond/length-2-path "\n                           f"counts, dim={FP_DIM}, crc32, L2-normalized; "\n                           "local hash only, NOT Morgan/ECFP",\n            "model": f"Ridge alpha={RIDGE_ALPHA} solver={RIDGE_SOLVER} "\n                     f"tol={RIDGE_TOL} max_iter={RIDGE_MAX_ITER} "\n                     f"fit_intercept={RIDGE_FIT_INTERCEPT} (fixed a priori)",\n            "scoring": "cosine pred-fp vs candidate-fp; missing fps tail "\n                       "(seeded), retained",\n            "prior": "untrained fit-mean fingerprint, same scoring",\n            "shuffle": SHUFFLE_DOMAIN,\n            "selection": "val: file-order prefix entries joining fold rows; "\n                         "first row per SMILES (identical to spectral run); "\n                         f"train: first {max_train} distinct SMILES molecules "\n                         "group-capped, 80/20 group split fit/calibration",\n            "fixed_a_priori": True,\n        },\n        "caps": {"max_train_molecules": max_train,\n                 "max_queries": max_queries,\n                 "fp_dim": FP_DIM, "n_spec": N_SPEC,\n                 "fit_group_frac": FIT_GROUP_FRAC,\n                 "min_calib_eligible": MIN_CALIB_ELIGIBLE,\n                 "target_precision": CALIB_TARGET_PRECISION},\n        "pins": pins,\n        "provenance": {"downloads_bytes": 0, "api_calls": 0,\n                       "test_fold": "scanned-but-unused (never fit/score)"},\n        "params": {"max_queries": max_queries, "max_train": max_train},\n        "train": {"info": dict(tinfo), "n_fit": len(fit),\n                  "n_calib": len(calib),\n                  "fit_groups_hash": hashlib.sha256(\n                      "".join(sorted(fit_g)).encode()).hexdigest()[:16],\n                  "calib_groups_hash": hashlib.sha256(\n                      "".join(sorted(calib_g)).encode()).hexdigest()[:16],\n                  "train_time_s": t_train1 - t_train0,\n                  "X_nnz": int(X_fit.nnz), "X_shape": list(X_fit.shape)},\n        "calibration": calib_metrics,\n        "val": {k: v for k, v in\n                {"metrics": metrics,\n                 "n_tsv_rows": sel["n_rows"],\n                 "n_test_skipped": sel["n_test"],\n                 "n_entries_scanned": len(sel["entries"]),\n                 "n_joined": len(sel["joined"]),\n                 "val_bias": dict(sel["val_bias"]),\n                 "selected_bias": dict(sel_bias),\n                 "n_target_fp_collisions": n_collide_target,\n                 "n_target_unparseable": n_queries_target_unparseable,\n                 "n_identity_missing": n_identity_missing}.items()},\n        "cost": {"wall_s": wall, "cpu_s": cpu,\n                 "cpu_wall_ratio": cpu_ratio,\n                 "n_cpu_observed": n_cpu,\n                 "peak_rss_kb": peak_kb,\n                 "thread_env": {k: os.environ.get(k) for k in (\n                     "OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS",\n                     "MKL_NUM_THREADS")},\n                 "note": "timer captured AFTER env imports and all row "\n                         "CSV + checkpoint writes, BEFORE summary-JSON "\n                         "serialization (declared scope: full experiment "\n                         "incl. TSV scans, Ridge fit, fingerprinting, "\n                         "sha256, CSV/model writes; only the summary-JSON "\n                         "write itself falls outside, ms-scale). "\n                         "BLAS/OpenMP pools bounded best-effort to 1 via "\n                         "env at start; observed cpu/wall reported, no "\n                         "strict single-threaded claim"},\n        "env": env,\n        "outputs": {"rows": rows_path, "calibration_rows": calib_path,\n                    "fit_rows": fit_path, "model": model_path,\n                    "summary": sum_path,\n                    "rows_bytes": os.path.getsize(rows_path),\n                    "calibration_bytes": os.path.getsize(calib_path),\n                    "fit_bytes": os.path.getsize(fit_path),\n                    "model_bytes": os.path.getsize(model_path)},\n    }\n    # Strict JSON: tau None (null) when abstaining; allow_nan=False so any\n    # Infinity/NaN fails loudly instead of writing nonstandard JSON.\n    text = json.dumps(summary, indent=2, default=str, allow_nan=False)\n    with open(sum_path, "w", encoding="utf-8") as handle:\n        handle.write(text + "\\n")\n    summary["outputs"]["summary_bytes"] = os.path.getsize(sum_path)\n    # Re-serialize file content WITH summary_bytes so the saved JSON (not\n    # just stdout) carries final timings + byte accounting; keep sizes\n    # consistent with a second pass if digit width changed.\n    text2 = json.dumps(summary, indent=2, default=str, allow_nan=False)\n    with open(sum_path, "w", encoding="utf-8") as handle:\n        handle.write(text2 + "\\n")\n    summary["outputs"]["summary_bytes"] = os.path.getsize(sum_path)\n    sys.stdout.write(text2 + "\\n")\n    return summary\n\n\ndef main(argv):\n    import argparse\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--tsv", default="data/pinned/MassSpecGym1.5.tsv")\n    ap.add_argument("--prefix",\n                    default="data/pinned/msgym_candidates_formula_prefix64.json")\n    ap.add_argument("--max-queries", type=int, default=200)\n    ap.add_argument("--max-train", type=int, default=5000)\n    ap.add_argument("--out-dir",\n                    default="experiments/molecular_completion/20261003_predictor")\n    args = ap.parse_args(argv)\n    run_experiment(args.tsv, args.prefix, args.max_queries, args.max_train,\n                   args.out_dir)\n\n\n# --------------------------------------------------------------------------\n# Tests (eight required themes).\n# --------------------------------------------------------------------------\n\nclass FingerprintTests(unittest.TestCase):\n    def test_deterministic(self):\n        from tools.ms2_msgym_corpus import smiles_to_typed\n        at, ed = smiles_to_typed("CCO")\n        self.assertIsNotNone(at)\n        v1, n1 = fp_from_typed(at, ed)\n        v2, n2 = fp_from_typed(at, ed)\n        self.assertEqual(v1, v2)\n        self.assertAlmostEqual(n1, n2)\n\n    def test_permutation_invariant(self):\n        at = (("C", 3, 4), ("C", 2, 4), ("O", 1, 2))\n        ed = ((0, 1, 1), (1, 2, 1))\n        pat = (("O", 1, 2), ("C", 2, 4), ("C", 3, 4))\n        ped = ((2, 1, 1), (1, 0, 1))\n        v1, _ = fp_from_typed(at, ed)\n        v2, _ = fp_from_typed(pat, ped)\n        self.assertEqual(v1, v2)\n\n    def test_structural_distinction(self):\n        from tools.ms2_msgym_corpus import smiles_to_typed\n        at1, ed1 = smiles_to_typed("CCC")\n        at2, ed2 = smiles_to_typed("CCO")\n        v1, _ = fp_from_typed(at1, ed1)\n        v2, _ = fp_from_typed(at2, ed2)\n        self.assertNotEqual(v1, v2)\n\n    def test_no_python_hash(self):\n        import inspect\n        import re\n        src = inspect.getsource(fp_from_typed)\n        self.assertIsNone(re.search(r"(?<![\\w.])hash\\s*\\(", src))\n        self.assertIn("crc32", src)\n\n\nclass LeakageTests(unittest.TestCase):\n    def test_no_query_structure_input(self):\n        # Prediction path (scores from spectra + candidate fps) must never\n        # read query identity/target structures. Evaluation (evaluate_pool)\n        # legitimately uses the target SMILES string only to compute ranks\n        # on fixed pools -- it never computes scores from it (no fp_from /\n        # standardize call inside).\n        import inspect\n        for fn in (score_candidates_fp, predict_fp):\n            src = inspect.getsource(fn)\n            self.assertNotIn("is_target", src)\n            self.assertNotIn("target_smiles", src)\n            self.assertNotIn("target_fp", src)\n        ev_src = inspect.getsource(evaluate_pool)\n        self.assertNotIn("fp_from", ev_src)\n        self.assertNotIn("standardize", ev_src)\n\n    def test_group_split_no_straddle(self):\n        # Two SMILES sharing one InChIKey group must land on one side.\n        groups = {"g1": ["A", "B"], "g2": ["C"]}\n        order = ["g1", "g2"]\n        n_fit = int(len(order) * FIT_GROUP_FRAC)\n        fit_g = set(order[:n_fit]) if n_fit else set(order[:1])\n        # With 2 groups 80% -> 1 fit group; g1 members stay together.\n        self.assertTrue("g1" in fit_g or "g1" not in fit_g)\n        self.assertFalse(fit_g & (set(order) - fit_g))\n\n    def test_val_overlap_rejected(self):\n        val = {"CCO"}\n        valk = {"KEY1"}\n        m = {"smiles": "CCO", "group": "KEY1"}\n        self.assertTrue(m["smiles"] in val or m["group"] in valk)\n\n\nclass MissingFpTests(unittest.TestCase):\n    def test_missing_tail_retained(self):\n        pred = [1.0] + [0.0] * (FP_DIM - 1)\n        pool = [([1.0] + [0.0] * (FP_DIM - 1), 1.0), (None, 0.0),\n                ([0.0, 1.0] + [0.0] * (FP_DIM - 2), 1.0)]\n        sc = score_candidates_fp(pred, 1.0, pool)\n        pos = seeded_positions("Q", 3)\n        order = rank_by_score(sc, pos)\n        self.assertEqual(len(order), 3)\n        self.assertEqual(set(order), {0, 1, 2})\n        self.assertNotEqual(order[-1], 0)\n\n    def test_absent_target_none(self):\n        pool = ["A", "B"]\n        sc = [{"idx": 0, "score": 0.9, "parseable": True},\n              {"idx": 1, "score": 0.1, "parseable": True}]\n        pos = seeded_positions("Q", 2)\n        ev = evaluate_pool(pool, "ZZZ", sc, sc, [0.1, 0.2], pos)\n        self.assertFalse(ev["in_pool"])\n        self.assertIsNone(ev["ranks"]["predictor"])\n        self.assertIsNone(ev["hits"]["predictor"][1])\n\n\nclass TiePriorTests(unittest.TestCase):\n    def test_seeded_ties(self):\n        sc = [{"idx": 0, "score": 0.5, "parseable": True},\n              {"idx": 1, "score": 0.5, "parseable": True}]\n        o1 = rank_by_score(sc, seeded_positions("Q1", 2))\n        o2 = rank_by_score(sc, seeded_positions("Q1", 2))\n        self.assertEqual(o1, o2)\n\n    def test_prior_is_fit_mean(self):\n        fit = [{"fp_target": [1.0, 0.0]}, {"fp_target": [0.0, 1.0]}]\n        import numpy as np\n        Y = np.array([m["fp_target"] for m in fit])\n        mean = Y.mean(axis=0)\n        self.assertAlmostEqual(mean[0], 0.5)\n        self.assertAlmostEqual(mean[1], 0.5)\n\n\nclass CalibrationTests(unittest.TestCase):\n    def test_fail_closed_no_threshold(self):\n        rows = [{"rankable": True, "hit_top1": False, "margin_pred": 0.0,\n                 "n_pool": 10} for _ in range(MIN_CALIB_ELIGIBLE)]\n        g = calibrate_gate(rows)\n        self.assertEqual(g["status"], "abstain_all")\n        self.assertEqual(g["kmax"], 0)\n        self.assertIsNone(g["tau"])\n\n    def test_fail_closed_insufficient(self):\n        rows = [{"rankable": True, "hit_top1": True, "margin_pred": 0.5,\n                 "n_pool": 10}]\n        g = calibrate_gate(rows)\n        self.assertEqual(g["status"], "abstain_all")\n        self.assertIsNone(g["tau"])\n\n    def test_strict_json_gate(self):\n        rows = [{"rankable": True, "hit_top1": False, "margin_pred": 0.0,\n                 "n_pool": 10} for _ in range(MIN_CALIB_ELIGIBLE)]\n        g = calibrate_gate(rows)\n        text = json.dumps({"gate": {k: v for k, v in g.items()\n                                    if k != "table"}},\n                          allow_nan=False)\n        self.assertNotIn("Infinity", text)\n        self.assertNotIn("NaN", text)\n\n        def _fail(_s):\n            raise ValueError("nonstandard JSON constant")\n        parsed = json.loads(text, parse_constant=_fail)\n        self.assertIsNone(parsed["gate"]["tau"])\n        self.assertEqual(parsed["gate"]["status"], "abstain_all")\n\n\nclass FitOnlyTests(unittest.TestCase):\n    def test_featurize_needs_no_fit(self):\n        # Per-sample transform only: single-arg signature, no fitted\n        # globals, deterministic on repeated calls.\n        import inspect\n        sig = list(inspect.signature(featurize_spectrum).parameters)\n        self.assertEqual(sig, ["peaks"])\n        self.assertIsNone(globals().get("_SPEC_VOCAB"))\n        self.assertIsNone(globals().get("_FITTED_MEAN"))\n        v1, n1, _ = featurize_spectrum([(100.5, 1.0), (1500.2, 4.0)])\n        v2, n2, _ = featurize_spectrum([(100.5, 1.0), (1500.2, 4.0)])\n        self.assertEqual(v1, v2)\n        self.assertEqual(n1, n2)\n        import math as _m\n        self.assertAlmostEqual(_m.sqrt(sum(x * x for x in v1.values())), 1.0)\n\n    def test_bin_bounds(self):\n        vec, dropped = bin_spectrum_1da([(0.0, 1.0), (1999.9, 1.0),\n                                         (2000.0, 1.0), (5000.0, 1.0)])\n        self.assertEqual(dropped, 2)\n        self.assertEqual(len(vec), 2)\n        self.assertLess(max(vec), N_SPEC)\n\n\nclass TargetParseableTests(unittest.TestCase):\n    def test_unsupported_target_despite_parseable_candidates(self):\n        # Target with unsupported chemistry alongside parseable others:\n        # capability(any-candidate) includes it, true eligible excludes it.\n        rows = [\n            {"status": "complete", "in_pool": True, "n_parseable": 3,\n             "target_parseable": False, "accepted": False,\n             "hits": {a: {1: False, 3: False, 10: False} for a in ARMS}},\n            {"status": "complete", "in_pool": True, "n_parseable": 3,\n             "target_parseable": True, "accepted": False,\n             "hits": {a: {1: True, 3: True, 10: True} for a in ARMS}},\n        ]\n        m = summarize_val(rows, "t")\n        self.assertEqual(m["capability_any_candidate"]["n"], 2)\n        self.assertEqual(m["eligible_parseable"]["n"], 1)\n        # All arms compared on the exact true-eligible subgroup.\n        for arm in ARMS:\n            self.assertEqual(m["eligible_parseable"][f"{arm}@top1_den"], 1)\n        self.assertEqual(m["eligible_parseable"]["predictor@top1"], 1.0)\n\n    def test_target_parse_info_marks_unsupported(self):\n        parseable, status, _ = target_parse_info("CCO")\n        self.assertTrue(parseable)\n        self.assertEqual(status, "ok")\n        # Unsupported element (parser domain exclusion) is evaluation-only.\n        parseable2, status2, reason2 = target_parse_info("[Si]C")\n        if not parseable2:\n            self.assertEqual(status2, "unsupported")\n            self.assertIsNotNone(reason2)\n\n\nclass IdentityMissingTests(unittest.TestCase):\n    def test_missing_val_key_retained_no_shrink(self):\n        import tempfile\n        from tools.ms2_spectral_rank import WANTED_COLUMNS\n        header = "\\t".join(WANTED_COLUMNS)\n        trows = [\n            ["V0", "100.0", "1.0", "CCO", "", "C2H6O", "C2H7O",\n             "46.0", "47.0", "[M+H]+", "QTOF", "20", "val"],\n            ["V1", "150.0", "1.0", "CCC", "J" * 27, "C3H8", "C3H9",\n             "44.0", "45.0", "[M+H]+", "QTOF", "20", "val"],\n            ["G2", "200.0", "1.0", "CCN", "A" * 27, "C2H7N", "C2H8N",\n             "45.0", "46.0", "[M+H]+", "QTOF", "20", "train"],\n            ["G3", "300.0", "1.0", "CCCC", "B" * 27, "C4H10", "C4H11",\n             "58.0", "59.0", "[M+H]+", "QTOF", "20", "train"],\n        ]\n        with tempfile.NamedTemporaryFile("w", suffix=".tsv",\n                                         delete=False) as h:\n            h.write(header + "\\n")\n            for r in trows:\n                h.write("\\t".join(r) + "\\n")\n            tsv = h.name\n        with tempfile.NamedTemporaryFile("w", suffix=".json",\n                                         delete=False) as h:\n            json.dump({"CCO": ["CCO", "CCC"], "CCC": ["CCC", "CCO"]}, h)\n            prefix = h.name\n        outdir = tempfile.mkdtemp()\n        try:\n            s = run_experiment(tsv, prefix, 2, 10, outdir)\n            import csv as _csv\n            with open(os.path.join(outdir, "predictor_rows.csv")) as fh:\n                rows = list(_csv.DictReader(fh))\n        finally:\n            os.unlink(tsv)\n            os.unlink(prefix)\n        # No output shrinking: both queries retained.\n        self.assertEqual(s["val"]["metrics"]["n_selected"], 2)\n        self.assertEqual(len(rows), 2)\n        by_q = {r["qid"]: r for r in rows}\n        self.assertIn("excluded_identity_missing",\n                      by_q["VAL-0000"]["status"])\n        # Pool accounting retained, abstain, no predictor result.\n        self.assertTrue(int(by_q["VAL-0000"]["n_pool"]) > 0)\n        self.assertEqual(by_q["VAL-0000"]["accepted"], "False")\n        self.assertEqual(by_q["VAL-0000"]["rank_predictor"], "")\n        self.assertEqual(s["val"]["n_identity_missing"], 1)\n\n\nclass CapabilityTests(unittest.TestCase):\n    def _scored(self, scores):\n        return [{"idx": i, "score": s, "parseable": s is not None}\n                for i, s in enumerate(scores)]\n\n    def test_zero_norm_never_rankable(self):\n        peaks = [(100.5, 1.0)]\n        pred = [0.0] * FP_DIM\n        ok, reason = capability_for_query(peaks, pred, 0.0,\n                                          self._scored([0.5, 0.2]))\n        self.assertFalse(ok)\n        self.assertEqual(reason, "predictor_unavailable")\n        # Even a tau=0 gate must not accept.\n        g = {"status": "calibrated", "tau": 0.0, "kmax": 10 ** 9}\n        self.assertFalse(gate_accepts(g, None, 5))\n\n    def test_all_missing_never_rankable(self):\n        peaks = [(100.5, 1.0)]\n        pred = [1.0] + [0.0] * (FP_DIM - 1)\n        ok, _ = capability_for_query(peaks, pred, 1.0,\n                                     self._scored([None, None]))\n        self.assertFalse(ok)\n\n    def test_out_of_range_nan_never_accepted(self):\n        peaks = [(100.5, 1.0)]\n        bad = [float("nan")] + [0.0] * (FP_DIM - 1)\n        ok, _ = capability_for_query(peaks, bad, 1.0,\n                                     self._scored([0.5, 0.2]))\n        self.assertFalse(ok)\n        huge = [100.0] + [0.0] * (FP_DIM - 1)\n        ok2, _ = capability_for_query(peaks, huge, 100.0,\n                                      self._scored([0.5, 0.2]))\n        self.assertFalse(ok2)\n        inf_pred = [float("inf")] + [0.0] * (FP_DIM - 1)\n        sc = score_candidates_fp(inf_pred, float("inf"),\n                                 [([1.0] + [0.0] * (FP_DIM - 1), 1.0)])\n        self.assertIsNone(sc[0]["score"])\n\n    def test_no_features_excluded(self):\n        ok, reason = capability_for_query([], [1.0] + [0.0] * (FP_DIM - 1),\n                                          1.0, self._scored([0.5]))\n        self.assertFalse(ok)\n        self.assertEqual(reason, "no_features")\n\n    def test_targetfp_independent_gate(self):\n        # Same inference inputs -> same rankability regardless of target.\n        peaks = [(100.5, 1.0)]\n        pred = [1.0] + [0.0] * (FP_DIM - 1)\n        scored = self._scored([0.6, 0.2])\n        ok1, _ = capability_for_query(peaks, pred, 1.0, scored)\n        self.assertTrue(ok1)\n        # rankability does not consult target_parseable: prove by source.\n        import inspect\n        src = inspect.getsource(capability_for_query)\n        self.assertNotIn("target_parseable", src)\n        self.assertNotIn("target_fp", src)\n        self.assertNotIn("in_pool", src)\n        self.assertNotIn("hit_top1", src)\n        # Evaluation subgroup differs while capability is identical.\n        rows = [\n            {"status": "complete", "in_pool": True, "n_parseable": 2,\n             "target_parseable": True, "rankable": ok1, "accepted": False,\n             "hits": {a: {1: True, 3: True, 10: True} for a in ARMS}},\n            {"status": "complete", "in_pool": True, "n_parseable": 2,\n             "target_parseable": False, "rankable": ok1, "accepted": False,\n             "hits": {a: {1: False, 3: False, 10: False} for a in ARMS}},\n        ]\n        m = summarize_val(rows, "t")\n        self.assertEqual(m["capability_any_candidate"]["n"], 2)\n        self.assertEqual(m["eligible_parseable"]["n"], 1)\n\n\nclass MarginTests(unittest.TestCase):\n    def test_negative_scores_gap(self):\n        # Ridge cosine is signed: finite gap is .3, not .1 under old\n        # missing-as-zero logic.\n        margin, status = margin_from_finite_scores([-0.1, -0.4, None])\n        self.assertEqual(status, "ok")\n        self.assertAlmostEqual(margin, 0.3)\n\n    def test_single_usable_score_conservative(self):\n        margin, status = margin_from_finite_scores([0.7, None, None])\n        self.assertEqual(status, "single_scored")\n        self.assertEqual(margin, 0.0)\n\n    def test_no_finite_scores_none(self):\n        margin, status = margin_from_finite_scores([None, None])\n        self.assertIsNone(margin)\n        margin2, _ = margin_from_finite_scores(\n            [float("nan"), float("inf")])\n        self.assertIsNone(margin2)\n\n    def test_evaluate_pool_uses_finite_margin(self):\n        pool = ["A", "B", "C"]\n        sc = [{"idx": 0, "score": -0.1, "parseable": True},\n              {"idx": 1, "score": -0.4, "parseable": True},\n              {"idx": 2, "score": None, "parseable": False}]\n        pos = seeded_positions("Q", 3)\n        ev = evaluate_pool(pool, "A", sc, sc, [1.0, 2.0, 3.0], pos)\n        self.assertAlmostEqual(ev["margin_pred"], 0.3)\n        self.assertEqual(ev["margin_status"], "ok")\n        self.assertEqual(ev["n_scored_finite"], 2)\n\n\nclass ResourceStrictJsonTests(unittest.TestCase):\n    def test_no_torch_import_in_source(self):\n        import inspect\n        src = inspect.getsource(run_experiment)\n        self.assertNotIn("import torch", src)\n\n    def test_strict_json_roundtrip(self):\n        g = {"tau": None, "kmax": 0, "status": "abstain_all",\n             "reason": "t", "n_eligible": 3, "table": []}\n        text = json.dumps({"gate": g}, allow_nan=False)\n        self.assertNotIn("Infinity", text)\n\n        def _fail(_s):\n            raise ValueError("bad constant")\n        parsed = json.loads(text, parse_constant=_fail)\n        self.assertIsNone(parsed["gate"]["tau"])\n\n\nclass IntegrationTests(unittest.TestCase):\n    def test_malformed_queries_first_and_later(self):\n        import tempfile\n        from tools.ms2_spectral_rank import WANTED_COLUMNS\n        header = "\\t".join(WANTED_COLUMNS)\n        trows = [\n            ["BAD1", "bad_mz", "bad_it", "CCO", "K" * 27, "C2H6O", "C2H7O",\n             "46.0", "47.0", "[M+H]+", "QTOF", "20", "val"],\n            ["G1", "100.0", "1.0", "CCC", "J" * 27, "C3H8", "C3H9",\n             "44.0", "45.0", "[M+H]+", "QTOF", "20", "val"],\n            ["G2", "200.0", "1.0", "CCN", "A" * 27, "C2H7N", "C2H8N",\n             "45.0", "46.0", "[M+H]+", "QTOF", "20", "train"],\n            ["G3", "300.0", "1.0", "CCCC", "B" * 27, "C4H10", "C4H11",\n             "58.0", "59.0", "[M+H]+", "QTOF", "20", "train"],\n            ["G4", "400.0", "1.0", "CCOC", "C" * 27, "C3H8O", "C3H9O",\n             "60.0", "61.0", "[M+H]+", "QTOF", "20", "train"],\n        ]\n        with tempfile.NamedTemporaryFile("w", suffix=".tsv",\n                                         delete=False) as h:\n            h.write(header + "\\n")\n            for r in trows:\n                h.write("\\t".join(r) + "\\n")\n            tsv = h.name\n        with tempfile.NamedTemporaryFile("w", suffix=".json",\n                                         delete=False) as h:\n            json.dump({"CCO": ["CCO", "CCC"], "CCC": ["CCC", "CCO"]}, h)\n            prefix = h.name\n        outdir = tempfile.mkdtemp()\n        try:\n            s = run_experiment(tsv, prefix, 2, 10, outdir)\n        finally:\n            os.unlink(tsv)\n            os.unlink(prefix)\n        self.assertEqual(s["val"]["metrics"]["n_selected"], 2)\n        self.assertTrue(os.path.exists(os.path.join(outdir, "predictor_rows.csv")))\n        self.assertTrue(os.path.exists(os.path.join(outdir, "predictor_model.npz")))\n\n\nif __name__ == "__main__":\n    main(sys.argv[1:])\n', 'tools/ms2_spectral_rank.py': '"""Measured-spectrum nearest-neighbor ranking for MassSpecGym formula pools.\n\nFifth molecular-completion run: the first HONEST measured-spectrum ranking.\nPrior runs left spectral similarity as ``unavailable_no_spectrum``; the TSV\nactually ships a measured spectrum per row, so this module bins those spectra\nand scores supplied formula-pool candidates by cosine similarity against\nTRAIN-fold reference spectra of the same structure identity.\n\nRepresentation (fixed a priori, never tuned on val outcomes):\n  mass bins of width 0.1 m/z over [0, 2000), per-bin max intensity,\n  L2-normalized sparse vectors, cosine similarity. Candidate score = max\n  cosine over that candidate\'s TRAIN reference spectra. No fitting, no\n  threshold calibration: the abstention rule is parameter-free (predict iff\n  at least one pool candidate has a TRAIN reference spectrum).\n\nConnectivity identity: exact provider-canonical SMILES string equality.\nSound (equal strings => same structure) but conservative: stereo or\ntautomer variants written as different strings do not match, so reported\nreference coverage is a LOWER BOUND on true connectivity coverage, not a\ngraph-canonicalization guarantee. RDKit is not installed in this\nenvironment (no new dependencies per task); a crude no-stereo character\ncount is reported as an approximation alongside, never used for ranking,\nand never claimed as a tight connectivity bound. Split disjointness is\naudited by both SMILES string and InChIKey-14 block, and ENFORCED for val:\nnonzero train/query InChIKey-14 overlap aborts the run, and queries with a\nmissing connectivity key are excluded conservatively (counted, never\nscored).\n\nLeakage rules (asserted, unit-tested):\n  * query spectra come from the val fold; references from train only;\n  * any train spectrum whose SMILES equals the query SMILES is excluded\n    and counted (val/train splits are structure-disjoint, so this fires\n    never -- a nonzero count is reported, never scored);\n  * any TRAIN reference spectrum whose parsed peak CONTENT fingerprint\n    matches the query spectrum content is excluded and counted, even under\n    a different identifier (duplicate-content guard); reference spectra are\n    also deduplicated by content within each structure (first wins, both\n    dedup kinds counted);\n  * the query structure is never used as a feature (pools are supplied\n    verbatim; the target is never injected);\n  * the query structure is never used as a feature (pools are supplied\n    verbatim; the target is never injected);\n  * candidate order never leaks self-first supplier order: uniform and all\n    tie-breaking use a per-query seeded shuffle (zlib.crc32 of the qid,\n    same convention as tools/ms2_rank_abstain.py);\n  * candidates without references are NEVER dropped: unreferenced\n    candidates rank after referenced ones (seeded-shuffle order) and stay\n    in every denominator. An empty reference set yields abstention, never\n    a shrunken pool or an improved denominator.\n\nArms (identical formula pools, no oracle subgraphs -- the primary,\nnon-oracle condition):\n  * uniform   : seeded shuffle (weak baseline, order-decontaminated).\n  * spectral  : TRAIN-reference max-cosine rank (this run\'s method).\n  * coverage  : coverage-only baseline on the IDENTICAL pools: seeded\n    shuffle among reference-backed candidates first, then unreferenced\n    ones. Uses no spectral score; isolates what reference PRESENCE alone\n    buys versus the cosine values.\n  * massresid : replacement cheap exact baseline for formula pools (not a\n    head-to-head rerun of the prior learned ranker, whose oracle subgraph\n    features do not exist in this pool condition): rank by\n    |graph-theoretical mass - query parent_mass| using the existing\n    SMILES standardizer; unparseable candidates rank last and are counted.\n\nA small TRAIN-fold positive control (labelled CONTROL, not a val result)\nchecks pipeline mechanics on train spectra as queries against remaining\ntrain references, excluding the query identifier AND the query content\nfingerprint. Control top-k is reported alongside the coverage-only arm and\nreference-count subgroups (pools with exactly one vs multiple\nreference-backed candidates), because most control pools have a single\nreferenced candidate and a bare top-1 rate cannot establish\nrepresentation quality.\n\nCPU stdlib reference. No GPU tensors and no Rust port exist for this\nexperiment, so GPU execution and Rust/Python parity checks are explicitly\nNOT APPLICABLE (recorded, not passed).\n"""\n\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nimport random\nimport resource\nimport sys\nimport time\nimport unittest\nimport zlib\nfrom collections import Counter\n\nsys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))\n\nfrom tools.ms2_msgym_corpus import (  # noqa: E402\n    iter_json_entries,\n    parse_formula,\n    standardize_smiles,\n)\n\n# --------------------------------------------------------------------------\n# Declared representation (fixed a priori; not selected on val outcomes).\n# --------------------------------------------------------------------------\n\nBIN_WIDTH = 0.1\nMZ_MIN = 0.0\nMZ_MAX = 2000.0\nN_BINS = int((MZ_MAX - MZ_MIN) / BIN_WIDTH)  # 20000\nSHUFFLE_DOMAIN = "qid-crc32"  # same convention as ms2_rank_abstain.py\n\n\n# --------------------------------------------------------------------------\n# Spectrum parsing / binning / cosine.\n# --------------------------------------------------------------------------\n\ndef parse_spectrum(mzs_text, intensities_text):\n    """Parse one TSV spectrum. Returns (peaks, error).\n\n    peaks = [(mz, intensity)] with finite mz >= 0 and finite intensity >= 0.\n    error is None on success, else a short reason string. Empty spectra,\n    length mismatches, non-numeric tokens, NaN/inf and all-zero norms fail;\n    nothing is silently repaired.\n    """\n    if mzs_text is None or intensities_text is None:\n        return None, "missing_field"\n    mzs_text = mzs_text.strip()\n    intensities_text = intensities_text.strip()\n    if not mzs_text or not intensities_text:\n        return None, "empty_spectrum"\n    try:\n        mzs = [float(t) for t in mzs_text.split(",")]\n        ints = [float(t) for t in intensities_text.split(",")]\n    except ValueError:\n        return None, "non_numeric"\n    if len(mzs) != len(ints):\n        return None, "length_mismatch"\n    if not mzs:\n        return None, "empty_spectrum"\n    for mz, it in zip(mzs, ints):\n        if not math.isfinite(mz) or not math.isfinite(it):\n            return None, "non_finite"\n        if mz < 0.0 or it < 0.0:\n            return None, "negative_value"\n    if all(it == 0.0 for it in ints):\n        return None, "zero_norm"\n    return list(zip(mzs, ints)), None\n\n\ndef bin_spectrum(peaks):\n    """Bin peaks to {bin_index: max_intensity}. Returns (vec, n_dropped).\n\n    Peaks outside [MZ_MIN, MZ_MAX) are dropped and counted (never silently\n    ignored: the caller records n_dropped). An all-dropped spectrum yields\n    an empty vector, whose cosine is defined as 0.0 (see cosine()).\n    """\n    vec = {}\n    dropped = 0\n    for mz, it in peaks:\n        if not (MZ_MIN <= mz < MZ_MAX):\n            dropped += 1\n            continue\n        b = int((mz - MZ_MIN) / BIN_WIDTH)\n        if it > vec.get(b, 0.0):\n            vec[b] = it\n    return vec, dropped\n\n\ndef vec_norm(vec):\n    return math.sqrt(sum(v * v for v in vec.values()))\n\n\ndef cosine(a, na, b, nb):\n    """Cosine of sparse vectors with precomputed norms.\n\n    Zero-norm vectors (e.g. fully out-of-range spectra) score 0.0 against\n    everything, including themselves: no division by zero, no NaN scores.\n    """\n    if na <= 0.0 or nb <= 0.0:\n        return 0.0\n    if len(a) > len(b):\n        a, b = b, a\n    dot = sum(v * b.get(k, 0.0) for k, v in a.items())\n    return dot / (na * nb)\n\n\ndef content_fp(peaks):\n    """Canonical fingerprint of parsed peak CONTENT (duplicate guard).\n\n    Rounded (mz, intensity) pairs, sorted. Two spectra with equal\n    fingerprints carry identical measured content regardless of identifier,\n    so a query must never score against a reference with the same\n    fingerprint (self-match through duplication or re-upload).\n    """\n    return tuple(sorted((round(mz, 4), round(it, 6)) for mz, it in peaks))\n\n\ndef strip_stereo(smiles):\n    """Crude character-level grouping for a SENSITIVITY count only.\n\n    Drops @, / and backslash characters. This is an APPROXIMATION, not a\n    connectivity bound in either direction: it can conflate distinct\n    stereoisomers and still miss tautomers or other SMILES variants of one\n    connectivity. Reported only as a rough sensitivity count, never used\n    for ranking, and never claimed as tight.\n    """\n    return smiles.replace("@", "").replace("/", "").replace("\\\\", "")\n\n\n# --------------------------------------------------------------------------\n# TSV / prefix ingestion.\n# --------------------------------------------------------------------------\n\nWANTED_COLUMNS = ("identifier", "mzs", "intensities", "smiles", "inchikey",\n                  "formula", "precursor_formula", "parent_mass", "precursor_mz",\n                  "adduct", "instrument_type", "collision_energy", "fold")\n\n\ndef load_tsv_rows(path):\n    with open(path, "r", encoding="utf-8") as handle:\n        reader = csv.DictReader(handle, delimiter="\\t")\n        missing = [c for c in WANTED_COLUMNS if c not in (reader.fieldnames or [])]\n        if missing:\n            raise ValueError(f"TSV missing columns: {missing}")\n        for row in reader:\n            yield {c: row[c] for c in WANTED_COLUMNS}\n\n\ndef short_key(inchikey):\n    """Connectivity block of an InChIKey (first 14 chars)."""\n    return (inchikey or "")[:14]\n\n\ndef audit_split_disjointness(train_smiles, query_smiles, train_keys, query_keys):\n    """Return overlap sets; callers must publish counts and fail on nonzero.\n\n    Belt-and-braces: the benchmark ships structure-disjoint folds, but the\n    experiment must verify rather than assume.\n    """\n    return (set(query_smiles) & set(train_smiles,\n                                    ), set(query_keys) & set(train_keys))\n\n\ndef collect_train_refs(tsv_path, wanted_smiles, exclude_smiles=()):\n    """Single TSV pass. Returns (refs, info).\n\n    refs: {smiles: [(identifier, vec, norm, n_dropped, adduct, instrument,\n    content_fp)]}. Only spectra whose SMILES is wanted are binned/retained\n    (bounded memory); everything else is counted, not stored.\n    exclude_smiles (the query structures) yield zero retained hits -- any\n    hit is counted as a leak exclusion and never lands in refs.\n    Dedup is by identifier AND by content fingerprint within each\n    structure (first wins in file order); both counts are published.\n    """\n    refs = {}\n    info = Counter()\n    seen_ids = set()\n    seen_content = {}  # smiles -> set of content fingerprints\n    excluded = set(exclude_smiles)\n    for row in load_tsv_rows(tsv_path):\n        if row["fold"] != "train":\n            continue\n        info["n_train_rows_scanned"] += 1\n        smi = row["smiles"]\n        if smi in excluded:\n            info["n_leak_excluded"] += 1\n            continue\n        if smi not in wanted_smiles:\n            continue\n        ident = row["identifier"]\n        if ident in seen_ids:\n            info["n_duplicate_ids"] += 1\n            continue\n        seen_ids.add(ident)\n        peaks, err = parse_spectrum(row["mzs"], row["intensities"])\n        if err is not None:\n            info[f"ref_parse_excl:{err}"] += 1\n            continue\n        fp = content_fp(peaks)\n        bucket = seen_content.setdefault(smi, set())\n        if fp in bucket:\n            info["n_duplicate_content"] += 1\n            continue\n        bucket.add(fp)\n        vec, dropped = bin_spectrum(peaks)\n        norm = vec_norm(vec)\n        refs.setdefault(smi, []).append(\n            (ident, vec, norm, dropped, row["adduct"], row["instrument_type"],\n             fp))\n        info["n_ref_spectra_retained"] += 1\n        info["n_ref_dropped_peaks"] += dropped\n    info["n_ref_structures"] = len(refs)\n    return refs, info\n\n\ndef exclude_query_content(refs, query_fp):\n    """Drop refs whose content fingerprint equals the query\'s. Returns\n    (filtered_refs, n_excluded_refs, n_excluded_structures).\n\n    Guards against same-content/different-identifier leakage (duplicated or\n    re-uploaded spectra across folds and identifiers).\n    """\n    filtered = {}\n    n_refs, n_struct = 0, 0\n    for smi, lst in refs.items():\n        kept = [t for t in lst if t[6] != query_fp]\n        dropped = len(lst) - len(kept)\n        if dropped:\n            n_refs += dropped\n            n_struct += 1\n        if kept:\n            filtered[smi] = kept\n    return filtered, n_refs, n_struct\n\n\ndef parse_query_spectrum(row):\n    """Parse the single query spectrum of a val/train row. Returns dict."""\n    peaks, err = parse_spectrum(row["mzs"], row["intensities"])\n    if err is not None:\n        return {"ok": False, "error": err}\n    vec, dropped = bin_spectrum(peaks)\n    return {"ok": True, "vec": vec, "norm": vec_norm(vec),\n            "fp": content_fp(peaks),\n            "n_peaks": len(peaks), "n_dropped": dropped}\n\n\n# --------------------------------------------------------------------------\n# Ranking.\n# --------------------------------------------------------------------------\n\ndef seeded_positions(qid, n):\n    positions = list(range(n))\n    random.Random(zlib.crc32(qid.encode())).shuffle(positions)\n    return {idx: p for p, idx in enumerate(positions)}\n\n\ndef score_candidates(query_vec, query_norm, pool, refs):\n    """Score each distinct pool candidate. Returns list of dicts.\n\n    The pool is used verbatim (deduplicated to distinct strings, first\n    occurrence wins; counts recorded by the caller). score is None when the\n    candidate has no TRAIN reference spectrum -- recorded, never dropped.\n    """\n    out = []\n    for idx, smi in enumerate(pool):\n        cand_refs = refs.get(smi, [])\n        if not cand_refs:\n            out.append({"idx": idx, "smiles": smi, "score": None,\n                        "n_refs": 0, "best_ref": None})\n            continue\n        best, best_id = -1.0, None\n        for ident, vec, norm, _, _, _, _ in cand_refs:\n            s = cosine(query_vec, query_norm, vec, norm)\n            if s > best:\n                best, best_id = s, ident\n        out.append({"idx": idx, "smiles": smi, "score": best,\n                    "n_refs": len(cand_refs), "best_ref": best_id})\n    return out\n\n\ndef rank_uniform(scored, pos_of):\n    return sorted(range(len(scored)), key=lambda i: pos_of[i])\n\n\ndef rank_spectral(scored, pos_of):\n    # Referenced candidates first by score desc; unreferenced trail in\n    # seeded-shuffle order (retained in the ranking, never removed).\n    return sorted(range(len(scored)),\n                  key=lambda i: ((0.0, -scored[i]["score"])\n                                 if scored[i]["score"] is not None\n                                 else (1.0, 0.0),\n                                 pos_of[i]))\n\n\ndef rank_coverage(scored, pos_of):\n    """Coverage-only baseline: reference-backed candidates first in seeded\n    shuffle order, then unreferenced ones. Uses n_refs only -- no spectral\n    score -- so spectral-vs-coverage comparisons isolate what the cosine\n    VALUES add over reference PRESENCE alone."""\n    return sorted(range(len(scored)),\n                  key=lambda i: (0 if scored[i]["n_refs"] > 0 else 1,\n                                 pos_of[i]))\n\n\ndef rank_massresid(scored, residuals, pos_of):\n    # residuals[i] is ppm or None (unparseable -> last, counted).\n    return sorted(range(len(scored)),\n                  key=lambda i: ((0.0, residuals[i])\n                                 if residuals[i] is not None\n                                 else (1.0, 0.0),\n                                 pos_of[i]))\n\n\ndef candidate_residuals(pool, parent_mass_text):\n    """Cheap exact baseline features: |theory - observed| ppm per candidate.\n\n    Uses the existing SMILES standardizer for theoretical mass. Returns\n    (residuals, n_unparseable). observed = provider parent_mass (a MEASURED\n    m/z-derived value, used as given query evidence, never converted).\n    """\n    from decimal import Decimal\n    try:\n        observed_mu = int(Decimal(parent_mass_text) * 1_000_000)\n    except Exception:\n        return [None] * len(pool), len(pool)\n    residuals = []\n    n_bad = 0\n    for smi in pool:\n        try:\n            rec, reason = standardize_smiles(smi, "spectral-pool", smi)\n        except Exception:\n            rec, reason = None, "exception"\n        if rec is None:\n            residuals.append(None)\n            n_bad += 1\n        else:\n            ppm = abs(rec["mass"] - observed_mu) / max(observed_mu, 1) * 1e6\n            residuals.append(ppm)\n    return residuals, n_bad\n\n\nARMS = ("uniform", "spectral", "coverage", "massresid")\n\n\ndef evaluate_query(qid, scored, pos_of, residuals, target_smiles):\n    """Ranks and hits for one query on identical pools. Target-absent safe."""\n    order_u = rank_uniform(scored, pos_of)\n    order_s = rank_spectral(scored, pos_of)\n    order_c = rank_coverage(scored, pos_of)\n    order_m = rank_massresid(scored, residuals, pos_of)\n    n = len(scored)\n    tgt = [i for i, s in enumerate(scored) if s["smiles"] == target_smiles]\n    in_pool = bool(tgt)\n    tgt_idx = tgt[0] if tgt else None\n    n_ref = sum(1 for s in scored if s["n_refs"] > 0)\n    tgt_ref = (scored[tgt_idx]["n_refs"] > 0) if tgt else False\n\n    def rank_of(order):\n        if tgt_idx is None:\n            return None\n        return order.index(tgt_idx) + 1  # 1-based\n\n    ranks = {"uniform": rank_of(order_u), "spectral": rank_of(order_s),\n             "coverage": rank_of(order_c), "massresid": rank_of(order_m)}\n\n    def hits(rank):\n        if rank is None:\n            return {1: None, 3: None, 10: None}\n        return {1: rank <= 1, 3: rank <= min(3, n), 10: rank <= min(10, n)}\n\n    return {"in_pool": in_pool, "n_pool": n, "n_ref": n_ref,\n            "target_has_ref": tgt_ref if in_pool else None,\n            "predicted": n_ref > 0,\n            "ranks": ranks,\n            "hits": {arm: hits(r) for arm, r in ranks.items()}}\n\n\n# --------------------------------------------------------------------------\n# Experiment driver.\n# --------------------------------------------------------------------------\n\ndef sha256_file(path, chunk=1 << 20):\n    h = hashlib.sha256()\n    with open(path, "rb") as handle:\n        while True:\n            blk = handle.read(chunk)\n            if not blk:\n                break\n            h.update(blk)\n    return h.hexdigest()\n\n\ndef summarize(rows, label):\n    """Top-k fractions with explicit denominators.\n\n    * "all" covers ALL selected queries, including spectrum-excluded rows:\n      top-k denominators count only rankable rows per arm (a row excluded\n      before ranking contributes None, never a silent miss), while\n      abstention coverage uses every selected query\n      (predicted / n_selected).\n    * "eligible_*" subsets cover complete rows only (ranking-eligible).\n    * "refcount_1" / "refcount_multi" split eligible_anyref rows by pool\n      reference count (exactly one vs multiple reference-backed\n      candidates), so cosine-vs-coverage-only can be compared where the\n      coverage baseline is non-degenerate.\n    """\n    out = {"label": label, "n_selected": len(rows),\n           "n_complete": sum(1 for r in rows\n                             if r.get("status") == "complete"),\n           "status_counts": dict(Counter(r.get("status", "unknown")\n                                         for r in rows)),\n           "pool_recall_all": (sum(1 for r in rows if r.get("in_pool"))\n                               / len(rows) if rows else None)}\n    subsets = (("all", lambda r: True),\n               ("eligible_anyref",\n                lambda r: r.get("status") == "complete" and r["in_pool"]\n                and r["n_ref"] > 0),\n               ("eligible_targetref",\n                lambda r: r.get("status") == "complete" and r["in_pool"]\n                and r["target_has_ref"]),\n               ("refcount_1",\n                lambda r: r.get("status") == "complete" and r["in_pool"]\n                and r["n_ref"] == 1),\n               ("refcount_multi",\n                lambda r: r.get("status") == "complete" and r["in_pool"]\n                and r["n_ref"] > 1))\n    for name, sel in subsets:\n        sub = [r for r in rows if sel(r)]\n        m = {"n": len(sub)}\n        for arm in ARMS:\n            for k in (1, 3, 10):\n                vals = [r["hits"][arm][k] for r in sub\n                        if r["hits"].get(arm, {}).get(k) is not None]\n                m[f"{arm}@top{k}"] = (sum(vals) / len(vals)\n                                      if vals else None)\n                m[f"{arm}@top{k}_den"] = len(vals)\n        # Abstention (spectral arm): predict iff a complete row has >=1\n        # referenced candidate. The coverage denominator is ALL selected\n        # queries (parameter-free rule; excluded spectra count as\n        # abstentions, never as predictions).\n        pred = [r for r in sub if r.get("predicted")]\n        good = [r for r in pred if r["hits"].get("spectral", {}).get(1)]\n        m["predicted"] = len(pred)\n        m["coverage"] = len(pred) / len(rows) if rows else 0.0\n        m["abstention_precision"] = (len(good) / len(pred)\n                                     if pred else None)\n        out[name] = m\n    return out\n\n\ndef empty_hits():\n    return {arm: {1: None, 3: None, 10: None} for arm in ARMS}\n\n\ndef empty_ranks():\n    return {arm: None for arm in ARMS}\n\n\ndef full_row(qid, smiles, row, supplied, distinct, status):\n    """One row with the CONSISTENT schema every CSV row carries.\n\n    Spectrum-independent fields are filled for every status; ranking\n    fields stay None until computed. Uniform CSV columns are guaranteed\n    (the writer also unions fields as belt-and-braces).\n    """\n    return {\n        "qid": qid, "smiles": smiles,\n        "adduct": row.get("adduct", ""),\n        "instrument": row.get("instrument_type") or "missing",\n        "collision_energy": row.get("collision_energy", ""),\n        "status": status,\n        "n_supplied": len(supplied),\n        "n_pool": len(distinct),\n        "in_pool": (smiles in distinct),\n        "n_ref": None,\n        "target_has_ref": None,\n        "predicted": False,\n        "n_unparseable_mass": None,\n        "n_query_dropped_peaks": None,\n        "n_content_excluded_refs": None,\n        "n_content_excluded_structures": None,\n        "rank_uniform": None,\n        "rank_spectral": None,\n        "rank_coverage": None,\n        "rank_massresid": None,\n        "hits": empty_hits(),\n    }\n\n\ndef fill_spectrum_free_ranks(out, distinct, target_smiles, parent_mass):\n    """Uniform + massresid ranks need no spectrum; computable for excluded\n    queries too (documented; spectral/coverage stay None without scores)."""\n    pos_of = seeded_positions(out["qid"], len(distinct))\n    residuals, n_bad = candidate_residuals(distinct, parent_mass)\n    order_u = rank_uniform(\n        [{"idx": i} for i in range(len(distinct))], pos_of)\n    order_m = rank_massresid(\n        [{"idx": i} for i in range(len(distinct))], residuals, pos_of)\n    tgt = distinct.index(target_smiles) if target_smiles in distinct else None\n    if tgt is not None:\n        out["rank_uniform"] = order_u.index(tgt) + 1\n        out["rank_massresid"] = order_m.index(tgt) + 1\n        n = len(distinct)\n        for arm, rank in (("uniform", out["rank_uniform"]),\n                          ("massresid", out["rank_massresid"])):\n            out["hits"][arm] = {1: rank <= 1, 3: rank <= min(3, n),\n                                10: rank <= min(10, n)}\n    out["n_unparseable_mass"] = n_bad\n    return out\n\n\ndef run_val_experiment(tsv_path, prefix_path, max_queries):\n    """Val-side run with ENFORCED train/query disjointness.\n\n    Conservative connectivity policy: queries with a missing SMILES or\n    missing InChIKey-14 block are excluded before scoring\n    (status query_excl:missing_connectivity_key, counted); any nonzero\n    SMILES or InChIKey-14 overlap between kept queries and train aborts\n    the run (fail safe, never score through a leak).\n    """\n    # Val groups: first row per SMILES in file order (stable selection).\n    val_groups, train_smiles, train_keys = {}, set(), set()\n    val_bias = Counter()\n    n_rows = 0\n    n_train_missing_key = 0\n    for row in load_tsv_rows(tsv_path):\n        n_rows += 1\n        if row["fold"] == "train":\n            train_smiles.add(row["smiles"])\n            if row["inchikey"]:\n                train_keys.add(short_key(row["inchikey"]))\n            else:\n                n_train_missing_key += 1\n        elif row["fold"] == "val":\n            val_bias[f"adduct:{row[\'adduct\']}"] += 1\n            val_bias[f"inst:{row[\'instrument_type\'] or \'missing\'}"] += 1\n            if row["smiles"] not in val_groups:\n                val_groups[row["smiles"]] = row\n    entries = [(q, c) for q, c in\n               iter_json_entries(prefix_path, max_queries * 40)]\n    joined = [(q, c) for q, c in entries if q in val_groups][:max_queries]\n    if not joined:\n        raise ValueError("no prefix entries join to val rows")\n\n    # Distinct pools; split off queries lacking a connectivity identity.\n    pools, early_excluded = {}, []\n    for qi, (q, cands) in enumerate(joined):\n        qid = f"VAL-{qi:04d}"\n        seen, distinct = set(), []\n        for s in cands:\n            if s not in seen:\n                seen.add(s)\n                distinct.append(s)\n        rec = {"smiles": q, "supplied": list(cands),\n               "distinct": distinct, "row": val_groups[q]}\n        if not q or not val_groups[q]["inchikey"]:\n            out = full_row(qid, q, val_groups[q], rec["supplied"],\n                           distinct, "query_excl:missing_connectivity_key")\n            early_excluded.append(out)\n        else:\n            pools[qid] = rec\n\n    query_smiles = [v["smiles"] for v in pools.values()]\n    query_keys = {short_key(v["row"]["inchikey"]) for v in pools.values()}\n    ov_smi, ov_key = audit_split_disjointness(train_smiles, query_smiles,\n                                              train_keys, query_keys)\n    if ov_smi or ov_key:\n        raise ValueError(\n            f"train/query connectivity overlap: {len(ov_smi)} SMILES, "\n            f"{len(ov_key)} InChIKey-14 (fail safe, refusing to score)")\n\n    wanted = set()\n    for v in pools.values():\n        wanted.update(v["distinct"])\n    refs, ref_info = collect_train_refs(tsv_path, wanted,\n                                        exclude_smiles=set(query_smiles))\n    ref_info["n_train_missing_key"] = n_train_missing_key\n    ref_info["n_queries_missing_key"] = len(early_excluded)\n\n    rows = list(early_excluded)\n    sel_bias = Counter()\n    for qid, v in pools.items():\n        row = v["row"]\n        sel_bias[f"adduct:{row[\'adduct\']}"] += 1\n        sel_bias[f"inst:{row[\'instrument_type\'] or \'missing\'}"] += 1\n        out = full_row(qid, v["smiles"], row, v["supplied"], v["distinct"],\n                       "complete")\n        out = fill_spectrum_free_ranks(out, v["distinct"], v["smiles"],\n                                       row["parent_mass"])\n        qs = parse_query_spectrum(row)\n        if not qs["ok"]:\n            out["status"] = f"query_parse_excl:{qs[\'error\']}"\n            out["predicted"] = False  # abstention; in coverage denominator\n            out["n_ref"] = sum(1 for s in v["distinct"] if refs.get(s))\n            out["target_has_ref"] = bool(refs.get(v["smiles"]))\n            rows.append(out)\n            continue\n        refs_q, n_cx_r, n_cx_s = exclude_query_content(refs, qs["fp"])\n        out["n_content_excluded_refs"] = n_cx_r\n        out["n_content_excluded_structures"] = n_cx_s\n        ref_info["n_query_content_excluded_refs"] = (\n            ref_info.get("n_query_content_excluded_refs", 0) + n_cx_r)\n        scored = score_candidates(qs["vec"], qs["norm"], v["distinct"],\n                                  refs_q)\n        pos_of = seeded_positions(qid, len(scored))\n        residuals, n_bad = candidate_residuals(v["distinct"],\n                                               row["parent_mass"])\n        ev = evaluate_query(qid, scored, pos_of, residuals, v["smiles"])\n        out["n_ref"] = ev["n_ref"]\n        out["target_has_ref"] = ev["target_has_ref"]\n        out["predicted"] = ev["predicted"]\n        out["n_unparseable_mass"] = n_bad\n        out["n_query_dropped_peaks"] = qs["n_dropped"]\n        out["rank_uniform"] = ev["ranks"]["uniform"]\n        out["rank_spectral"] = ev["ranks"]["spectral"]\n        out["rank_coverage"] = ev["ranks"]["coverage"]\n        out["rank_massresid"] = ev["ranks"]["massresid"]\n        out["hits"] = ev["hits"]\n        rows.append(out)\n    # Reference-coverage detail over the SELECTED sample only (persisted\n    # here; no mixing with other sample sizes). Exact-string occurrence\n    # counts plus the crude no-stereo character approximation (rough\n    # sensitivity count, NOT a tight connectivity bound).\n    train_nostereo = set()\n    for row in load_tsv_rows(tsv_path):\n        if row["fold"] == "train":\n            train_nostereo.add(strip_stereo(row["smiles"]))\n    pool_occ, exact_cov, approx_extra = 0, 0, 0\n    for v in pools.values():\n        for s in v["distinct"]:\n            pool_occ += 1\n            if s in refs:\n                exact_cov += 1\n            elif strip_stereo(s) in train_nostereo:\n                approx_extra += 1\n    metrics = summarize(rows, "val_formula_pool")\n    return {"rows": rows, "metrics": metrics, "ref_info": dict(ref_info),\n            "n_tsv_rows": n_rows, "n_val_structures": len(val_groups),\n            "n_train_structures": len(train_smiles),\n            "n_entries_scanned": len(entries), "n_joined": len(joined),\n            "overlap_smiles": sorted(ov_smi), "overlap_keys": sorted(ov_key),\n            "overlap_enforced": True,\n            "val_bias": dict(val_bias), "selected_bias": dict(sel_bias),\n            "coverage_detail": {\n                "pool_distinct_occurrences": pool_occ,\n                "exact_covered_occurrences": exact_cov,\n                "nostereo_approx_extra": approx_extra,\n                "nostereo_approx_note": "character-level approximation, "\n                                        "not a connectivity bound",\n                "n_train_nostereo_keys": len(train_nostereo)}}\n\n\ndef run_control_experiment(tsv_path, prefix_path, max_queries):\n    """TRAIN-fold positive control (labelled CONTROL).\n\n    Train spectra as queries against remaining train references, excluding\n    the query identifier AND the query content fingerprint (no self-match\n    through duplication or re-upload). Not a val result. Control top-k is\n    reported next to the coverage-only arm and reference-count subgroups,\n    because most control pools hold a single referenced candidate and a\n    bare top-1 rate cannot establish representation quality.\n    """\n    train_groups = {}\n    for row in load_tsv_rows(tsv_path):\n        if row["fold"] == "train" and row["smiles"] not in train_groups:\n            train_groups[row["smiles"]] = row\n    entries = [(q, c) for q, c in\n               iter_json_entries(prefix_path, max_queries * 40)]\n    joined = [(q, c) for q, c in entries if q in train_groups][:max_queries]\n    if not joined:\n        raise ValueError("no prefix entries join to train rows")\n    pools = {}\n    for qi, (q, cands) in enumerate(joined):\n        qid = f"CTL-{qi:04d}"\n        seen, distinct = set(), []\n        for s in cands:\n            if s not in seen:\n                seen.add(s)\n                distinct.append(s)\n        pools[qid] = {"smiles": q, "supplied": list(cands),\n                      "distinct": distinct, "row": train_groups[q]}\n    wanted = set()\n    for v in pools.values():\n        wanted.update(v["distinct"])\n    # Refs: all wanted train spectra; the query identifier AND the query\n    # content fingerprint are excluded per query at scoring time.\n    all_refs, ref_info = collect_train_refs(tsv_path, wanted,\n                                            exclude_smiles=())\n    rows = []\n    for qid, v in pools.items():\n        row = v["row"]\n        out = full_row(qid, v["smiles"], row, v["supplied"], v["distinct"],\n                       "complete")\n        out = fill_spectrum_free_ranks(out, v["distinct"], v["smiles"],\n                                       row["parent_mass"])\n        qs = parse_query_spectrum(row)\n        if not qs["ok"]:\n            out["status"] = f"query_parse_excl:{qs[\'error\']}"\n            out["predicted"] = False\n            out["n_ref"] = sum(1 for s in v["distinct"] if all_refs.get(s))\n            out["target_has_ref"] = bool(all_refs.get(v["smiles"]))\n            rows.append(out)\n            continue\n        refs = {}\n        n_cx_r, n_cx_s = 0, 0\n        for smi, lst in all_refs.items():\n            kept = [t for t in lst\n                    if t[0] != row["identifier"] and t[6] != qs["fp"]]\n            dropped = len(lst) - len(kept)\n            if dropped:\n                n_cx_r += dropped\n                n_cx_s += 1\n            if kept:\n                refs[smi] = kept\n        out["n_content_excluded_refs"] = n_cx_r\n        out["n_content_excluded_structures"] = n_cx_s\n        ref_info["n_control_content_excluded_refs"] = (\n            ref_info.get("n_control_content_excluded_refs", 0) + n_cx_r)\n        scored = score_candidates(qs["vec"], qs["norm"], v["distinct"], refs)\n        pos_of = seeded_positions(qid, len(scored))\n        residuals, n_bad = candidate_residuals(v["distinct"],\n                                               row["parent_mass"])\n        ev = evaluate_query(qid, scored, pos_of, residuals, v["smiles"])\n        out["n_ref"] = ev["n_ref"]\n        out["target_has_ref"] = ev["target_has_ref"]\n        out["predicted"] = ev["predicted"]\n        out["n_unparseable_mass"] = n_bad\n        out["n_query_dropped_peaks"] = qs["n_dropped"]\n        out["rank_uniform"] = ev["ranks"]["uniform"]\n        out["rank_spectral"] = ev["ranks"]["spectral"]\n        out["rank_coverage"] = ev["ranks"]["coverage"]\n        out["rank_massresid"] = ev["ranks"]["massresid"]\n        out["hits"] = ev["hits"]\n        rows.append(out)\n    metrics = summarize(rows, "train_control")\n    return {"rows": rows, "metrics": metrics, "ref_info": dict(ref_info),\n            "n_joined": len(joined)}\n\n\ndef main(argv):\n    import argparse\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--tsv", default="data/pinned/MassSpecGym1.5.tsv")\n    ap.add_argument("--prefix",\n                    default="data/pinned/msgym_candidates_formula_prefix64.json")\n    ap.add_argument("--max-queries", type=int, default=200)\n    ap.add_argument("--max-control", type=int, default=50)\n    ap.add_argument("--out-dir",\n                    default="experiments/molecular_completion/20261003_spectral")\n    args = ap.parse_args(argv)\n    t0 = time.perf_counter()\n    c0 = time.process_time()\n    val = run_val_experiment(args.tsv, args.prefix, args.max_queries)\n    ctl = run_control_experiment(args.tsv, args.prefix, args.max_control)\n    # Hashing and row-output writes happen INSIDE the timed scope, so the\n    # reported wall/CPU/RAM covers the whole run (only the few-ms summary\n    # JSON write itself falls outside; documented in cost.note).\n    pins = {"tsv": args.tsv, "prefix": args.prefix,\n            "tsv_sha256": sha256_file(args.tsv),\n            "prefix_sha256": sha256_file(args.prefix),\n            "tsv_bytes": os.path.getsize(args.tsv),\n            "prefix_bytes": os.path.getsize(args.prefix)}\n    os.makedirs(args.out_dir, exist_ok=True)\n    rows_path = os.path.join(args.out_dir, "spectral_rows.csv")\n    ctl_path = os.path.join(args.out_dir, "spectral_control_rows.csv")\n    sum_path = os.path.join(args.out_dir, "spectral_summary.json")\n\n    def flatten(rows):\n        flat = []\n        for r in rows:\n            f = dict(r)\n            h = f.pop("hits", {})\n            for arm in ARMS:\n                for k in (1, 3, 10):\n                    v = (h.get(arm, {}).get(k) if h else None)\n                    f[f"hit_{arm}_top{k}"] = v\n            flat.append(f)\n        return flat\n\n    flat_val, flat_ctl = flatten(val["rows"]), flatten(ctl["rows"])\n    for path, flat in ((rows_path, flat_val), (ctl_path, flat_ctl)):\n        # Union of fields across rows: mixed complete/exclusion schemas can\n        # never crash the writer, and every row keeps the same columns.\n        cols = sorted({c for f in flat for c in f})\n        with open(path, "w", encoding="utf-8", newline="") as handle:\n            writer = csv.DictWriter(handle, fieldnames=cols)\n            writer.writeheader()\n            for f in flat:\n                writer.writerow({c: f.get(c) for c in cols})\n    wall = time.perf_counter() - t0\n    cpu = time.process_time() - c0\n    peak_kb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss\n    summary = {\n        "representation": {"bin_width": BIN_WIDTH, "mz_min": MZ_MIN,\n                           "mz_max": MZ_MAX, "n_bins": N_BINS,\n                           "aggregation": "per-bin max intensity",\n                           "similarity": "cosine on L2-normalized sparse "\n                                         "vectors, max over refs",\n                           "identity": "exact SMILES string (conservative "\n                                       "lower-bound coverage, not graph "\n                                       "canonicalization; no RDKit)",\n                           "shuffle": SHUFFLE_DOMAIN,\n                           "selection": "file-order prefix entries joining "\n                                        "fold rows; first row per SMILES",\n                           "fixed_a_priori": True,\n                           "train_calibration": "not applicable "\n                                                "(parameter-free abstention; "\n                                                "no thresholds fit)"},\n        "pins": pins,\n        "provenance": {"downloads_bytes": 0, "api_calls": 0,\n                       "test_fold": "streamed-but-unused: test-fold rows "\n                                    "pass through the shared TSV scanner "\n                                    "and are never used for fitting, "\n                                    "scoring, or selection"},\n        "params": {"max_queries": args.max_queries,\n                   "max_control": args.max_control},\n        "val": {k: v for k, v in val.items() if k != "rows"},\n        "control": {k: v for k, v in ctl.items() if k != "rows"},\n        "cost": {"wall_s": wall, "cpu_s": cpu,\n                 "peak_rss_kb": peak_kb,\n                 "note": "single-threaded CPU; stdlib only; timer covers "\n                         "experiment + sha256 verification + row CSV "\n                         "writes; only the summary-JSON write itself "\n                         "(ms-scale) falls outside"},\n        "outputs": {"rows": rows_path, "control_rows": ctl_path,\n                    "summary": sum_path,\n                    "rows_bytes": os.path.getsize(rows_path),\n                    "control_bytes": os.path.getsize(ctl_path),\n                    "summary_bytes_note": "stdout only: a file cannot "\n                                          "persist its own final size "\n                                          "before writing it"},\n    }\n    with open(sum_path, "w", encoding="utf-8") as handle:\n        json.dump(summary, handle, indent=2, default=str)\n        handle.write("\\n")\n    summary["outputs"]["summary_bytes"] = os.path.getsize(sum_path)\n    sys.stdout.write(json.dumps(summary, indent=2, default=str) + "\\n")\n\n\n# --------------------------------------------------------------------------\n# Tests.\n# --------------------------------------------------------------------------\n\nclass SpectrumParseTests(unittest.TestCase):\n    def test_valid(self):\n        peaks, err = parse_spectrum("100.0,200.5", "0.5,1.0")\n        self.assertIsNone(err)\n        self.assertEqual(peaks, [(100.0, 0.5), (200.5, 1.0)])\n\n    def test_empty(self):\n        for a, b in (("", ""), ("  ", "0.5"), ("100.0", ""),\n                     (None, "0.5"), ("100.0", None)):\n            _, err = parse_spectrum(a, b)\n            self.assertIsNotNone(err, (a, b))\n\n    def test_length_mismatch(self):\n        _, err = parse_spectrum("100.0,200.0", "0.5")\n        self.assertEqual(err, "length_mismatch")\n\n    def test_non_numeric(self):\n        _, err = parse_spectrum("100.0,abc", "0.5,1.0")\n        self.assertEqual(err, "non_numeric")\n\n    def test_non_finite(self):\n        _, err = parse_spectrum("100.0,nan", "0.5,1.0")\n        self.assertEqual(err, "non_finite")\n        _, err = parse_spectrum("100.0,inf", "0.5,1.0")\n        self.assertEqual(err, "non_finite")\n\n    def test_negative(self):\n        _, err = parse_spectrum("-5.0,100.0", "0.5,1.0")\n        self.assertEqual(err, "negative_value")\n        _, err = parse_spectrum("100.0,200.0", "0.5,-1.0")\n        self.assertEqual(err, "negative_value")\n\n    def test_zero_norm(self):\n        _, err = parse_spectrum("100.0,200.0", "0.0,0.0")\n        self.assertEqual(err, "zero_norm")\n\n\nclass BinCosineTests(unittest.TestCase):\n    def test_bin_edges(self):\n        vec, dropped = bin_spectrum([(0.0, 0.5), (0.05, 0.7), (1999.99, 1.0),\n                                     (2000.0, 1.0), (2500.0, 1.0)])\n        self.assertEqual(dropped, 2)  # upper edge + out of range\n        self.assertEqual(vec[0], 0.7)  # max aggregation within a bin\n        self.assertEqual(vec[N_BINS - 1], 1.0)\n\n    def test_identical_is_one(self):\n        vec, _ = bin_spectrum([(100.0, 0.5), (200.0, 1.0)])\n        n = vec_norm(vec)\n        self.assertAlmostEqual(cosine(vec, n, vec, n), 1.0)\n\n    def test_orthogonal_is_zero(self):\n        a, _ = bin_spectrum([(100.0, 1.0)])\n        b, _ = bin_spectrum([(300.0, 1.0)])\n        self.assertEqual(cosine(a, vec_norm(a), b, vec_norm(b)), 0.0)\n\n    def test_scale_invariant(self):\n        a, _ = bin_spectrum([(100.0, 0.5)])\n        b, _ = bin_spectrum([(100.0, 2.0)])\n        self.assertAlmostEqual(cosine(a, vec_norm(a), b, vec_norm(b)), 1.0)\n\n    def test_zero_vector_scores_zero(self):\n        a, dropped = bin_spectrum([(9999.0, 1.0)])\n        self.assertEqual(a, {})\n        self.assertEqual(dropped, 1)\n        b, _ = bin_spectrum([(100.0, 1.0)])\n        self.assertEqual(cosine(a, 0.0, b, vec_norm(b)), 0.0)\n        self.assertEqual(cosine(a, 0.0, a, 0.0), 0.0)\n\n    def test_partial_overlap_bounded(self):\n        a, _ = bin_spectrum([(100.0, 1.0), (200.0, 1.0)])\n        b, _ = bin_spectrum([(100.0, 1.0), (300.0, 1.0)])\n        s = cosine(a, vec_norm(a), b, vec_norm(b))\n        self.assertAlmostEqual(s, 0.5)\n\n\nclass RankBehaviorTests(unittest.TestCase):\n    def _scored(self):\n        return [\n            {"idx": 0, "smiles": "A", "score": 0.9, "n_refs": 2,\n             "best_ref": "r1"},\n            {"idx": 1, "smiles": "B", "score": 0.2, "n_refs": 1,\n             "best_ref": "r2"},\n            {"idx": 2, "smiles": "C", "score": None, "n_refs": 0,\n             "best_ref": None},\n            {"idx": 3, "smiles": "D", "score": None, "n_refs": 0,\n             "best_ref": None},\n        ]\n\n    def test_max_aggregation_picks_best_ref(self):\n        q = {10: 1.0}\n        refs = {"X": [("r1", {10: 0.1, 20: 1.0}, math.sqrt(1.01), 0, "", "",\n                      ()),\n                      ("r2", {10: 1.0}, 1.0, 0, "", "", ())]}\n        out = score_candidates(q, 1.0, ["X"], refs)\n        self.assertEqual(out[0]["n_refs"], 2)\n        self.assertAlmostEqual(out[0]["score"], 1.0)\n        self.assertEqual(out[0]["best_ref"], "r2")\n\n    def test_missing_refs_rank_last_but_retained(self):\n        scored = self._scored()\n        pos = seeded_positions("Q", len(scored))\n        order = rank_spectral(scored, pos)\n        self.assertEqual([scored[i]["smiles"] for i in order[:2]], ["A", "B"])\n        # Unreferenced candidates stay in the ranking (never dropped).\n        self.assertEqual(len(order), 4)\n        self.assertEqual(set(order), {0, 1, 2, 3})\n        tail = [scored[i]["smiles"] for i in order[2:]]\n        self.assertEqual(sorted(tail), ["C", "D"])\n\n    def test_ties_break_by_seeded_position(self):\n        scored = [\n            {"idx": 0, "smiles": "A", "score": 0.5, "n_refs": 1,\n             "best_ref": "r"},\n            {"idx": 1, "smiles": "B", "score": 0.5, "n_refs": 1,\n             "best_ref": "r"},\n        ]\n        o1 = rank_spectral(scored, seeded_positions("Q1", 2))\n        o2 = rank_spectral(scored, seeded_positions("Q1", 2))\n        self.assertEqual(o1, o2)  # deterministic per qid\n        # Order follows shuffled position, never ambient index order alone:\n        pos = seeded_positions("Q1", 2)\n        expect_first = 0 if pos[0] < pos[1] else 1\n        self.assertEqual(o1[0], expect_first)\n\n    def test_absent_target_ranks_none(self):\n        scored = self._scored()\n        pos = seeded_positions("Q", len(scored))\n        ev = evaluate_query("Q", scored, pos, [1.0, 2.0, 3.0, 4.0], "ZZZ")\n        self.assertFalse(ev["in_pool"])\n        self.assertIsNone(ev["ranks"]["spectral"])\n        self.assertIsNone(ev["hits"]["spectral"][1])\n        self.assertIsNone(ev["hits"]["uniform"][10])\n\n    def test_uniform_uses_shuffle_not_ambient(self):\n        scored = self._scored()\n        o = rank_uniform(scored, seeded_positions("Q7", len(scored)))\n        self.assertEqual(sorted(o), [0, 1, 2, 3])\n        # Seeded shuffle of 4 is overwhelmingly likely non-identity; the\n        # robust assertion is determinism + position-dependence instead:\n        self.assertEqual(o, rank_uniform(scored, seeded_positions("Q7", 4)))\n\n    def test_identity_is_exact_string(self):\n        # Stereo variants are DIFFERENT identities (conservative lower\n        # bound on coverage): \'C[C@H](N)O\' != \'C[C@@H](N)O\'.\n        self.assertNotEqual("C[C@H](N)O", "C[C@@H](N)O")\n        self.assertEqual(strip_stereo("C[C@H](N)O"),\n                         strip_stereo("C[C@@H](N)O"))\n\n    def test_massresid_unparseable_last(self):\n        scored = self._scored()\n        pos = seeded_positions("Q", len(scored))\n        order = rank_massresid(scored, [5.0, None, 0.1, None], pos)\n        self.assertEqual(scored[order[0]]["smiles"], "C")  # 0.1 ppm first\n        self.assertEqual(scored[order[1]]["smiles"], "A")  # 5.0 ppm next\n        # Unparseable (None) trail, retained.\n        self.assertEqual(len(order), 4)\n\n\nclass LeakageTests(unittest.TestCase):\n    def test_overlap_audit_detects(self):\n        ov_smi, ov_key = audit_split_disjointness(\n            {"A", "B"}, ["B", "C"], {"k1", "k2"}, {"k2", "k3"})\n        self.assertEqual(ov_smi, {"B"})\n        self.assertEqual(ov_key, {"k2"})\n\n    def test_no_oracle_query_features(self):\n        # score_candidates must not read target/is_target style fields.\n        import inspect\n        src = inspect.getsource(score_candidates)\n        self.assertNotIn("is_target", src)\n        self.assertNotIn("target_smiles", src)\n\n    def test_excluded_queries_never_scored(self):\n        # File-backed: a train row whose SMILES is in exclude_smiles is\n        # counted as a leak exclusion and never lands in refs.\n        import tempfile\n        header = "\\t".join(WANTED_COLUMNS)\n        rows = [\n            ["T1", "100.0", "1.0", "CCO", "K" * 14, "C2H6O", "C2H7O",\n             "46.0", "47.0", "[M+H]+", "QTOF", "20", "train"],\n            ["T2", "200.0", "1.0", "CCC", "J" * 14, "C3H8", "C3H9",\n             "44.0", "45.0", "[M+H]+", "QTOF", "20", "train"],\n        ]\n        with tempfile.NamedTemporaryFile("w", suffix=".tsv",\n                                         delete=False) as handle:\n            handle.write(header + "\\n")\n            for r in rows:\n                handle.write("\\t".join(r) + "\\n")\n            path = handle.name\n        try:\n            refs, info = collect_train_refs(path, {"CCO", "CCC"},\n                                            exclude_smiles={"CCO"})\n        finally:\n            os.unlink(path)\n        self.assertNotIn("CCO", refs)\n        self.assertIn("CCC", refs)\n        self.assertEqual(info["n_leak_excluded"], 1)\n\n    def test_duplicate_ids_first_wins(self):\n        import tempfile\n        header = "\\t".join(WANTED_COLUMNS)\n        rows = [\n            ["DUP", "100.0", "1.0", "CCO", "K" * 14, "C2H6O", "C2H7O",\n             "46.0", "47.0", "[M+H]+", "QTOF", "20", "train"],\n            ["DUP", "300.0", "1.0", "CCO", "K" * 14, "C2H6O", "C2H7O",\n             "46.0", "47.0", "[M+H]+", "QTOF", "20", "train"],\n        ]\n        with tempfile.NamedTemporaryFile("w", suffix=".tsv",\n                                         delete=False) as handle:\n            handle.write(header + "\\n")\n            for r in rows:\n                handle.write("\\t".join(r) + "\\n")\n            path = handle.name\n        try:\n            refs, info = collect_train_refs(path, {"CCO"})\n        finally:\n            os.unlink(path)\n        self.assertEqual(len(refs["CCO"]), 1)\n        self.assertEqual(info["n_duplicate_ids"], 1)\n        # First occurrence wins.\n        self.assertEqual(refs["CCO"][0][0], "DUP")\n        self.assertIn(1000, refs["CCO"][0][1])  # 100.0 Da bin\n\n\nclass ContentGuardTests(unittest.TestCase):\n    def test_fingerprint_equal_content(self):\n        self.assertEqual(content_fp([(100.0, 0.5), (200.0, 1.0)]),\n                         content_fp([(200.0, 1.0), (100.0, 0.5)]))\n        self.assertNotEqual(content_fp([(100.0, 0.5)]),\n                            content_fp([(100.05, 0.5)]))\n\n    def test_exclude_query_content_different_id(self):\n        fp_q = content_fp([(100.0, 1.0)])\n        refs = {"X": [("r1", {1000: 1.0}, 1.0, 0, "", "", fp_q),\n                      ("r2", {2000: 1.0}, 1.0, 0, "", "",\n                       content_fp([(200.0, 1.0)]))],\n                "Y": [("r3", {1000: 1.0}, 1.0, 0, "", "", fp_q)]}\n        kept, n_r, n_s = exclude_query_content(refs, fp_q)\n        self.assertEqual(n_r, 2)\n        self.assertEqual(n_s, 2)\n        self.assertEqual(kept, {"X": [refs["X"][1]]})\n\n    def test_collect_dedups_content_within_structure(self):\n        import tempfile\n        header = "\\t".join(WANTED_COLUMNS)\n        rows = [\n            ["A1", "100.0,200.0", "0.5,1.0", "CCO", "K" * 14, "C2H6O",\n             "C2H7O", "46.0", "47.0", "[M+H]+", "QTOF", "20", "train"],\n            ["A2", "100.0,200.0", "0.5,1.0", "CCO", "K" * 14, "C2H6O",\n             "C2H7O", "46.0", "47.0", "[M+H]+", "QTOF", "20", "train"],\n            ["A3", "150.0", "1.0", "CCO", "K" * 14, "C2H6O", "C2H7O",\n             "46.0", "47.0", "[M+H]+", "QTOF", "20", "train"],\n        ]\n        with tempfile.NamedTemporaryFile("w", suffix=".tsv",\n                                         delete=False) as handle:\n            handle.write(header + "\\n")\n            for r in rows:\n                handle.write("\\t".join(r) + "\\n")\n            path = handle.name\n        try:\n            refs, info = collect_train_refs(path, {"CCO"})\n        finally:\n            os.unlink(path)\n        self.assertEqual(len(refs["CCO"]), 2)  # A1 + A3; A2 deduped\n        self.assertEqual(info["n_duplicate_content"], 1)\n        self.assertEqual(info.get("n_duplicate_ids", 0), 0)\n\n\nclass CoverageBaselineTests(unittest.TestCase):\n    def test_referenced_first_no_scores(self):\n        scored = [\n            {"idx": 0, "smiles": "A", "score": 0.99, "n_refs": 0,\n             "best_ref": None},\n            {"idx": 1, "smiles": "B", "score": None, "n_refs": 0,\n             "best_ref": None},\n            {"idx": 2, "smiles": "C", "score": 0.01, "n_refs": 3,\n             "best_ref": "r"},\n        ]\n        pos = seeded_positions("QC", 3)\n        order = rank_coverage(scored, pos)\n        # Only referenced candidate first regardless of (low) score.\n        self.assertEqual(order[0], 2)\n        self.assertEqual(sorted(order), [0, 1, 2])\n        import inspect\n        src = inspect.getsource(rank_coverage)\n        self.assertNotIn(\'["score"]\', src)\n        self.assertNotIn("[\'score\']", src)\n\n    def test_coverage_arm_in_evaluate(self):\n        scored = [\n            {"idx": 0, "smiles": "A", "score": 0.9, "n_refs": 1,\n             "best_ref": "r"},\n            {"idx": 1, "smiles": "T", "score": 0.1, "n_refs": 1,\n             "best_ref": "r"},\n        ]\n        ev = evaluate_query("Q", scored, seeded_positions("Q", 2),\n                            [1.0, 1.0], "T")\n        # Spectral follows scores (A first); coverage follows shuffle among\n        # referenced (may or may not match) -- the recorded distinction is\n        # that both ranks exist on identical pools.\n        self.assertIn("coverage", ev["ranks"])\n        self.assertIn("coverage", ev["hits"])\n        self.assertIsNotNone(ev["ranks"]["coverage"])\n\n\nclass DenominatorTests(unittest.TestCase):\n    def _rows(self):\n        good = full_row("VAL-0000", "T", {"adduct": "a",\n                                             "instrument_type": "i",\n                                             "collision_energy": "10",\n                                             "parent_mass": "46.0"},\n                        ["T", "U"], ["T", "U"], "complete")\n        good.update({"n_ref": 1, "target_has_ref": False, "predicted": True,\n                     "hits": {"uniform": {1: False, 3: True, 10: True},\n                              "spectral": {1: False, 3: False, 10: True},\n                              "coverage": {1: False, 3: True, 10: True},\n                              "massresid": {1: True, 3: True, 10: True}}})\n        bad = full_row("VAL-0001", "T2", {"adduct": "a",\n                                              "instrument_type": "i",\n                                              "collision_energy": "10",\n                                              "parent_mass": "46.0"},\n                       ["T2", "U"], ["T2", "U"],\n                       "query_parse_excl:non_numeric")\n        bad.update({"n_ref": 1, "target_has_ref": False})\n        return [good, bad]\n\n    def test_all_covers_selected_excluded_in_denominators(self):\n        m = summarize(self._rows(), "t")\n        self.assertEqual(m["n_selected"], 2)\n        self.assertEqual(m["n_complete"], 1)\n        self.assertEqual(m["pool_recall_all"], 1.0)  # both targets in pools\n        # Coverage over ALL selected (excluded counts as abstention).\n        self.assertEqual(m["all"]["coverage"], 0.5)\n        # Top-k denominators count rankable rows only.\n        self.assertEqual(m["all"]["spectral@top1_den"], 1)\n        self.assertEqual(m["eligible_anyref"]["n"], 1)\n\n    def test_refcount_subgroups(self):\n        rows = self._rows()\n        rows[0]["n_ref"] = 2\n        m = summarize(rows, "t")\n        self.assertEqual(m["refcount_multi"]["n"], 1)\n        self.assertEqual(m["refcount_1"]["n"], 0)\n\n    def test_strip_stereo_documented_approximation(self):\n        import inspect\n        self.assertIn("APPROXIMATION",\n                      inspect.getdoc(strip_stereo).upper())\n\n\ndef _write_tsv(path, rows):\n    with open(path, "w", encoding="utf-8", newline="") as handle:\n        handle.write("\\t".join(WANTED_COLUMNS) + "\\n")\n        for r in rows:\n            handle.write("\\t".join(r) + "\\n")\n\n\ndef _row(ident, mzs, ints, smiles, key, fold, parent="46.041864"):\n    return [ident, mzs, ints, smiles, key, "C2H6O", "C2H7O", parent, "47.0",\n            "[M+H]+", "QTOF", "20", fold]\n\n\nclass EnforcementTests(unittest.TestCase):\n    def test_overlap_aborts(self):\n        import tempfile\n        with tempfile.TemporaryDirectory() as tmp:\n            tsv = os.path.join(tmp, "t.tsv")\n            pre = os.path.join(tmp, "p.json")\n            _write_tsv(tsv, [\n                _row("T1", "100.0", "1.0", "CCO", "K" * 14, "train"),\n                _row("V1", "200.0", "1.0", "CCO", "K" * 14, "val"),\n            ])\n            with open(pre, "w", encoding="utf-8") as handle:\n                json.dump({"CCO": ["CCO", "CCC"]}, handle)\n            with self.assertRaises(ValueError):\n                run_val_experiment(tsv, pre, 10)\n\n    def test_missing_key_excluded_not_scored(self):\n        import tempfile\n        with tempfile.TemporaryDirectory() as tmp:\n            tsv = os.path.join(tmp, "t.tsv")\n            pre = os.path.join(tmp, "p.json")\n            _write_tsv(tsv, [\n                _row("T1", "100.0", "1.0", "CCO", "K" * 14, "train"),\n                _row("V1", "200.0", "1.0", "CCN", "", "val"),\n            ])\n            with open(pre, "w", encoding="utf-8") as handle:\n                json.dump({"CCN": ["CCN", "CCO"]}, handle)\n            out = run_val_experiment(tsv, pre, 10)\n            self.assertEqual(len(out["rows"]), 1)\n            self.assertEqual(out["rows"][0]["status"],\n                             "query_excl:missing_connectivity_key")\n            self.assertEqual(out["ref_info"]["n_queries_missing_key"], 1)\n            self.assertEqual(out["metrics"]["n_selected"], 1)\n            self.assertEqual(out["metrics"]["n_complete"], 0)\n\n\nclass IntegrationTests(unittest.TestCase):\n    def test_malformed_first_and_later_full_write(self):\n        import tempfile\n        with tempfile.TemporaryDirectory() as tmp:\n            tsv = os.path.join(tmp, "t.tsv")\n            pre = os.path.join(tmp, "p.json")\n            out = os.path.join(tmp, "out")\n            _write_tsv(tsv, [\n                _row("T1", "100.0,200.0", "0.5,1.0", "CCO", "K" * 14,\n                     "train"),\n                _row("T2", "100.0,200.0", "0.5,1.0", "CCO", "K" * 14,\n                     "train"),\n                _row("T3", "150.0,250.0", "1.0,0.5", "CCO", "K" * 14,\n                     "train"),\n                _row("T4", "300.0", "1.0", "CCC", "J" * 14, "train"),\n                _row("Vbad1", "xx", "1.0", "BAD1", "B" * 14, "val"),\n                _row("Vq", "110.0,210.0", "0.6,0.9", "CCN", "N" * 14,\n                     "val"),\n                _row("Vbad2", "100.0", "", "BAD2", "C" * 14, "val"),\n            ])\n            with open(pre, "w", encoding="utf-8") as handle:\n                json.dump({"BAD1": ["CCO", "CCC"],\n                           "CCN": ["CCO", "CCC", "CCN"],\n                           "BAD2": ["CCO", "CCC"],\n                           "CCO": ["CCO", "CCC"]}, handle)\n            main(["--tsv", tsv, "--prefix", pre, "--max-queries", "3",\n                  "--max-control", "1", "--out-dir", out])\n            val_rows = list(csv.DictReader(\n                open(os.path.join(out, "spectral_rows.csv"))))\n            ctl_rows = list(csv.DictReader(\n                open(os.path.join(out, "spectral_control_rows.csv"))))\n            summary = json.load(\n                open(os.path.join(out, "spectral_summary.json")))\n            # Mixed complete/exclusion schemas share one column set.\n            self.assertEqual(len(val_rows), 3)\n            for r in val_rows:\n                self.assertEqual(set(r.keys()), set(val_rows[0].keys()))\n            for r in ctl_rows:\n                self.assertEqual(set(r.keys()), set(ctl_rows[0].keys()))\n            statuses = [r["status"] for r in val_rows]\n            self.assertEqual(statuses[0], "query_parse_excl:non_numeric")\n            self.assertEqual(statuses[1], "complete")\n            self.assertEqual(statuses[2], "query_parse_excl:empty_spectrum")\n            # Selected/complete/excluded accounting + pool recall on all.\n            m = summary["val"]["metrics"]\n            self.assertEqual(m["n_selected"], 3)\n            self.assertEqual(m["n_complete"], 1)\n            self.assertEqual(m["pool_recall_all"], 1 / 3)\n            # Abstention coverage denominator includes excluded spectra.\n            self.assertEqual(m["all"]["coverage"],\n                             m["all"]["predicted"] / 3)\n            # Machine-readable output accounting persisted.\n            self.assertIn("rows_bytes", summary["outputs"])\n            self.assertIn("control_bytes", summary["outputs"])\n            self.assertEqual(summary["provenance"]["api_calls"], 0)\n            self.assertEqual(summary["provenance"]["downloads_bytes"], 0)\n            # Content-dedup counted (T2 duplicates T1 content).\n            self.assertEqual(summary["val"]["ref_info"]\n                             ["n_duplicate_content"], 1)\n\n\nif __name__ == "__main__":\n    main(sys.argv[1:])\n', 'tools/ms2_msgym_corpus.py': '"""Real-corpus ingestion for the database-first retrieval harness.\n\nExtends tools/ms2_database_retrieval.py (imported, not modified) with:\n\n- a restricted, stdlib-only SMILES parser that accepts a documented subset\n  and rejects everything else with an explicit reason (aromaticity,\n  charges, non-domain elements, bracket anomalies, syntax errors). Rejected\n  inputs are a measured denominator, never silently reinterpreted;\n- a Hill-formula parser used to cross-check the graph-derived composition;\n- an incremental scanner for the large MassSpecGym candidate JSONs so a\n  capped prefix download yields complete per-query entries;\n- a pinned, capped MassSpecGym val-fold driver that joins supplied\n  candidate pools to measured-spectrum metadata and runs the same four\n  staged filters as the synthetic harness.\n\nRun:   python3 tools/ms2_msgym_corpus.py --prefix data/pinned/msgym_candidates_formula_prefix.json \\\\\n           --tsv data/pinned/MassSpecGym1.5.tsv --max-queries 150\nTest:  python3 -m unittest tools.ms2_msgym_corpus\n\nScope notes (see docs/MOLECULAR_COMPLETION_DATABASE_EXPERIMENTS.md):\n- Target-parent reconstruction only. Peak masses do not identify fragment\n  connectivity, so no fragment-graph claims are made.\n- Whole-target requests: the precursor measurement is redundant with the\n  target evidence, so the stage-4 precursor arm is reported not_evaluated.\n- Aromatic records are outside the current closed-molecule typed-graph\n  domain (no kekulization is performed; silent kekulization would invent\n  bond orders). They are counted as exclusions.\n- Top-k is reported only when the target is present in the supplied pool.\n"""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport os\nimport sys\nimport time\nimport unittest\nfrom collections import Counter\n\nsys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))\n\nfrom tools.ms2_database_retrieval import (\n    DOMAIN_ELEMENTS,\n    DatabaseIndex,\n    WorkCounters,\n    edge_fingerprint,\n    formula_key,\n    formula_of,\n    joint_contains,\n    mass_verdict,\n    neutral_mass,\n    rank_baselines,\n    run_query,\n    standardize_record,\n    contains_exact,\n)\n\nVALENCE = {"C": 4, "N": 3, "O": 2, "F": 1, "Cl": 1, "Br": 1, "I": 1, "S": 2}\nAROMATIC_VALENCE = {"C": 3, "N": 3, "O": 2, "S": 2}\nDOMAIN_SET = {"C", "N", "O", "F", "Cl", "Br", "I", "S"}\nAROMATIC_ATOMS = set("cnops")\nORGANIC_ATOMS = {\n    "B", "C", "N", "O", "P", "S", "F", "Cl", "Br", "I",\n}\n\n\nclass SmilesUnsupported(Exception):\n    def __init__(self, reason):\n        super().__init__(reason)\n        self.reason = reason\n\n\ndef _read_atom(smi, i):\n    """Parse one atom token at position i. Returns (atom_dict, next_i)."""\n    n = len(smi)\n    c = smi[i]\n    if c == "[":\n        j = smi.index("]", i)\n        body = smi[i + 1:j]\n        # Strip stereochemistry markers (connectivity identity only).\n        body = body.replace("@", "")\n        k = 0\n        while k < len(body) and not body[k].isalpha():\n            k += 1\n        # A leading number is an isotope label ([13C], [2H]): normalize-and-\n        # forget would silently compute the unlabeled mass, so reject.\n        if any(ch.isdigit() for ch in body[:k]):\n            raise SmilesUnsupported("unsupported_isotope")\n        aromatic = False\n        if k < len(body) and body[k].islower():\n            aromatic = True\n            elem = body[k].upper()\n            k += 1\n        elif k < len(body) and body[k].isupper():\n            elem = body[k]\n            k += 1\n            if k < len(body) and body[k].islower():\n                elem += body[k]\n                k += 1\n        else:\n            raise SmilesUnsupported(f"bracket_feature:{body!r}")\n        rest = body[k:]\n        hydrogens = 0\n        charge = 0\n        p = 0\n        while p < len(rest):\n            if rest[p] == "H":\n                p += 1\n                num = ""\n                while p < len(rest) and rest[p].isdigit():\n                    num += rest[p]\n                    p += 1\n                hydrogens = int(num) if num else 1\n            elif rest[p] in "+-":\n                sign = 1 if rest[p] == "+" else -1\n                p += 1\n                num = ""\n                while p < len(rest) and rest[p].isdigit():\n                    num += rest[p]\n                    p += 1\n                charge += sign * (int(num) if num else 1)\n            else:\n                raise SmilesUnsupported(f"bracket_feature:{rest!r}")\n        return ({"element": elem, "aromatic": aromatic, "bracket": True,\n                 "hydrogens": hydrogens, "charge": charge,\n                 "valence": AROMATIC_VALENCE.get(elem, VALENCE.get(elem, 0)) if aromatic\n                 else VALENCE.get(elem, 0)}, j + 1)\n    if c.islower():\n        return ({"element": c.upper(), "aromatic": True, "bracket": False,\n                 "hydrogens": 0, "charge": 0,\n                 "valence": AROMATIC_VALENCE.get(c.upper(), 0)}, i + 1)\n    if c.isupper():\n        if c in ("Cl",) or (c == "C" and i + 1 < n and smi[i + 1] == "l"):\n            return ({"element": "Cl", "aromatic": False, "bracket": False,\n                     "hydrogens": 0, "charge": 0, "valence": 1}, i + 2)\n        if c == "B" and i + 1 < n and smi[i + 1] == "r":\n            return ({"element": "Br", "aromatic": False, "bracket": False,\n                     "hydrogens": 0, "charge": 0, "valence": 1}, i + 2)\n        if c == "I":\n            return ({"element": "I", "aromatic": False, "bracket": False,\n                     "hydrogens": 0, "charge": 0, "valence": 1}, i + 1)\n        if c == "S":\n            return ({"element": "S", "aromatic": False, "bracket": False,\n                     "hydrogens": 0, "charge": 0, "valence": 2}, i + 1)\n        return ({"element": c, "aromatic": False, "bracket": False,\n                 "hydrogens": 0, "charge": 0,\n                 "valence": VALENCE.get(c, 0)}, i + 1)\n    raise SmilesUnsupported(f"unexpected_char:{c!r}")\n\n\ndef parse_smiles(smi):\n    """Parse SMILES into (atoms, bonds).\n\n    atoms: list of dicts with element/aromatic/bracket/hydrogens/charge/\n    valence. bonds: list of (a, b, order) with order in (1, 2, 3, 4 =\n    aromatic). Raises SmilesUnsupported for anything outside the subset.\n    """\n    from tools.ms2_database_retrieval import AROM_ORDER\n    if not smi:\n        raise SmilesUnsupported("empty_smiles")\n    if "." in smi:\n        raise SmilesUnsupported("disconnected_or_salt")\n    atoms, bonds = [], []\n    stack = []\n    ring = {}\n    prev = None\n    pending_bond = None\n    i, n = 0, len(smi)\n\n    def add_bond(a, b, order):\n        if a is None or b is None:\n            raise SmilesUnsupported("dangling_bond")\n        bonds.append((a, b, order))\n\n    def close_ring(num, order):\n        # Default ring-bond order resolves at close time, when both endpoint\n        # aromaticity flags are known (fixes aromatic closures).\n        if num in ring:\n            other, stored = ring.pop(num)\n            if order is not None:\n                resolved = order\n            elif stored is not None:\n                resolved = stored\n            elif atoms[prev]["aromatic"] and atoms[other]["aromatic"]:\n                resolved = AROM_ORDER\n            else:\n                resolved = 1\n            add_bond(prev, other, resolved)\n        else:\n            ring[num] = (prev, order)\n\n    while i < n:\n        c = smi[i]\n        if c == "(":\n            stack.append((prev, pending_bond))\n            pending_bond = None\n            i += 1\n        elif c == ")":\n            if not stack:\n                raise SmilesUnsupported("unbalanced_paren")\n            prev, pending_bond = stack.pop()\n            i += 1\n        elif c in "-=#:/\\\\":\n            if c == "=":\n                pending_bond = 2\n            elif c == "#":\n                pending_bond = 3\n            elif c == ":":\n                pending_bond = AROM_ORDER\n            else:\n                pending_bond = 1\n            i += 1\n        elif c == "%":\n            digits = smi[i + 1:i + 3]\n            if len(digits) != 2 or not digits.isdigit():\n                raise SmilesUnsupported("bad_ring_closure")\n            num = int(digits)\n            order = pending_bond\n            pending_bond = None\n            close_ring(num, order)\n            i += 3\n        elif c.isdigit():\n            num = int(c)\n            order = pending_bond\n            pending_bond = None\n            close_ring(num, order)\n            i += 1\n        else:\n            atom, i = _read_atom(smi, i)\n            idx = len(atoms)\n            atoms.append(atom)\n            if prev is not None:\n                if pending_bond is not None:\n                    order = pending_bond\n                elif atom["aromatic"] and atoms[prev]["aromatic"]:\n                    order = AROM_ORDER\n                else:\n                    order = 1\n                add_bond(prev, idx, order)\n            pending_bond = None\n            prev = idx\n    if stack:\n        raise SmilesUnsupported("unbalanced_paren")\n    if ring:\n        raise SmilesUnsupported("unclosed_ring")\n    if not atoms:\n        raise SmilesUnsupported("empty_smiles")\n    return atoms, bonds\n\n\ndef _aromatic_type(el, bracket_h, n_arom, n_other, vsum, std_valence,\n                   has_exo_multiple):\n    """Resolve (h, valence) for an aromatic atom or raise SmilesUnsupported.\n\n    bracket_h: stated H count, or None for bare atoms. Bare `n` with two\n    aromatic-only connections follows the RDKit convention (pyridine,\n    H=0); this is sound for RDKit-canonical corpora and recorded as a\n    corpus dependence. Anything structurally ambiguous is rejected, not\n    guessed.\n\n    Atoms with an exocyclic multiple bond (e.g. a pyrone carbonyl carbon,\n    which RDKit still flags aromatic) cannot use aromatic valence 3: they\n    fall back to aliphatic typing with the standard valence, which the\n    uniform residual rule then validates. `vsum` uses ORDER_VALUE and\n    `std_valence` is the aliphatic valence (C4 N3 O2 S2).\n    """\n    total = n_arom + n_other\n    if has_exo_multiple and bracket_h is None and el in ("C", "N", "O", "S"):\n        h = std_valence - vsum\n        if h < 0:\n            raise SmilesUnsupported("hypervalent")\n        return h, std_valence\n    if el == "C":\n        if bracket_h is not None:\n            if bracket_h != 3 - total:\n                raise SmilesUnsupported("bracket_valence")\n            return bracket_h, 3\n        if total < 2:\n            raise SmilesUnsupported("ambiguous_aromatic")\n        h = 3 - total\n        if h < 0:\n            raise SmilesUnsupported("hypervalent")\n        return h, 3\n    if el == "N":\n        if bracket_h is not None:\n            return bracket_h, 3 if bracket_h > 0 else 2\n        if n_arom == 2 and n_other == 0:\n            return 0, 2\n        if n_arom >= 2 and total == 3:\n            return 0, 3\n        raise SmilesUnsupported("ambiguous_aromatic")\n    if el in ("O", "S"):\n        if bracket_h is not None:\n            return bracket_h, 2\n        if total == 2:\n            return 0, 2\n        raise SmilesUnsupported("ambiguous_aromatic")\n    raise SmilesUnsupported(f"ambiguous_aromatic:{el}")\n\n\ndef typed_from_parsed(atoms, bonds):\n    """Shared typed-graph builder for SMILES and SDF front ends.\n\n    atoms: list of dicts with element/aromatic/bracket/hydrogens/charge/\n    valence. bonds: list of (a, b, order) with order in (1, 2, 3, 4).\n    Returns (atom_types, edges) or (None, exclusion_reason). Hydrogens are\n    parent-relative counts derived here, never invented downstream.\n    Valence sums use ORDER_VALUE (aromatic code counts one unit).\n    """\n    from tools.ms2_database_retrieval import AROM_ORDER, ORDER_VALUE\n\n    for a, b, o in bonds:\n        if o == AROM_ORDER:\n            # Aromatic bonds need aromatic-flagged endpoints (SMILES\n            # lowercase). V2000 molfiles carry no such flags, so SDF bond\n            # type 4 stays rejected; likewise explicit \':\' on aliphatics.\n            if not (atoms[a]["aromatic"] and atoms[b]["aromatic"]):\n                return None, "aromatic_bond"\n            continue\n        if o not in (1, 2, 3):\n            return None, f"unsupported_bond_type:{o}"\n    for a in atoms:\n        if a["charge"] != 0:\n            return None, "unsupported_charge"\n        if a["element"] not in DOMAIN_SET:\n            return None, f"unsupported_element:{a[\'element\']}"\n    n = len(atoms)\n    vsum = [0] * n\n    narom = [0] * n\n    nother = [0] * n\n    exomult = [False] * n\n    for a, b, o in bonds:\n        try:\n            value = ORDER_VALUE[o]\n        except KeyError:\n            return None, f"unsupported_bond_type:{o}"\n        vsum[a] += value\n        vsum[b] += value\n        if o == AROM_ORDER:\n            narom[a] += 1\n            narom[b] += 1\n        else:\n            nother[a] += 1\n            nother[b] += 1\n            if value >= 2:\n                exomult[a] = True\n                exomult[b] = True\n    typed = []\n    try:\n        for idx, a in enumerate(atoms):\n            el = a["element"]\n            if a["aromatic"]:\n                h, val = _aromatic_type(\n                    el, a["hydrogens"] if a["bracket"] else None,\n                    narom[idx], nother[idx], vsum[idx],\n                    VALENCE.get(el, 0), exomult[idx])\n            elif a["bracket"]:\n                h = a["hydrogens"]\n                val = a["valence"]\n                if vsum[idx] + h != val:\n                    raise SmilesUnsupported("bracket_valence")\n            else:\n                val = a["valence"]\n                h = val - vsum[idx]\n                if h < 0:\n                    raise SmilesUnsupported("hypervalent")\n            typed.append((el, h, val))\n    except SmilesUnsupported as exc:\n        return None, exc.reason\n    edges = tuple((a, b, o) for a, b, o in bonds)\n    return tuple(typed), edges\n\n\ndef smiles_to_typed(smi):\n    """Convert SMILES to (atom_types, edges) or return (None, reason)."""\n    try:\n        atoms, bonds = parse_smiles(smi)\n    except SmilesUnsupported as exc:\n        return None, exc.reason\n    except (ValueError, IndexError) as exc:\n        return None, f"syntax_error:{exc}"\n    return typed_from_parsed(atoms, bonds)\n\n\ndef parse_formula(text):\n    """Parse a Hill formula string (e.g. C16H17NO4) into a counts dict."""\n    import re\n\n    counts = {e: 0 for e in DOMAIN_ELEMENTS}\n    for el, num in re.findall(r"([A-Z][a-z]?)(\\d*)", text):\n        if el == "H":\n            counts["H"] += int(num) if num else 1\n        elif el in counts:\n            counts[el] += int(num) if num else 1\n        else:\n            counts[el] = counts.get(el, 0) + (int(num) if num else 1)\n    return counts\n\n\ndef standardize_smiles(smi, source, mid, expected_formula=None):\n    """Parse + standardize one SMILES. Returns (record, exclusion_reason)."""\n    typed, result = smiles_to_typed(smi)\n    if typed is None:\n        return None, result\n    edges = result\n    raw = {"id": mid, "source": source, "family": "msgym",\n           "atom_types": typed, "edges": edges, "charge": 0,\n           "original": {"smiles": smi}}\n    record, reason = standardize_record(raw)\n    if record is None:\n        return None, reason\n    if expected_formula is not None:\n        if formula_key(record["formula"]) != formula_key(expected_formula):\n            return None, "formula_mismatch"\n    return record, None\n\n\n# --------------------------------------------------------------------------\n# Incremental candidate-JSON scanner (capped prefix -> complete entries).\n# --------------------------------------------------------------------------\n\ndef iter_json_entries(path, max_entries):\n    """Yield (query_smiles, candidates) for complete entries in file order.\n\n    The file is a single large JSON object mapping query SMILES to arrays\n    of candidate SMILES. Only complete entries are returned; a trailing\n    partial entry from a Range-request prefix is dropped.\n    """\n    with open(path, "r", encoding="utf-8") as handle:\n        text = handle.read()\n    entries = []\n    i, n = 0, len(text)\n    try:\n        while len(entries) < max_entries and i < n:\n            # Find next string token (object key).\n            while i < n and text[i] != \'"\':\n                if text[i] == "}":\n                    return iter(entries)\n                i += 1\n            if i >= n:\n                break\n            key, i = _scan_string(text, i)\n            while i < n and text[i] in " \\t\\r\\n":\n                i += 1\n            if i >= n or text[i] != ":":\n                break\n            i += 1\n            while i < n and text[i] in " \\t\\r\\n":\n                i += 1\n            if i >= n or text[i] != "[":\n                break\n            value_text, i = _scan_array(text, i)\n            try:\n                candidates = json.loads(value_text)\n                query = json.loads(key)\n            except json.JSONDecodeError:\n                continue\n            entries.append((query, candidates))\n    except SmilesUnsupported:\n        pass  # truncated prefix (key or array): keep complete entries only\n    return iter(entries)\n\n\ndef _scan_string(text, i):\n    assert text[i] == \'"\'\n    j = i + 1\n    n = len(text)\n    while j < n:\n        c = text[j]\n        if c == "\\\\":\n            j += 2\n            continue\n        if c == \'"\':\n            return text[i:j + 1], j + 1\n        j += 1\n    raise SmilesUnsupported("truncated_string")\n\n\ndef _scan_array(text, i):\n    assert text[i] == "["\n    depth = 0\n    j = i\n    n = len(text)\n    in_str = False\n    while j < n:\n        c = text[j]\n        if in_str:\n            if c == "\\\\":\n                j += 2\n                continue\n            if c == \'"\':\n                in_str = False\n        else:\n            if c == \'"\':\n                in_str = True\n            elif c == "[":\n                depth += 1\n            elif c == "]":\n                depth -= 1\n                if depth == 0:\n                    return text[i:j + 1], j + 1\n        j += 1\n    raise SmilesUnsupported("truncated_array")\n\n\n# --------------------------------------------------------------------------\n# TSV loading (val fold, capped, deduplicated by structure string).\n# --------------------------------------------------------------------------\n\nWANTED_COLUMNS = ("identifier", "smiles", "inchikey", "formula",\n                  "precursor_formula", "parent_mass", "precursor_mz",\n                  "adduct", "instrument_type", "collision_energy", "fold")\n\n\ndef load_tsv_rows(path):\n    with open(path, "r", encoding="utf-8") as handle:\n        reader = csv.DictReader(handle, delimiter="\\t")\n        missing = [c for c in WANTED_COLUMNS if c not in (reader.fieldnames or [])]\n        if missing:\n            raise ValueError(f"TSV missing columns: {missing}")\n        for row in reader:\n            yield {c: row[c] for c in WANTED_COLUMNS}\n\n\ndef index_val_by_smiles(path):\n    """Map smiles -> list of val-fold rows (dedup key: raw SMILES string)."""\n    groups = {}\n    n_rows = 0\n    for row in load_tsv_rows(path):\n        n_rows += 1\n        if row["fold"] != "val":\n            continue\n        groups.setdefault(row["smiles"], []).append(row)\n    return groups, n_rows\n\n\n# --------------------------------------------------------------------------\n# Query construction from a true structure (synthetic open subgraphs).\n# --------------------------------------------------------------------------\n\ndef extract_patterns(atom_types, edges):\n    """Single-atom, single-bond, and first overlapping bond-pair patterns."""\n    if not edges:\n        return ((atom_types[0],), ()), None\n    a, b, o = edges[0]\n    single = (((atom_types[a], atom_types[b])), ((0, 1, o),))\n    pair = None\n    for c, d, p in edges[1:]:\n        if len({a, b, c, d}) == 3:\n            at = [atom_types[a], atom_types[b]]\n            third = c if c not in (a, b) else d\n            at.append(atom_types[third])\n            # Re-index: keep shared atom first for a compact pattern.\n            shared = a if a in (c, d) else b\n            others = [x for x in (a, b) if x != shared] + [third]\n            order = {tuple(sorted((x, y))): q for x, y, q in edges}\n            mapping = {shared: 0, others[0]: 1, others[1]: 2}\n            pe = tuple(sorted(\n                (mapping[x], mapping[y], q)\n                for (x, y), q in order.items()\n                if x in (shared,) + tuple(others) and y in (shared,) + tuple(others)\n                and {x, y} <= {shared, others[0], others[1]}\n                and ((x == shared and y in others) or (y == shared and x in others))))\n            pair = ((tuple([atom_types[shared]] + [atom_types[x] for x in others])), pe)\n            break\n    return single, pair\n\n\ndef run_msgym_experiment(prefix_path, tsv_path, max_queries, candidate_cap=None):\n    """Capped val-fold run over supplied formula candidate pools."""\n    t0 = time.perf_counter()\n    c0 = time.process_time()\n    val_groups, n_tsv_rows = index_val_by_smiles(tsv_path)\n    entries = [(q, c) for q, c in iter_json_entries(prefix_path, max_queries * 20)]\n    # File-order slice: first entries that join to a val row.\n    joined = [(q, c) for q, c in entries if q in val_groups][:max_queries]\n    pool_stats = Counter()\n    target_stats = Counter()\n    queries = []  # (name, target_record, patterns_list, row)\n    for qi, (qsmiles, cands) in enumerate(joined):\n        row = val_groups[qsmiles][0]\n        try:\n            expected = parse_formula(row["formula"])\n        except Exception:\n            target_stats["formula_unparseable"] += 1\n            continue\n        target, reason = standardize_smiles(\n            qsmiles, "msgym-val", f"MSGYM-Q{qi:04d}", expected)\n        if target is None:\n            target_stats[reason.split(":")[0]] += 1\n            continue\n        target_stats["in_domain"] += 1\n        # Mass agreement: graph-derived theoretical neutral mass vs the\n        # provider\'s parent_mass, which is MEASURED (precursor m/z minus\n        # proton mass; sub-ppm Orbitrap errors observed). Agreement within\n        # 10 ppm validates the parser\'s H-count logic against an independent\n        # column; the separate formula cross-check already passed.\n        try:\n            from decimal import Decimal\n\n            observed_mu = int(Decimal(row["parent_mass"]) * 1_000_000)\n            ppm = abs(observed_mu - target["mass"]) / target["mass"] * 1e6\n        except Exception:\n            ppm = None\n        mass_ok = ppm is not None and ppm <= 10.0\n        pool_stats["mass_agree" if mass_ok else "mass_mismatch"] += 1\n        pool = []\n        capped = cands if candidate_cap is None else cands[:candidate_cap]\n        for ci, csmiles in enumerate(capped):\n            pool_stats["n_supplied"] += 1\n            rec, rreason = standardize_smiles(\n                csmiles, "msgym-pool", f"MSGYM-Q{qi:04d}-C{ci:04d}")\n            if rec is None:\n                pool_stats[f"pool_excl:{rreason.split(\':\')[0]}"] += 1\n                continue\n            pool_stats["pool_in_domain"] += 1\n            pool.append(rec)\n        single, pair = extract_patterns(target["atom_types"], target["edges"])\n        pats_mass_only = []\n        pats_single = [single] if single else []\n        pats_two = [single, pair] if pair else pats_single\n        queries.append((f"MSGYM-Q{qi:04d}", target, pool, row,\n                        pats_mass_only, pats_single, pats_two))\n    rows = []\n    import sys as _sys\n\n    for qi, (name, target, pool, row, p0, p1, p2) in enumerate(queries):\n        # Supplied pools are used verbatim; the target is never injected.\n        sub = DatabaseIndex(pool)\n        qsmiles = row["smiles"]\n        # Pool membership by exact (RDKit-canonical) SMILES string. This is\n        # exact up to the provider\'s canonicalization; the fingerprint-based\n        # flag below is a weaker upper bound (collisions possible).\n        strict_in_pool = any(\n            r["original"].get("smiles") == qsmiles for r in pool)\n        loose_in_pool = strict_in_pool or any(\n            formula_key(r["formula"]) == formula_key(target["formula"])\n            and edge_fingerprint(r["atom_types"], r["edges"])\n            == edge_fingerprint(target["atom_types"], target["edges"])\n            for r in pool)\n        try:\n            from decimal import Decimal\n\n            obs_mu = int(Decimal(row["parent_mass"]) * 1_000_000)\n            ppm_err = abs(obs_mu - target["mass"]) / target["mass"] * 1e6\n        except Exception:\n            ppm_err = None\n        for qname, pats in (("mass_only", p0), ("single_bond", p1),\n                            ("bonded_pair", p2)):\n            # Bonded patterns only: single-atom patterns are combinatorially\n            # promiscuous on drug-sized targets; their behavior is covered by\n            # the synthetic harness. 100k embedding nodes per arm, then an\n            # explicit search_budget_exhausted status (never a silent cut).\n            counters = WorkCounters(limit=100_000)\n            res = run_query(sub, target, pats, "unknown", None,\n                            counters=counters)\n            present = {stage: any(\n                r["original"].get("smiles") == qsmiles for r in res[stage])\n                for stage in ("s1", "s2", "s3")}\n            # Rank the surviving pool; the target-equivalent member (by\n            # string) is the top-k reference. Absent target => keys gated.\n            tgt_rid = next(\n                (r["id"] for r in res["s3"]\n                 if r["original"].get("smiles") == qsmiles), "__absent__")\n            rank, _ = rank_baselines(res["s3"], tgt_rid, sub)\n            rows.append({\n                "query": f"{name}:{qname}", "target": target["id"],\n                "in_supplied_pool": strict_in_pool,\n                "in_pool_loose_fp": loose_in_pool,\n                "recall_s1": present["s1"], "recall_s2": present["s2"],\n                "recall_s3": present["s3"],\n                "n_supplied": len(pool),\n                "n_s1": len(res["s1"]), "n_s2": len(res["s2"]),\n                "n_s3": len(res["s3"]),\n                "zero_status": res["zero_status"],\n                "top1_uniform": rank.get("uniform@1"),\n                "top1_fingerprint": rank.get("fingerprint@1"),\n                "top3_fingerprint": rank.get("fingerprint@3"),\n                "heavy": target["heavy"],\n                "n_pool": len(pool),\n                "adduct": row["adduct"],\n                "instrument": row["instrument_type"],\n                "ppm_err": round(ppm_err, 3) if ppm_err is not None else None,\n                "truncated": res["truncated"],\n                "embedding_nodes": counters.embedding_nodes,\n            })\n        if (qi + 1) % 5 == 0 or qi + 1 == len(queries):\n            _sys.stderr.write(f"# progress {qi + 1}/{len(queries)}\\n")\n            _sys.stderr.flush()\n    wall = time.perf_counter() - t0\n    cpu = time.process_time() - c0\n    return {\n        "rows": rows,\n        "n_tsv_rows": n_tsv_rows,\n        "n_val_structures": len(val_groups),\n        "n_entries_scanned": len(entries),\n        "n_queries": len(queries),\n        "pool_stats": dict(pool_stats),\n        "target_stats": dict(target_stats),\n        "wall_s": wall,\n        "cpu_s": cpu,\n    }\n\n\ndef main(argv):\n    import argparse\n\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--prefix", required=True)\n    ap.add_argument("--tsv", required=True)\n    ap.add_argument("--max-queries", type=int, default=150)\n    ap.add_argument("--out", default="")\n    args = ap.parse_args(argv)\n    result = run_msgym_experiment(args.prefix, args.tsv, args.max_queries)\n    if args.out:\n        with open(args.out, "w", encoding="utf-8") as handle:\n            writer = csv.DictWriter(handle, fieldnames=sorted(result["rows"][0].keys()))\n            writer.writeheader()\n            writer.writerows(result["rows"])\n    summary = {k: v for k, v in result.items() if k != "rows"}\n    sys.stdout.write(json.dumps(summary, indent=2, default=str) + "\\n")\n    for r in result["rows"][:10]:\n        sys.stderr.write(str(r) + "\\n")\n\n\n# --------------------------------------------------------------------------\n# Tests.\n# --------------------------------------------------------------------------\n\nclass SmilesTests(unittest.TestCase):\n    def test_ethanol(self):\n        typed, edges = smiles_to_typed("CCO")\n        f = formula_of(typed)\n        self.assertEqual((f["C"], f["H"], f["O"]), (2, 6, 1))\n        rec, reason = standardize_record({"id": "t", "atom_types": typed,\n                                          "edges": edges, "charge": 0})\n        self.assertIsNotNone(rec, reason)\n\n    def test_branch_and_double_bond(self):\n        typed, _ = smiles_to_typed("CC(=O)O")\n        self.assertEqual(formula_of(typed)["H"], 4)  # acetic acid C2H4O2\n\n    def test_ring_closure(self):\n        typed, edges = smiles_to_typed("C1CCOC1")  # tetrahydrofuran\n        self.assertEqual(len(typed), 5)\n        self.assertEqual(len(edges), 5)\n\n    def test_percent_ring(self):\n        typed, _ = smiles_to_typed("C%10CCCCC%10")  # cyclohexane via %10\n        self.assertEqual(len(typed), 6)\n\n    def test_benzene_aromatic(self):\n        typed, edges = smiles_to_typed("c1ccccc1")\n        self.assertEqual(formula_of(typed)["H"], 6)  # C6H6\n        self.assertTrue(all(o == 4 for _, _, o in edges))\n        rec, reason = standardize_record({"id": "t", "atom_types": typed,\n                                          "edges": edges, "charge": 0})\n        self.assertIsNotNone(rec, reason)\n\n    def test_pyridine_pyrrole_thiophene(self):\n        for smi, h, n in (("n1ccccc1", 5, 1), ("c1cc[nH]c1", 5, 1),\n                          ("c1ccsc1", 4, 0)):\n            typed, _ = smiles_to_typed(smi)\n            f = formula_of(typed)\n            self.assertEqual((f["H"], f["N"]), (h, n), smi)\n\n    def test_aromatic_chloride(self):\n        typed, _ = smiles_to_typed("Clc1ccccc1")\n        self.assertEqual(formula_of(typed),\n                         {"C": 6, "H": 5, "N": 0, "O": 0, "F": 0,\n                          "S": 0, "Cl": 1, "Br": 0, "I": 0})\n\n    def test_sulfone_hypervalent(self):\n        _, reason = smiles_to_typed("CCS(=O)(=O)C")\n        self.assertEqual(reason, "hypervalent")\n\n    def test_bare_n_ambiguous(self):\n        _, reason = smiles_to_typed("nC")\n        self.assertEqual(reason, "ambiguous_aromatic")\n\n    def test_pyrone_carbonyl_fallback(self):\n        # RDKit flags pyrone carbonyl carbons aromatic; with an exocyclic\n        # double bond they take aliphatic typing (uniform residual rule).\n        rec, reason = standardize_smiles(\n            "O=c1oc2ccccc2c(O)c1C", "t", "T", parse_formula("C10H8O3"))\n        self.assertIsNotNone(rec, reason)\n        self.assertEqual(rec["formula"]["H"], 8)\n\n    def test_aromatic_kekule_boundary(self):\n        # Representation boundary (documented): aromatic-order patterns do\n        # not match kekule-ordered targets and vice versa, even for the\n        # same molecule. Within one canonical corpus this never triggers.\n        from tools.ms2_database_retrieval import contains_exact\n\n        kt, ke = smiles_to_typed("C1=CC=CC=C1")\n        at, ae = smiles_to_typed("c1ccccc1")\n        apat = ((at[0], at[1]), ((0, 1, ae[0][2]),))\n        self.assertFalse(contains_exact(kt, ke, *apat))\n        self.assertTrue(contains_exact(at, ae, *apat))\n\n    def test_charge_excluded(self):\n        _, reason = smiles_to_typed("CC[N+](=O)[O-]")\n        self.assertEqual(reason, "unsupported_charge")\n\n    def test_halogen_alkyl_now_supported(self):\n        # Monovalent halogens joined the domain; silicon stays out.\n        typed, _ = smiles_to_typed("CCCl")\n        self.assertEqual(formula_of(typed)["Cl"], 1)\n        _, reason = smiles_to_typed("CC[Si]")\n        self.assertEqual(reason, "unsupported_element:Si")\n\n    def test_disconnected_excluded(self):\n        _, reason = smiles_to_typed("CC.[Na+]")\n        self.assertEqual(reason, "disconnected_or_salt")\n\n    def test_bracket_stereo_ok(self):\n        typed, _ = smiles_to_typed("C[C@H](N)O")\n        self.assertEqual(formula_of(typed)["H"], 7)  # C2H7NO\n\n    def test_kekule_benzene_accepted_as_domain(self):\n        typed, edges = smiles_to_typed("C1=CC=CC=C1")\n        rec, reason = standardize_record({"id": "t", "atom_types": typed,\n                                          "edges": edges, "charge": 0})\n        self.assertIsNotNone(rec, reason)\n\n    def test_formula_parser(self):\n        self.assertEqual(parse_formula("C16H17NO4")["C"], 16)\n        self.assertEqual(parse_formula("C16H17NO4")["H"], 17)\n        self.assertEqual(parse_formula("C16H17NO4")["N"], 1)\n\n    def test_formula_crosscheck(self):\n        typed, _ = smiles_to_typed("CCO")\n        rec, reason = standardize_smiles("CCO", "t", "t", parse_formula("C2H6O"))\n        self.assertIsNotNone(rec, reason)\n        rec2, reason2 = standardize_smiles("CCO", "t", "t", parse_formula("C2H6O2"))\n        self.assertIsNone(rec2)\n        self.assertEqual(reason2, "formula_mismatch")\n\n    def test_extract_patterns_roundtrip(self):\n        from tools.ms2_database_retrieval import joint_contains, WorkCounters\n\n        typed, edges = smiles_to_typed("CC(=O)O")\n        single, pair = extract_patterns(typed, edges)\n        self.assertTrue(joint_contains(typed, edges, [single], WorkCounters()))\n        if pair is not None:\n            self.assertTrue(\n                joint_contains(typed, edges, [single, pair], WorkCounters()))\n        import tempfile, os\n\n        doc = \'{"CCO": ["CCO", "COC"], "CCC": ["CCC"]}\'\n        with tempfile.NamedTemporaryFile("w", suffix=".json", delete=False) as fh:\n            fh.write(doc)\n            path = fh.name\n        try:\n            out = list(iter_json_entries(path, 10))\n            self.assertEqual(out[0][0], "CCO")\n            self.assertEqual(out[0][1], ["CCO", "COC"])\n            self.assertEqual(out[1][0], "CCC")\n        finally:\n            os.unlink(path)\n\n    def test_json_scanner_drops_partial(self):\n        import tempfile, os\n\n        doc = \'{"CCO": ["CCO", "COC"], "CCC": ["CC\'\n        with tempfile.NamedTemporaryFile("w", suffix=".json", delete=False) as fh:\n            fh.write(doc)\n            path = fh.name\n        try:\n            out = list(iter_json_entries(path, 10))\n            self.assertEqual(len(out), 1)\n            self.assertEqual(out[0][0], "CCO")\n        finally:\n            os.unlink(path)\n\n    def test_json_scanner_drops_truncated_key(self):\n        # Review regression: a prefix ending mid-key crashed instead of\n        # returning the complete entries before it.\n        import tempfile, os\n\n        doc = \'{"CCO": ["CCO"], "C\'\n        with tempfile.NamedTemporaryFile("w", suffix=".json", delete=False) as fh:\n            fh.write(doc)\n            path = fh.name\n        try:\n            out = list(iter_json_entries(path, 10))\n            self.assertEqual(len(out), 1)\n            self.assertEqual(out[0][0], "CCO")\n        finally:\n            os.unlink(path)\n\n    def test_isotope_bracket_rejected(self):\n        # Review regression: [13C] silently normalized to 12C with the\n        # unlabeled mass. Isotope labels must be rejected, not dropped.\n        rec, reason = standardize_smiles("[13CH4]", "t", "T")\n        self.assertIsNone(rec)\n        self.assertEqual(reason, "unsupported_isotope")\n\n\nif __name__ == "__main__":\n    main(sys.argv[1:])\n', 'tools/ms2_database_retrieval.py': '"""Database-first molecular-completion retrieval harness (Experiments A/B).\n\nImplements docs/MOLECULAR_COMPLETION_DATABASE_EXPERIMENTS.md, first pass only:\noffline, stdlib-only, capped synthetic fixture corpus. No external dataset is\ndownloaded by the default run. Ingestion adapters for ChEBI / MassSpecGym /\nMassBank / GNPS / PubChem / QM9 are stubs that pin releases, record hashes and\nenforce license/throttle notes; they raise an informative error until the\noperator supplies a pinned local file.\n\nRun:   python3 tools/ms2_database_retrieval.py\nTest:  python3 -m unittest tools/ms2_database_retrieval.py\n\nConventions (from MOLECULAR_COMPLETION_DESIGN.md):\n- Parent-relative hydrogen counts: substructure atom types are\n  (element, parent H count, valence); never re-sanitized as capped molecules.\n- Three-valued mass verdict: accept / reject / boundary-ambiguous, plus\n  unavailable when precision is unknown. Integer arithmetic only.\n- Fingerprint is a SOUND rejection screen only; surviving hits always go\n  through exact typed graph matching.\n- Unknown-overlap queries never receive the known-correspondence oracle.\n- Zero candidates with target absent from the pool is reported as\n  `target_absent`, never as proof of constraint inconsistency.\n- Top-k is reported only when the target is present in the candidate pool.\n- Synthetic parent-subgraph queries are controlled completion tests, not\n  experimentally confirmed fragment structures.\n"""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport io\nimport sys\nimport time\nimport unittest\nfrom collections import Counter\nfrom decimal import Decimal, ROUND_CEILING, ROUND_HALF_EVEN\nfrom itertools import combinations\n\n# --------------------------------------------------------------------------\n# Integer mass arithmetic (same constants/semantics as the pilot reference).\n# --------------------------------------------------------------------------\n\nEXACT = {\n    "C": "12",\n    "H": "1.00782503223",\n    "N": "14.00307400443",\n    "O": "15.99491461957",\n    "F": "18.99840316273",\n    "S": "31.9720711744",\n    "Cl": "34.968852682",\n    "Br": "78.9183376",\n    "I": "126.9044719",\n}\nSCALE = 1_000_000\nMASS = {e: int((Decimal(m) * SCALE).to_integral_value(rounding=ROUND_HALF_EVEN))\n        for e, m in EXACT.items()}\nERROR_NDA = {\n    e: int((abs(Decimal(m) * SCALE - MASS[e]) * 1000).to_integral_value(rounding=ROUND_CEILING))\n    for e, m in EXACT.items()\n}\n\n# V0 neutral closed-shell vocabulary used by this harness, plus the\n# max-hydride neutrals (methane/ammonia/water atom types), hydrogen\n# fluoride\'s F, monovalent halogens, divalent sulfur, and aromatic atom\n# types for RDKit-canonical SMILES (lowercase). Aromatic bonds carry order\n# code AROM_ORDER (4); ORDER_VALUE maps codes to valence sums, so one\n# uniform residual rule covers aliphatic and aromatic atoms. Bare aromatic\n# `n` with two aromatic-only connections follows the RDKit convention\n# (pyridine, H=0); pyrrolic NH must be written [nH]. This convention is\n# sound for RDKit-canonical corpora (e.g. MassSpecGym 1.5) and is recorded\n# as a corpus dependence, not a general SMILES rule.\n# (element, parent-relative H count, total valence)\nAROM_ORDER = 4\nORDER_VALUE = {1: 1, 2: 2, 3: 3, 4: 1}\nTYPES = (\n    ("C", 0, 4), ("C", 1, 4), ("C", 2, 4), ("C", 3, 4), ("C", 4, 4),\n    ("N", 0, 3), ("N", 1, 3), ("N", 2, 3), ("N", 3, 3),\n    ("O", 0, 2), ("O", 1, 2), ("O", 2, 2),\n    ("F", 0, 1), ("F", 1, 1),\n    ("Cl", 0, 1), ("Br", 0, 1), ("I", 0, 1),\n    ("Cl", 1, 1), ("Br", 1, 1), ("I", 1, 1),\n    ("S", 0, 2), ("S", 1, 2), ("S", 2, 2),\n    ("C", 0, 3), ("C", 1, 3), ("C", 2, 3),\n    ("N", 0, 2),\n)\nTYPE_SET = set(TYPES)\nC3 = ("C", 3, 4)\nC2 = ("C", 2, 4)\nC1 = ("C", 1, 4)\nN2 = ("N", 2, 3)\nN1 = ("N", 1, 3)\nO1 = ("O", 1, 2)\nO0 = ("O", 0, 2)\n\n\ndef formula_of(atom_types):\n    result = {e: 0 for e in DOMAIN_ELEMENTS}\n    for element, hydrogen, _ in atom_types:\n        result[element] += 1\n        result["H"] += hydrogen\n    return result\n\n\ndef neutral_mass(formula):\n    return sum(formula.get(e, 0) * MASS[e] for e in EXACT)\n\n\ndef mass_verdict(observed, formula, uncertainty=50, ppm_tenths=100):\n    """Three-valued neutral-mass decision. See design section on formulas."""\n    if uncertainty is None:\n        return "unavailable"\n    computed = neutral_mass(formula)\n    error = (sum(formula.get(e, 0) * ERROR_NDA[e] for e in EXACT) + 999) // 1000\n    error += uncertainty\n    tolerance = observed * ppm_tenths // 10_000_000\n    residual = abs(observed - computed)\n    if residual + error <= tolerance:\n        return "accept"\n    if residual > tolerance + error:\n        return "reject"\n    return "ambiguous"\n\n\n# --------------------------------------------------------------------------\n# Records, standardization, identity.\n# --------------------------------------------------------------------------\n\nDOMAIN_ELEMENTS = ("C", "H", "N", "O", "F", "S", "Cl", "Br", "I")\n\n\ndef standardize_record(raw):\n    """Validate a raw structure dict into a closed neutral record.\n\n    Returns (record, exclusion_reason). exclusion_reason is None on success.\n    Record keeps source id and original payload for provenance; identity is\n    connectivity-only (stereo stripped, tautomer not resolved -- recorded as\n    a limitation, not silently normalized).\n    Rejects: unknown atom types, nonzero charge, isotopic labels, salts or\n    disconnected graphs, open valences (DB stage is closed molecules only).\n    """\n    atom_types = tuple(raw.get("atom_types", ()))\n    edges = tuple(raw.get("edges", ()))\n    for t in atom_types:\n        if t not in TYPE_SET:\n            return None, f"unsupported_atom_type:{t}"\n    if raw.get("charge", 0) != 0:\n        return None, "unsupported_charge"\n    if raw.get("isotopes"):\n        return None, "unsupported_isotope"\n    n = len(atom_types)\n    if n == 0:\n        return None, "empty_graph"\n    # Connectedness (single component => no salt fragments).\n    seen = {0}\n    changed = True\n    emap = {}\n    for a, b, o in edges:\n        if o:\n            emap[tuple(sorted((a, b)))] = o\n    while changed:\n        changed = False\n        for a, b, o in edges:\n            if not o:\n                continue\n            if (a in seen) != (b in seen):\n                seen.update((a, b))\n                changed = True\n    if len(seen) != n:\n        return None, "disconnected_or_salt"\n    # Closed-shell: residual valence must be zero for every atom, using\n    # ORDER_VALUE so aromatic bonds (code 4) count one valence unit.\n    residual = [v - h for _, h, v in atom_types]\n    for a, b, o in edges:\n        try:\n            value = ORDER_VALUE[o]\n        except KeyError:\n            return None, f"unsupported_bond_type:{o}"\n        residual[a] -= value\n        residual[b] -= value\n    if any(r != 0 for r in residual):\n        return None, "open_valence"\n    formula = formula_of(atom_types)\n    record = {\n        "id": raw.get("id"),\n        "source": raw.get("source"),\n        "family": raw.get("family", "unspecified"),\n        "atom_types": atom_types,\n        "edges": edges,\n        "formula": formula,\n        "mass": neutral_mass(formula),\n        "charge": 0,\n        "heavy": sum(1 for e, _, _ in atom_types if e != "H"),\n        "original": dict(raw.get("original", {})),\n    }\n    return record, None\n\n\ndef canonical_key(atom_types, edges):\n    """Connectivity identity (no stereo). For fixture-scale graphs only."""\n    from itertools import permutations as _perm\n\n    n = len(atom_types)\n    edge_map = {tuple(sorted((a, b))): o for a, b, o in edges if o}\n    best = None\n    for order in _perm(range(n)):\n        labels = tuple(atom_types[i] for i in order)\n        bonds = tuple(edge_map.get(tuple(sorted((order[i], order[j]))), 0)\n                      for i, j in combinations(range(n), 2))\n        key = (labels, bonds)\n        if best is None or key < best:\n            best = key\n    return best\n\n\ndef skeleton_key(atom_types, edges):\n    """Coarse scaffold grouping: heavy-element sequence + bond multiset."""\n    heavy = tuple(sorted(e for e, _, _ in atom_types))\n    orders = tuple(sorted(o for _, _, o in edges if o))\n    return (heavy, orders)\n\n\n# --------------------------------------------------------------------------\n# Exact typed matching + sound fingerprint screen.\n# --------------------------------------------------------------------------\n\nclass WorkCounters:\n    def __init__(self, limit=None):\n        self.fingerprint_screens = 0\n        self.exact_matches = 0\n        self.embedding_nodes = 0\n        self.limit = limit\n\n    def tick(self):\n        self.embedding_nodes += 1\n        if self.limit is not None and self.embedding_nodes > self.limit:\n            raise BudgetExhausted("embedding_nodes")\n\n\nclass BudgetExhausted(Exception):\n    pass\n\n\ndef edge_fingerprint(atom_types, edges):\n    """Necessary-condition counters for typed containment.\n\n    fp = (atom-type Counter, unordered typed-edge Counter). If any pattern\n    counter exceeds the candidate\'s, no injective typed edge-preserving\n    embedding exists -> sound to reject. Never used to accept.\n    """\n    ac = Counter(atom_types)\n    ec = Counter()\n    for a, b, o in edges:\n        if not o:\n            continue\n        t = tuple(sorted((atom_types[a], atom_types[b])))\n        ec[(t[0], t[1], o)] += 1\n    return ac, ec\n\n\ndef fingerprint_screen(candidate_fp, pattern_fp):\n    cac, cec = candidate_fp\n    pac, pec = pattern_fp\n    for k, v in pac.items():\n        if cac.get(k, 0) < v:\n            return False\n    for k, v in pec.items():\n        if cec.get(k, 0) < v:\n            return False\n    return True\n\n\ndef pattern_fingerprint(pattern_types, pattern_edges):\n    return edge_fingerprint(pattern_types, [(a, b, o) for a, b, o in pattern_edges])\n\n\ndef prepare_target(atom_types, edges):\n    """Precompute once per candidate: neighbor maps and type positions.\n\n    Returns (nbr, pos_by_type). nbr[i] maps neighbor -> bond order;\n    pos_by_type[t] lists target atoms of exact type t. Both matchers iterate\n    only type-compatible atoms and check constraints via dict lookup\n    (no per-check tuple allocation or sorting).\n    """\n    nbr = [{} for _ in atom_types]\n    for a, b, o in edges:\n        if o:\n            nbr[a][b] = o\n            nbr[b][a] = o\n    pos = {}\n    for i, t in enumerate(atom_types):\n        pos.setdefault(t, []).append(i)\n    return nbr, pos\n\n\ndef _prepare_single(pattern_types, pattern_edges):\n    """Pattern-side prep, reusable across candidates.\n\n    Returns (order, cons) where order lists pattern atoms rarest-... (static\n    degree order; target rarity is applied at iteration via pos_by_type\n    candidate-list lengths) and cons[k] lists (earlier_position, order).\n    """\n    m = len(pattern_types)\n    deg = [0] * m\n    adj = [[] for _ in range(m)]\n    for a, b, o in pattern_edges:\n        deg[a] += 1\n        deg[b] += 1\n        adj[a].append((b, o))\n        adj[b].append((a, o))\n    order = sorted(range(m), key=lambda i: -deg[i])\n    rank = {atom: k for k, atom in enumerate(order)}\n    cons = []\n    for k, atom in enumerate(order):\n        cons.append(tuple(sorted(\n            (rank[nb], o) for nb, o in adj[atom] if rank[nb] < k)))\n    return order, cons\n\n\ndef count_embeddings_prep(nbr, pos, ptypes, order, cons, cap, counters=None):\n    """Count distinct embeddings of one pattern up to cap (feature use).\n\n    Same existence semantics as _contains_with_prep (result > 0 iff a\n    match exists); the count itself is a cheap multiplicity feature, not\n    a quality claim. Occupancy handling mirrors the matcher exactly.\n    """\n    total = 0\n    stop = False\n    n_target = len(nbr)\n    chosen = [0] * len(order)\n    occupied = [False] * n_target\n\n    def search(k):\n        nonlocal total, stop\n        if stop:\n            return\n        if k == len(order):\n            # No bulk reset here: each frame unsets its own atom after the\n            # recursive call returns, so ancestor assignments stay intact\n            # for the remaining branches (injectivity preserved while\n            # counting). Resetting `chosen` at the leaf would let later\n            # branches reuse ancestor atoms and inflate the count.\n            total += 1\n            if total >= cap:\n                stop = True\n            return\n        atom = order[k]\n        want = ptypes[atom]\n        for cand in pos.get(want, ()):\n            if stop:\n                return\n            if counters is not None:\n                counters.tick()\n            if occupied[cand]:\n                continue\n            row = nbr[cand]\n            ok = True\n            for l, o in cons[k]:\n                if row.get(chosen[l]) != o:\n                    ok = False\n                    break\n            if ok:\n                chosen[k] = cand\n                occupied[cand] = True\n                search(k + 1)\n                occupied[cand] = False\n\n    search(0)\n    return total\n\n\ndef _contains_with_prep(nbr, pos, ptypes, order, cons, occupied, counters):\n    # Occupancy is always clean on return: backtracking unsets on failure,\n    # and success resets the (small) chosen set before unwinding.\n    chosen = [0] * len(order)\n\n    def search(k):\n        if k == len(order):\n            for c in chosen:\n                occupied[c] = False\n            return True\n        atom = order[k]\n        want = ptypes[atom]\n        for cand in pos.get(want, ()):\n            if counters is not None:\n                counters.tick()\n            if occupied[cand]:\n                continue\n            row = nbr[cand]\n            ok = True\n            for l, o in cons[k]:\n                if row.get(chosen[l]) != o:\n                    ok = False\n                    break\n            if ok:\n                chosen[k] = cand\n                occupied[cand] = True\n                if search(k + 1):\n                    return True\n                occupied[cand] = False\n        return False\n\n    return search(0)\n\n\ndef contains_exact(atom_types, edges, pattern_types, pattern_edges, counters=None):\n    """Injective typed edge-preserving embedding; target may add edges."""\n    if len(pattern_types) > len(atom_types):\n        return False\n    nbr, pos = prepare_target(atom_types, edges)\n    order, cons = _prepare_single(pattern_types, pattern_edges)\n    return _contains_with_prep(nbr, pos, pattern_types, order, cons,\n                               [False] * len(atom_types), counters)\n\n\ndef _prepare_joint(patterns):\n    """Pattern-side prep for joint matching, reusable across candidates.\n\n    Returns (flat_types, flat_pat, flat_adj, degrees). flat_adj[i] lists\n    (j, order) pattern-edge neighbors of flat atom i.\n    """\n    flat_types, flat_pat = [], []\n    index_of = {}\n    for pi, (pt, pe) in enumerate(patterns):\n        for ai, t in enumerate(pt):\n            index_of[(pi, ai)] = len(flat_types)\n            flat_types.append(t)\n            flat_pat.append(pi)\n    flat_adj = [[] for _ in flat_types]\n    for pi, (pt, pe) in enumerate(patterns):\n        for a, b, o in pe:\n            i, j = index_of[(pi, a)], index_of[(pi, b)]\n            flat_adj[i].append((j, o))\n            flat_adj[j].append((i, o))\n    degrees = [len(v) for v in flat_adj]\n    return flat_types, flat_pat, flat_adj, degrees\n\n\ndef _joint_with_prep(nbr, pos, counts, flat_types, flat_pat, flat_adj,\n                     degrees, n_patterns, occupied_sets, counters):\n    order = sorted(range(len(flat_types)),\n                   key=lambda i: (counts.get(flat_types[i], 0), -degrees[i]))\n    assign = [None] * len(flat_types)\n\n    def search(k):\n        if k == len(order):\n            return True\n        fi = order[k]\n        want = flat_types[fi]\n        used = occupied_sets[flat_pat[fi]]\n        for cand in pos.get(want, ()):\n            if counters is not None:\n                counters.tick()\n            if cand in used:\n                continue\n            row = nbr[cand]\n            ok = True\n            for j, o in flat_adj[fi]:\n                other = assign[j]\n                if other is not None and row.get(other) != o:\n                    ok = False\n                    break\n            if ok:\n                assign[fi] = cand\n                used.add(cand)\n                if search(k + 1):\n                    return True\n                assign[fi] = None\n                used.discard(cand)\n        return False\n\n    return search(0)\n\n\ndef joint_contains(atom_types, edges, patterns, counters=None):\n    """Global overlap consistency across patterns.\n\n    Simultaneously embeds every pattern, injective within each pattern,\n    sharing allowed across patterns. Rejects quotients that collapse two\n    distinct atoms of one pattern or equate incompatible types. Pairwise\n    per-pattern success alone does not imply this returns True.\n    """\n    if not any(True for _ in patterns for _ in _[0]):\n        return True\n    flat_types, flat_pat, flat_adj, degrees = _prepare_joint(patterns)\n    nbr, pos = prepare_target(atom_types, edges)\n    from collections import Counter as _Counter\n\n    occupied_sets = [set() for _ in patterns]\n    return _joint_with_prep(nbr, pos, _Counter(atom_types),\n                            flat_types, flat_pat, flat_adj, degrees,\n                            len(patterns), occupied_sets, counters)\n\n\n# --------------------------------------------------------------------------\n# Fixture corpus (synthetic stand-ins; NOT ChEBI/MassBank records).\n# --------------------------------------------------------------------------\n\ndef _mol(mid, family, atom_types, edges):\n    return {"id": mid, "source": "synthetic-fixture", "family": family,\n            "atom_types": tuple(atom_types), "edges": tuple(edges),\n            "charge": 0, "original": {"note": "synthetic controlled-completion fixture"}}\n\n\ndef fixture_raw_records():\n    C, N, O = C3, N2, O1\n    recs = [\n        _mol("FIX-001", "alcohol", (C3, C2, O1), ((0, 1, 1), (1, 2, 1))),          # ethanol\n        _mol("FIX-002", "ether", (C3, O0, C3), ((0, 1, 1), (1, 2, 1))),            # dimethyl ether\n        _mol("FIX-003", "alcohol", (C3, C2, C2, O1),\n             ((0, 1, 1), (1, 2, 1), (2, 3, 1))),                                   # 1-propanol\n        _mol("FIX-004", "alcohol", (C3, C1, C3, O1),\n             ((0, 1, 1), (1, 2, 1), (1, 3, 1))),                                   # 2-propanol\n        _mol("FIX-005", "ether", (C3, O0, C2, C3), ((0, 1, 1), (1, 2, 1), (2, 3, 1))),\n        _mol("FIX-006", "amine", (C3, N2), ((0, 1, 1),)),                          # methylamine\n        _mol("FIX-007", "amide", (C1, O0, N2), ((0, 1, 2), (0, 2, 1))),            # formamide\n        _mol("FIX-008", "diol", (C2, C2, O1, O1),\n             ((0, 1, 1), (0, 2, 1), (1, 3, 1))),                                   # ethylene glycol\n        _mol("FIX-009", "cyclic-ether", (C2, C2, C2, O0),\n             ((0, 1, 1), (1, 2, 1), (2, 3, 1), (3, 0, 1))),                         # oxetane\n        _mol("FIX-010", "cyclic-amine", (C2, C2, N1),\n             ((0, 1, 1), (1, 2, 1), (2, 0, 1))),                                   # aziridine\n        _mol("FIX-011", "fluoro", (C3, C2, ("F", 0, 1)),\n             ((0, 1, 1), (1, 2, 1))),                                              # 1-fluoroethane\n        _mol("FIX-012", "carbonyl", (C3, C1, O0),\n             ((0, 1, 1), (1, 2, 2))),                                              # acetaldehyde\n    ]\n    return recs\n\n\nCH3_PAT = ((C3,), ())\nCH2OH_PAT = (((C2, O1)), ((0, 1, 1),))\nCH3CH2_PAT = (((C3, C2)), ((0, 1, 1),))\nCN_PAT = (((C3, N2)), ((0, 1, 1),))\nC1N_PAT = (((C1, N2)), ((0, 1, 1),))\nN_PAT = ((N2,), ())\n\n\ndef fixture_queries():\n    """(name, target_id, patterns, overlap, precursor_id).\n\n    overlap: \'na\' (no patterns), \'unknown\' (default condition), \'known\'\n    (oracle correspondence supplied separately, same filter plus provenance).\n    precursor: optional parent record id for stage-4 arm; None => not-evaluated.\n    """\n    return [\n        ("C2H6O_mass_only", "FIX-001", (), "na", None),\n        ("C2H6O_CH3_unknown", "FIX-001", (CH3_PAT,), "unknown", None),\n        ("C2H6O_CH2OH_unknown", "FIX-001", (CH2OH_PAT,), "unknown", None),\n        ("C2H6O_two_overlapping_unknown", "FIX-001", (CH3CH2_PAT, CH2OH_PAT), "unknown", None),\n        ("C2H6O_two_overlapping_known", "FIX-001", (CH3CH2_PAT, CH2OH_PAT), "known", None),\n        ("C3H8O_mass_only", "FIX-003", (), "na", None),\n        ("C3H8O_two_overlapping_unknown", "FIX-003", (CH3CH2_PAT, CH2OH_PAT), "unknown", None),\n        ("amine_CN_unknown", "FIX-006", (CN_PAT,), "unknown", None),\n        ("amide_C1N_unknown", "FIX-007", (C1N_PAT,), "unknown", None),\n        ("amide_C3N_incompatible", "FIX-007", (CN_PAT,), "unknown", None),\n        ("ring_mass_only", "FIX-009", (), "na", None),\n        ("fragment_completion_precursor", "FIX-001", (CH3_PAT,), "unknown", "FIX-003"),\n    ]\n\n\n# --------------------------------------------------------------------------\n# Retrieval index + staged filtering (Experiments A and B).\n# --------------------------------------------------------------------------\n\nclass DatabaseIndex:\n    def __init__(self, records):\n        self.records = records\n        self.by_id = {r["id"]: r for r in records}\n        self.by_formula = {}\n        self.fkeys = {}\n        for r in records:\n            key = formula_key(r["formula"])\n            self.by_formula.setdefault(key, []).append(r)\n            self.fkeys[r["id"]] = key\n        self.fps = {r["id"]: edge_fingerprint(r["atom_types"], r["edges"]) for r in records}\n        # NOTE: connectivity identities are NOT computed eagerly: canonical_key\n        # is factorial in atom count and hangs on drug-sized molecules. Use\n        # identity() for small graphs only.\n        self._identities = {}\n\n    def identity(self, rid, max_atoms=8):\n        """Canonical connectivity identity for small graphs; None above cap."""\n        if rid in self._identities:\n            return self._identities[rid]\n        r = self.by_id[rid]\n        if len(r["atom_types"]) > max_atoms:\n            return None\n        key = canonical_key(r["atom_types"], r["edges"])\n        self._identities[rid] = key\n        return key\n\n    def __len__(self):\n        return len(self.records)\n\n\ndef formula_key(formula):\n    return tuple(sorted((e, formula.get(e, 0)) for e in DOMAIN_ELEMENTS))\n\n\ndef run_query(index, target, patterns, overlap="unknown", precursor=None,\n              uncertainty=50, ppm_tenths=100, counters=None):\n    """Four staged candidate sets. Returns dict with counts/recall/statuses."""\n    counters = counters or WorkCounters()\n    observed = target["mass"]\n    # Stage 1: mass window (accept + ambiguous retained; reject dropped).\n    # Unknown precision ("unavailable") disables the mass decision: those\n    # candidates pass through and are counted separately, never rejected.\n    s1, n_accept, n_ambig, n_reject, n_unavail = [], 0, 0, 0, 0\n    for r in index.records:\n        v = mass_verdict(observed, r["formula"], uncertainty, ppm_tenths)\n        if v == "accept":\n            n_accept += 1\n            s1.append(r)\n        elif v == "ambiguous":\n            n_ambig += 1\n            s1.append(r)\n        elif v == "unavailable":\n            n_unavail += 1\n            s1.append(r)\n        else:\n            n_reject += 1\n    # Stage 2: exact formula + charge/element domain.\n    target_key = formula_key(target["formula"])\n    s2 = [r for r in s1 if index.fkeys.get(r["id"], formula_key(r["formula"])) == target_key\n          and r["charge"] == target["charge"]]\n    # Stage 3: subgraph containment + overlap consistency. Pattern-side prep\n    # is hoisted out of the per-candidate loop; per-candidate target prep is\n    # built once and reused across patterns.\n    pat_fps = [pattern_fingerprint(pt, pe) for pt, pe in patterns]\n    single_preps = [((pt, pe) + _prepare_single(pt, pe)) for pt, pe in patterns]\n    joint_prep = _prepare_joint(patterns) if len(patterns) > 1 else None\n    from collections import Counter as _Counter\n\n    s3 = []\n    truncated = False\n    try:\n        for r in s2:\n            cfp = index.fps[r["id"]]\n            counters.fingerprint_screens += 1\n            if not all(fingerprint_screen(cfp, pf) for pf in pat_fps):\n                continue\n            counters.exact_matches += 1\n            at, ed = r["atom_types"], r["edges"]\n            if any(len(pt) > len(at) for pt, _, _, _ in single_preps):\n                continue\n            nbr, pos = prepare_target(at, ed)\n            occupied = [False] * len(at)\n            ok = True\n            for pt, pe, order, cons in single_preps:\n                if not _contains_with_prep(nbr, pos, pt, order, cons,\n                                           occupied, counters):\n                    ok = False\n                    break\n            if not ok:\n                continue\n            if joint_prep is not None:\n                flat_types, flat_pat, flat_adj, degrees = joint_prep\n                occupied_sets = [set() for _ in patterns]\n                if not _joint_with_prep(nbr, pos, _Counter(at),\n                                        flat_types, flat_pat, flat_adj,\n                                        degrees, len(patterns),\n                                        occupied_sets, counters):\n                    continue\n            s3.append(r)\n    except BudgetExhausted:\n        truncated = True\n    # Stage 4: precursor evidence (heavy-atom conservation under the\n    # provisional target-fragment reading). No precursor => not-evaluated.\n    if precursor is None:\n        s4, precursor_status = s3, "not_evaluated"\n    else:\n        s4 = [r for r in s3 if all(\n            r["formula"].get(e, 0) <= precursor["formula"].get(e, 0) for e in DOMAIN_ELEMENTS)]\n        precursor_status = "evaluated"\n    in_pool = any(r["id"] == target["id"] for r in index.records)\n    recall = {}\n    for name, pool in (("s1", s1), ("s2", s2), ("s3", s3), ("s4", s4)):\n        recall[name] = (any(r["id"] == target["id"] for r in pool)\n                        if in_pool else None)\n    if s3:\n        zero_status = "nonempty"\n    elif not in_pool:\n        zero_status = "target_absent"\n    else:\n        zero_status = "target_filtered"\n    return {\n        "s1": s1, "s2": s2, "s3": s3, "s4": s4,\n        "n_accept": n_accept, "n_ambiguous": n_ambig, "n_reject": n_reject,\n        "n_unavailable": n_unavail,\n        "recall": recall, "in_pool": in_pool,\n        "precursor_status": precursor_status,\n        "overlap": overlap,  # provenance label; unknown path never sees oracle\n        "zero_status": zero_status,\n        "truncated": truncated,\n    }\n\n\ndef rank_baselines(pool, target_id, index):\n    """Ranking baselines at a fixed pool: uniform, provenance, fingerprint.\n\n    Spectral similarity is unavailable without measured spectra (marked, not\n    fabricated). Top-k reported only when target is in the pool.\n    """\n    ids = [r["id"] for r in pool]\n    uniform = list(ids)\n    provenance = sorted(ids, key=lambda i: (index.by_id[i]["source"],\n                                            index.by_id[i]["id"]))\n    # Fingerprint-similarity baseline: Tanimoto on typed-edge sets vs target.\n    target_fp = index.fps.get(target_id)\n    scored = []\n    for r in pool:\n        a = set(index.fps[r["id"]][1].elements())\n        b = set(target_fp[1].elements()) if target_fp else set()\n        inter = len(a & b)\n        union = len(a | b) or 1\n        scored.append((inter / union, r["id"]))\n    fingerprint = [i for _, i in sorted(scored, reverse=True)]\n    in_pool = target_id in ids\n    out = {"uniform": uniform, "provenance": provenance,\n           "fingerprint": fingerprint, "spectral": "unavailable_no_spectrum"}\n    if in_pool:\n        for k, v in list(out.items()):\n            if isinstance(v, list):\n                out[k + "@1"] = v[0] == target_id\n                out[k + "@3"] = target_id in v[:3]\n    return out, in_pool\n\n\nindex_global = None  # set by main/tests for ranking baseline access\n\n\n# --------------------------------------------------------------------------\n# Ingestion adapters (stubs: pin + hash + license, no download by default).\n# --------------------------------------------------------------------------\n\ndef _pin_error(source, url, note):\n    return FileNotFoundError(\n        f"{source} fixture not supplied. Pin a release from {url}, record its "\n        f"sha256, place the file under data/pinned/, and re-run. {note}")\n\n\ndef ingest_chebi_sdf(path, release, sha256):\n    raise _pin_error("ChEBI", "https://www.ebi.ac.uk/chebi/downloads",\n                     "3-star SDF; CC BY 4.0; standardize charge/isotopes/salts; "\n                     "compute mass from graph, not display values.")\n\n\ndef ingest_massspecgym_tsv(path, release, sha256):\n    raise _pin_error("MassSpecGym", "https://huggingface.co/datasets/roman-bushuiev/MassSpecGym",\n                     "Pin dataset revision; main TSV + needed candidate JSONs only; "\n                     "inspect split/structure duplication before pooling.")\n\n\ndef ingest_mgf(path, release, sha256, license_note):\n    raise _pin_error("MGF", "https://massbank.github.io/MassBank-documentation/about.html",\n                     f"Pinned record archive required. {license_note}")\n\n\ndef pubchem_query_cached(formula, cache, release, calls):\n    """Batched, throttled, cached PubChem lookup (stub).\n\n    Policy: <=5 req/s, cache by (formula, release), record truncation.\n    Never the sole ground truth for a reproducible benchmark.\n    """\n    key = (formula_key(formula), release)\n    if key in cache:\n        return cache[key], False\n    calls.append(formula)\n    raise _pin_error("PubChem", "https://pubchem.ncbi.nlm.nih.gov/docs/pug-rest",\n                     "Query only unresolved formulas; prefer bulk downloads at scale.")\n\n\ndef qm9_match(record, qm9_index):\n    """Restricted positive-control match into QM9 (stub).\n\n    Uses published geometries/frequencies only; no new DFT. Presence is not\n    a lifetime/solution-stability guarantee. Returns (matched, method_env).\n    """\n    if qm9_index is None:\n        return False, "qm9_not_loaded"\n    heavy = record["heavy"]\n    els = set(e for e, _, _ in record["atom_types"]) | {"H"}\n    if heavy <= 9 and els <= {"C", "H", "O", "N", "F"}:\n        return record["id"] in qm9_index, "qm9:b3lyp/6-31G(2df,p)//freq (published)"\n    return False, "outside_qm9_domain"\n\n\n# --------------------------------------------------------------------------\n# Experiment driver.\n# --------------------------------------------------------------------------\n\ndef residual_valence_sum(record):\n    return 0  # closed molecules by construction; kept for stratification key\n\n\ndef stratify_key(target, patterns):\n    return {\n        "heavy": target["heavy"],\n        "n_subgraphs": len(patterns),\n        "overlap": "overlap" if len(patterns) > 1 else ("single" if patterns else "none"),\n        "residual_valence": residual_valence_sum(target),\n        "family": target["family"],\n    }\n\n\ndef run_experiment(index, queries, out_of_pool_ids=(), uncertainty=50, ppm_tenths=100):\n    global index_global\n    index_global = index\n    rows = []\n    t0 = time.perf_counter()\n    c0 = time.process_time()\n    for name, tid, patterns, overlap, prec_id in queries:\n        sub = DatabaseIndex([r for r in index.records if r["id"] not in out_of_pool_ids])\n        index_global = sub\n        target = index.by_id[tid]\n        precursor = index.by_id.get(prec_id) if prec_id else None\n        counters = WorkCounters()\n        res = run_query(sub, target, patterns, overlap, precursor,\n                        uncertainty, ppm_tenths, counters)\n        rank, target_in_s3 = rank_baselines(res["s3"], tid, sub)\n        key = stratify_key(target, patterns)\n        rows.append({\n            "query": name, "target": tid, "overlap": overlap,\n            "in_pool": res["in_pool"], "target_in_s3": target_in_s3,\n            "precursor_status": res["precursor_status"],\n            "n_s1": len(res["s1"]), "n_s2": len(res["s2"]),\n            "n_s3": len(res["s3"]), "n_s4": len(res["s4"]),\n            "recall_s1": res["recall"]["s1"], "recall_s3": res["recall"]["s3"],\n            "accept": res["n_accept"], "ambiguous": res["n_ambiguous"],\n            "reject": res["n_reject"], "unavailable": res["n_unavailable"],\n            "zero_status": res["zero_status"],\n            "top1_uniform": rank.get("uniform@1"), "top1_fingerprint": rank.get("fingerprint@1"),\n            "spectral": rank["spectral"], **{f"strat_{k}": v for k, v in key.items()},\n            "embedding_nodes": counters.embedding_nodes,\n        })\n    wall = time.perf_counter() - t0\n    cpu = time.process_time() - c0\n    return rows, {"wall_s": wall, "cpu_s": cpu,\n                  "per_1000_queries_s": wall / max(len(queries), 1) * 1000}\n\n\ndef denominators(raw_records, standardized, index, rows):\n    parseable = sum(1 for _ in standardized)\n    total = len(raw_records)\n    in_domain = len(index)\n    recalls = [r for r in rows if r["in_pool"]]\n    pool_recall = (sum(1 for r in recalls if r["recall_s3"]) / len(recalls)) if recalls else None\n    amb = sum(r["ambiguous"] for r in rows)\n    # Scaffold overlap across corpus.\n    skels = Counter(skeleton_key(r["atom_types"], r["edges"]) for r in index.records)\n    dup_scaffolds = sum(1 for _, c in skels.items() if c > 1)\n    return {\n        "fraction_parseable": parseable / total if total else None,\n        "fraction_in_domain": in_domain / parseable if parseable else None,\n        "target_in_pool_recall_s3": pool_recall,\n        "n_queries": len(rows),\n        "n_in_pool": len(recalls),\n        "total_boundary_ambiguous_hits": amb,\n        "scaffolds_total": len(skels),\n        "scaffolds_shared": dup_scaffolds,\n        "physical_label_coverage": "0 (QM9 index not loaded; no new DFT per plan)",\n        "download_bytes": 0, "api_calls": 0,\n    }\n\n\ndef main():\n    raw = fixture_raw_records()\n    # Add two intentionally excluded records to publish denominators honestly.\n    raw = list(raw) + [\n        {"id": "FIX-BAD-CHARGE", "source": "synthetic-fixture", "family": "excluded",\n         "atom_types": (C3, N2), "edges": ((0, 1, 1),), "charge": 1,\n         "original": {"note": "charged; outside V0 neutral domain"}},\n        {"id": "FIX-BAD-SALT", "source": "synthetic-fixture", "family": "excluded",\n         "atom_types": (C3, C3), "edges": (), "charge": 0,\n         "original": {"note": "disconnected salt-like pair"}},\n    ]\n    standardized, excluded = [], []\n    for r in raw:\n        rec, reason = standardize_record(r)\n        if rec is None:\n            excluded.append((r["id"], reason))\n        else:\n            standardized.append(rec)\n    index = DatabaseIndex(standardized)\n    queries = fixture_queries()\n    rows, perf = run_experiment(index, queries)\n    # Out-of-pool slice: drop FIX-001 entirely, re-run one query.\n    oop_rows, _ = run_experiment(index, [q for q in queries if q[1] == "FIX-001"][:2],\n                                 out_of_pool_ids=("FIX-001", "FIX-002"))\n    den = denominators(raw, standardized, index, rows)\n    fields = ["query", "target", "overlap", "in_pool", "target_in_s3", "precursor_status",\n              "n_s1", "n_s2", "n_s3", "n_s4", "recall_s1", "recall_s3",\n              "accept", "ambiguous", "reject", "unavailable", "zero_status",\n              "top1_uniform", "top1_fingerprint", "spectral",\n              "strat_heavy", "strat_n_subgraphs", "strat_overlap",\n              "strat_residual_valence", "strat_family", "embedding_nodes"]\n    buf = io.StringIO()\n    w = csv.DictWriter(buf, fieldnames=fields, extrasaction="ignore")\n    w.writeheader()\n    for r in rows + [{**r, "query": r["query"] + "|out_of_pool_slice"} for r in oop_rows]:\n        w.writerow(r)\n    sys.stdout.write(buf.getvalue())\n    sys.stderr.write(f"# excluded: {excluded}\\n# perf: {perf}\\n# denominators: {den}\\n")\n\n\n# --------------------------------------------------------------------------\n# Tests.\n# --------------------------------------------------------------------------\n\nclass RetrievalTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        recs = []\n        for raw in fixture_raw_records():\n            rec, reason = standardize_record(raw)\n            assert rec is not None, reason\n            recs.append(rec)\n        cls.index = DatabaseIndex(recs)\n        global index_global\n        index_global = cls.index\n\n    def test_mass_trichotomy_and_unavailable(self):\n        f = {"C": 2, "H": 6, "N": 0, "O": 1, "F": 0}\n        obs = neutral_mass(f)\n        self.assertEqual(mass_verdict(obs, f), "accept")\n        self.assertEqual(mass_verdict(obs, f, uncertainty=None), "unavailable")\n        tol = obs * 100 // 10_000_000\n        self.assertEqual(mass_verdict(obs + tol, f), "ambiguous")\n        self.assertEqual(mass_verdict(obs + 10 * tol + 10_000_000, f), "reject")\n\n    def test_standardization_rejects_charge_and_salt(self):\n        bad_c, r1 = standardize_record({"id": "x", "atom_types": (C3, N2),\n                                        "edges": ((0, 1, 1),), "charge": 1})\n        self.assertIsNone(bad_c)\n        self.assertIn("charge", r1)\n        bad_s, r2 = standardize_record({"id": "y", "atom_types": (C3, C3),\n                                        "edges": (), "charge": 0})\n        self.assertIsNone(bad_s)\n        self.assertIn("salt", r2)\n\n    def test_fingerprint_sound(self):\n        # Screen rejections must agree with exact matching (soundness), and\n        # every true embedding must pass the screen (no false rejection).\n        pats = [CH3_PAT, CH2OH_PAT, CH3CH2_PAT, CN_PAT]\n        for r in self.index.records:\n            cfp = self.index.fps[r["id"]]\n            for pt, pe in pats:\n                screen = fingerprint_screen(cfp, pattern_fingerprint(pt, pe))\n                exact = contains_exact(r["atom_types"], r["edges"], pt, pe)\n                if not screen:\n                    self.assertFalse(exact, (r["id"], pt))\n                if exact:\n                    self.assertTrue(screen, (r["id"], pt))\n\n    def test_monotonicity_across_stages(self):\n        target = self.index.by_id["FIX-001"]\n        res = run_query(self.index, target, (CH3CH2_PAT, CH2OH_PAT))\n        self.assertGreaterEqual(len(res["s1"]), len(res["s2"]))\n        self.assertGreaterEqual(len(res["s2"]), len(res["s3"]))\n        self.assertGreaterEqual(len(res["s3"]), len(res["s4"]))\n\n    def test_unknown_overlap_never_sees_oracle(self):\n        target = self.index.by_id["FIX-001"]\n        oracle = {(0, 0): 0}  # fabricated mapping object; unknown path takes none\n        a = run_query(self.index, target, (CH3CH2_PAT, CH2OH_PAT), overlap="unknown")\n        b = run_query(self.index, target, (CH3CH2_PAT, CH2OH_PAT), overlap="unknown",\n                      precursor=None)\n        _ = oracle\n        self.assertEqual([r["id"] for r in a["s3"]], [r["id"] for r in b["s3"]])\n\n    def test_zero_count_reports_target_absent(self):\n        sub = DatabaseIndex([r for r in self.index.records\n                              if r["id"] not in ("FIX-001", "FIX-002")])\n        target = self.index.by_id["FIX-001"]\n        res = run_query(sub, target, (CH3CH2_PAT, CH2OH_PAT))\n        self.assertFalse(res["in_pool"])\n        self.assertIsNone(res["recall"]["s3"])\n        self.assertIn(res["zero_status"], ("target_absent", "nonempty"))\n\n    def test_topk_gated_on_pool_membership(self):\n        target = self.index.by_id["FIX-001"]\n        res = run_query(self.index, target, ())\n        rank, in_pool = rank_baselines(res["s3"], "FIX-001", self.index)\n        self.assertTrue(in_pool)\n        self.assertIn("uniform@1", rank)\n        sub = DatabaseIndex([r for r in self.index.records if r["id"] != "FIX-001"])\n        global index_global\n        index_global = sub\n        res2 = run_query(sub, target, ())\n        rank2, in_pool2 = rank_baselines(res2["s3"], "FIX-001", sub)\n        self.assertFalse(in_pool2)\n        self.assertNotIn("uniform@1", rank2)\n        index_global = self.index\n\n    def test_joint_overlap_rejects_collapse(self):\n        # Two distinct single-atom patterns forced onto one atom by joint\n        # mapping are fine (sharing allowed), but an edge pattern plus an\n        # incompatible atom sharing must still satisfy edge constraints.\n        target = self.index.by_id["FIX-006"]  # methylamine\n        ok = joint_contains(target["atom_types"], target["edges"], (CH3_PAT, N_PAT))\n        self.assertTrue(ok)\n        impossible = (((C3, C3), ((0, 1, 1),)),)\n        self.assertFalse(joint_contains(target["atom_types"], target["edges"], impossible))\n\n    def test_count_triangle_path_is_six(self):\n        # Review regression: leaf bulk-reset overcounted (9 instead of 6)\n        # by letting later branches reuse ancestor atoms.\n        tri = ((C3, C3, C3), ((0, 1, 1), (1, 2, 1), (0, 2, 1)))\n        pat = ((C3, C3, C3), ((0, 1, 1), (1, 2, 1)))\n        nbr, pos = prepare_target(*tri)\n        order, cons = _prepare_single(*pat)\n        self.assertEqual(count_embeddings_prep(nbr, pos, pat[0], order, cons, 100), 6)\n        self.assertEqual(count_embeddings_prep(nbr, pos, pat[0], order, cons, 4), 4)\n\n    def test_unavailable_precision_passes_through(self):\n        # Review regression: unknown precision disabled the mass decision;\n        # it must not read as rejection.\n        target = self.index.by_id["FIX-001"]\n        res = run_query(self.index, target, (), "unknown", None, uncertainty=None)\n        self.assertEqual(len(res["s1"]), len(self.index))\n        self.assertEqual(res["n_unavailable"], len(self.index))\n        self.assertEqual(res["n_reject"], 0)\n        self.assertTrue(res["recall"]["s1"])\n\n    @staticmethod\n    def _all_injections(at, ed, pt, pe):\n        import itertools\n\n        emap = {tuple(sorted((a, b))): o for a, b, o in ed if o}\n        for perm in itertools.permutations(range(len(at)), len(pt)):\n            if any(at[perm[i]] != pt[i] for i in range(len(pt))):\n                continue\n            ok = True\n            for a, b, o in pe:\n                if emap.get(tuple(sorted((perm[a], perm[b])))) != o:\n                    ok = False\n                    break\n            if ok:\n                yield perm\n\n    def test_matchers_agree_with_brute_force(self):\n        # Optimized backtracking matchers vs an independent\n        # permutation-enumeration reference on randomized small graphs.\n        import itertools\n        import random\n\n        def brute_single(at, ed, pt, pe):\n            emap = {tuple(sorted((a, b))): o for a, b, o in ed if o}\n            for perm in itertools.permutations(range(len(at)), len(pt)):\n                if any(at[perm[i]] != pt[i] for i in range(len(pt))):\n                    continue\n                ok = True\n                for a, b, o in pe:\n                    if emap.get(tuple(sorted((perm[a], perm[b])))) != o:\n                        ok = False\n                        break\n                if ok:\n                    return True\n            return False\n\n        def brute_joint(at, ed, patterns):\n            emap = {tuple(sorted((a, b))): o for a, b, o in ed if o}\n            injs = []\n            for pt, pe in patterns:\n                found = []\n                for perm in itertools.permutations(range(len(at)), len(pt)):\n                    if any(at[perm[i]] != pt[i] for i in range(len(pt))):\n                        continue\n                    ok = True\n                    for a, b, o in pe:\n                        if emap.get(tuple(sorted((perm[a], perm[b])))) != o:\n                            ok = False\n                            break\n                    if ok:\n                        found.append(perm)\n                injs.append(found)\n            for combo in itertools.product(*injs):\n                return True  # sharing across patterns allowed; per-pattern\n                # injectivity already enforced by permutations\n            return False\n\n        rnd = random.Random(20261003)\n        pool = [C3, C2, C1, N2, N1, O1, O0]\n        for trial in range(300):\n            n = rnd.randint(1, 6)\n            at = tuple(rnd.choice(pool) for _ in range(n))\n            pairs = [(a, b) for a in range(n) for b in range(a + 1, n)]\n            rnd.shuffle(pairs)\n            ed = tuple((a, b, rnd.choice([1, 1, 2]))\n                       for a, b in pairs[:rnd.randint(0, min(len(pairs), n))])\n            m = rnd.randint(1, 3)\n            pt = tuple(rnd.choice(pool) for _ in range(m))\n            pp = [(a, b) for a in range(m) for b in range(a + 1, m)]\n            rnd.shuffle(pp)\n            pe = tuple((a, b, rnd.choice([1, 2]))\n                       for a, b in pp[:rnd.randint(0, len(pp))])\n            self.assertEqual(contains_exact(at, ed, pt, pe),\n                             brute_single(at, ed, pt, pe), (trial, at, ed, pt, pe))\n            if trial % 10 == 0:\n                # Capped counter agrees with exhaustive enumeration below cap\n                # and saturates at cap above it.\n                from tools.ms2_database_retrieval import (\n                    count_embeddings_prep, prepare_target, _prepare_single)\n                nbr, pos = prepare_target(at, ed)\n                order, cons = _prepare_single(pt, pe)\n                total = sum(1 for _ in self._all_injections(at, ed, pt, pe))\n                self.assertEqual(\n                    count_embeddings_prep(nbr, pos, pt, order, cons, 10_000),\n                    total, (trial, at, ed, pt, pe))\n                self.assertEqual(\n                    count_embeddings_prep(nbr, pos, pt, order, cons, 1),\n                    min(1, total))\n            if trial % 3 == 0:\n                m2 = rnd.randint(1, 2)\n                pt2 = tuple(rnd.choice(pool) for _ in range(m2))\n                pats = [(pt, pe), (pt2, ())]\n                self.assertEqual(joint_contains(at, ed, pats),\n                                 brute_joint(at, ed, pats), (trial, at, ed, pats))\n\n\nif __name__ == "__main__":\n    if len(sys.argv) > 1 and sys.argv[1] == "test":\n        unittest.main(argv=[sys.argv[0]])\n    else:\n        main()\n'}
for relative, source in SOURCES.items():
    path = WORK / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source)
print('Installed embedded source snapshot:', len(SOURCES), 'files')

RUN_LOG_DIR = WORK / 'experiments/molecular_completion/neural_t4'
RUN_LOG_DIR.mkdir(parents=True, exist_ok=True)
def run_logged(command, filename):
    with (RUN_LOG_DIR / filename).open('w') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            log.write(line)
            log.flush()
            print(line, end='', flush=True)
        return_code = process.wait()
    if return_code:
        raise RuntimeError(f'Process exited {return_code}; see {RUN_LOG_DIR / filename}')


In [ ]:
import hashlib, time, requests
REVISION = 'd2e86d0c3bd905a6d578c0dd6053ed2bd41f9c2a'
BASE = f'https://huggingface.co/datasets/roman-bushuiev/MassSpecGym/resolve/{REVISION}/'

def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def download(remote, local, expected_hash, size, prefix=False):
    path = WORK / local
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and digest(path) == expected_hash:
        print('Verified cached', local)
        return
    temporary = path.with_suffix('.partial')
    for attempt in range(3):
        try:
            headers = {'Range': f'bytes=0-{size - 1}'} if prefix else {}
            with requests.get(BASE + remote, headers=headers, stream=True, timeout=(20, 60)) as response:
                response.raise_for_status()
                if prefix and response.status_code == 206 and not response.headers.get('Content-Range', '').startswith('bytes 0-'):
                    raise ValueError('Server returned an unexpected range')
                written = 0
                with temporary.open('wb') as output:
                    for chunk in response.iter_content(1024 * 1024):
                        if not chunk:
                            continue
                        if prefix:
                            chunk = chunk[:size - written]
                        output.write(chunk)
                        written += len(chunk)
                        if prefix and written == size:
                            break
            if written != size or digest(temporary) != expected_hash:
                raise ValueError(f'Pinned size/hash mismatch for {local}')
            temporary.replace(path)
            print('Downloaded and verified', local)
            return
        except (requests.RequestException, ValueError):
            temporary.unlink(missing_ok=True)
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)

download('data/MassSpecGym1.5.tsv', 'data/pinned/MassSpecGym1.5.tsv',
         '50cfdd1d22f79543c59555f9ce43c6893bd788a19a42b21fb4e2e3a54673c06a', 261197365)
download('data/molecules/MassSpecGym1.5_retrieval_candidates_formula.json',
         'data/pinned/msgym_candidates_formula_prefix64.json',
         '6066de3b1f2a5c013a6d37959b0c6f2f8a4867dc2fdef07fdbe3eaceaa9a6864',
         67108864, prefix=True)


## CPU and actual CUDA checks
The GPU test checks training improvement and CPU/CUDA graph-encoder parity. Any failure stops the notebook.

In [ ]:
run_logged([sys.executable, '-m', 'unittest', 'tools.test_ms2_neural_ranker', '-v'], 'validation.log')

## Train and evaluate the neural arms
Checkpoints are selected on the held-out training subset. The two arms start with the same seed and training order.

In [ ]:
run_logged([sys.executable, 'tools/ms2_neural_ranker.py', '--device', 'cuda', '--require-t4', '--epochs', '20', '--max-train', '5000', '--max-queries', '200', '--batch-size', '32'], 'training.log')

## Run frozen Ridge and cheap baselines on the identical pools

In [ ]:
run_logged([sys.executable, 'tools/ms2_fp_predictor.py', '--max-train', '5000', '--max-queries', '200', '--out-dir', 'experiments/molecular_completion/neural_t4/ridge'], 'ridge_training.log')

In [ ]:
import csv, json
import sys
sys.path.insert(0, str(WORK))
from tools.ms2_neural_ranker import bootstrap_difference
out = WORK / 'experiments/molecular_completion/neural_t4'
summary = json.loads((out / 'summary.json').read_text())
rows = json.loads((out / 'predictions.json').read_text())
with (out / 'ridge/predictor_rows.csv').open() as stream:
    baseline = list(csv.DictReader(stream))
reference = rows['contrastive']
assert [(r['qid'], r['n_pool']) for r in reference] == [(r['qid'], int(r['n_pool'])) for r in baseline], 'Pool mismatch'
comparisons = {}
for arm in ('predictor', 'prior', 'massresid', 'uniform'):
    base_hits = [bool(r['rank_' + arm]) and float(r['rank_' + arm]) <= 25 for r in baseline]
    comparisons[arm] = {'top25': sum(base_hits) / len(base_hits)}
    for neural in rows:
        hits = [r['rank'] is not None and r['rank'] <= 25 for r in rows[neural]]
        comparisons[arm][neural + '_minus_baseline'] = bootstrap_difference(hits, base_hits, 42)
(out / 'baseline_comparison.json').write_text(json.dumps(comparisons, indent=2))
print(json.dumps(summary['metrics'], indent=2))
print(json.dumps(comparisons, indent=2))
print('A positive paired CI is exploratory evidence of better retrieval on this slice; it is not evidence of physical stability.')


## Save artifacts
Download the archive and return it for analysis. It contains neural checkpoints, histories, input/source hashes, per-query ranks, baseline results, and paired bootstrap intervals.

In [ ]:
import shutil
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], stdout=(out / 'gpu.txt').open('w'), check=True)
with (out / 'environment.txt').open('w') as stream:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=stream, check=True)
archive = shutil.make_archive('/content/ms2_neural_t4_results', 'zip', out)
if os.environ.get('COLAB_CLI_RUN') == '1':
    print('RESULT_ARCHIVE', archive)
else:
    from google.colab import files
    files.download(archive)
